---
# >>> Section from `notebooks/p1_data.ipynb`


# Milestone 1: F1 Podium Prediction. Part 1: Data audit, cleaning and EDA

**Target:** `podium = 1` if the driver finishes in the top 3. **One row = one driver in one race.** We predict after qualifying and before the race starts, so only pre-race information can be used as features.

Sections: 1) setup, 2) loading and key audit, 3) EDA before cleaning, 4) cleaning and types, 5) duplicates, 6) Indy 500 and DNQs, 7) EDA after cleaning, 8) `load_clean()` and data dictionary.

## 1. Setup
Fixed random seed and one `DATA_DIR` variable that finds the CSVs on Kaggle or on a laptop, so the notebook runs with a single "Run All".

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")


import os, random
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

SENT = "\\N"                    # the CSVs write a missing value as the text \N
KEYS = ["raceId", "driverId"]   # one row per driver per race
TABLES = ["circuits", "constructors", "constructor_results", "constructor_standings", "drivers",
          "driver_standings", "races", "results", "qualifying", "lap_times", "pit_stops",
          "sprint_results", "seasons", "status"]


def find_data_dir():
    """Works on Kaggle (/kaggle/input/...) and on a laptop (data/raw)."""
    folders = [Path("data/raw"), Path("../data/raw"), Path("data"), Path(".")]
    if Path("/kaggle/input").exists():
        folders = [p.parent for p in Path("/kaggle/input").rglob("races.csv")] + folders
    for folder in folders:
        if all((folder / f"{t}.csv").exists() for t in TABLES):
            return folder
    raise FileNotFoundError("F1 CSV files not found. Add the Kaggle dataset or put the CSVs in data/raw/")


def load_raw_tables(data_dir):
    """Read every CSV as plain text so nothing is guessed or silently changed."""
    raw = {}
    for t in TABLES:
        raw[t] = pd.read_csv(Path(data_dir) / f"{t}.csv", dtype=str, keep_default_na=False,
                             encoding_errors="replace")
    return raw


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def save_fig(name):
    """Save the current figure as a PNG for the report."""
    os.makedirs("reports", exist_ok=True)
    plt.savefig(f"reports/{name}.png", dpi=130, bbox_inches="tight")


print("data folder:", DATA_DIR)

## 2. Loading the 14 tables and checking the keys

**Why this choice?** We read every file as text. The files write missing values as the text `\N`, and if pandas guessed the types, a numeric column with one `\N` would silently turn into text. Reading everything as text lets us count the `\N`s before changing anything, and the raw tables stay untouched (we only work on copies).

**Why check keys first?** Everything later is a join. If a primary key is duplicated, a join multiplies rows. If a foreign key points to a row that does not exist, a join silently drops rows. So we check that every primary key is unique and not empty, and that every foreign key exists in the table it points to.

In [ ]:
RAW = load_raw_tables(DATA_DIR)
display(pd.DataFrame({"rows": {t: len(df) for t, df in RAW.items()}, "columns": {t: df.shape[1] for t, df in RAW.items()}}))

In [ ]:
primary_keys = {
    "circuits": ["circuitId"], "constructors": ["constructorId"], "constructor_results": ["constructorResultsId"],
    "constructor_standings": ["constructorStandingsId"], "drivers": ["driverId"], "driver_standings": ["driverStandingsId"],
    "races": ["raceId"], "results": ["resultId"], "qualifying": ["qualifyId"],
    "lap_times": ["raceId", "driverId", "lap"], "pit_stops": ["raceId", "driverId", "stop"],
    "sprint_results": ["resultId"], "seasons": ["year"], "status": ["statusId"],
}
rows = []
for table, cols in primary_keys.items():
    df = RAW[table]
    empty = ((df[cols] == SENT) | (df[cols] == "")).any(axis=1).sum()
    duplicated = df.duplicated(cols, keep=False).sum()
    rows.append({"table": table, "primary key": "+".join(cols), "rows": len(df),
                 "empty keys": empty, "rows with duplicate key": duplicated, "OK": empty == 0 and duplicated == 0})
pk_audit = pd.DataFrame(rows)
display(pk_audit)

In [ ]:
foreign_keys = [  # (child table, child column, parent table, parent column)
    ("results", "raceId", "races", "raceId"), ("results", "driverId", "drivers", "driverId"),
    ("results", "constructorId", "constructors", "constructorId"), ("results", "statusId", "status", "statusId"),
    ("qualifying", "raceId", "races", "raceId"), ("qualifying", "driverId", "drivers", "driverId"),
    ("qualifying", "constructorId", "constructors", "constructorId"),
    ("races", "circuitId", "circuits", "circuitId"), ("races", "year", "seasons", "year"),
    ("sprint_results", "raceId", "races", "raceId"), ("sprint_results", "driverId", "drivers", "driverId"),
    ("sprint_results", "constructorId", "constructors", "constructorId"), ("sprint_results", "statusId", "status", "statusId"),
    ("driver_standings", "raceId", "races", "raceId"), ("driver_standings", "driverId", "drivers", "driverId"),
    ("constructor_standings", "raceId", "races", "raceId"), ("constructor_standings", "constructorId", "constructors", "constructorId"),
    ("constructor_results", "raceId", "races", "raceId"), ("constructor_results", "constructorId", "constructors", "constructorId"),
    ("lap_times", "raceId", "races", "raceId"), ("lap_times", "driverId", "drivers", "driverId"),
    ("pit_stops", "raceId", "races", "raceId"), ("pit_stops", "driverId", "drivers", "driverId"),
]
rows = []
for child, col, parent, pcol in foreign_keys:
    values = RAW[child][col]
    values = values[(values != SENT) & (values != "")]          # empty foreign keys are "missing", not "broken"
    orphans = (~values.isin(RAW[parent][pcol])).sum()
    rows.append({"child column": f"{child}.{col}", "points to": f"{parent}.{pcol}",
                 "orphans": orphans, "OK": orphans == 0})
fk_audit = pd.DataFrame(rows)
display(fk_audit)
print("primary keys OK:", int(pk_audit["OK"].sum()), "of", len(pk_audit), "| foreign keys OK:", int(fk_audit["OK"].sum()), "of", len(fk_audit))

**What this means.** A table marked OK can safely be the "one" side of a join. Any orphan row is reported, not repaired: we never invent a missing parent. Later we use inner joins on `raceId`, so a result with no race is dropped (it could not be placed in time anyway). Note that a surrogate key like `resultId` is unique by construction, so the real question for our table is whether `(raceId, driverId)` is unique. We check that in section 5.

## 3. EDA before cleaning

Everything here is computed from the raw tables, so we can compare with the cleaned data in section 7.

**3.1 Missing values (`\N`) per column.** We count the `\N` cells in every column and plot the worst columns and the overall distribution.

In [ ]:
rows = []
for table, df in RAW.items():
    for col in df.columns:
        n = (df[col] == SENT).sum()
        rows.append({"table": table, "column": col, "missing": n, "pct": 100 * n / len(df)})
missing = pd.DataFrame(rows)
with_missing = missing[missing["missing"] > 0].sort_values("pct", ascending=False)
print(len(with_missing), "of", len(missing), "columns contain at least one \\N")
display(with_missing.head(25))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
top = with_missing.head(20)
ax[0].barh(top["table"] + "." + top["column"], top["pct"], color="tab:blue")
ax[0].invert_yaxis(); ax[0].set_xlabel("% of rows that are \\N"); ax[0].set_title("Top 20 columns by missing values")
ax[1].hist(missing["pct"], bins=20, color="tab:orange"); ax[1].set_yscale("log")
ax[1].set_xlabel("% missing in a column"); ax[1].set_ylabel("number of columns (log scale)")
ax[1].set_title("Missingness over all columns")
save_fig("fig_p1_01_missing_values_raw")
plt.show()

**What we see.** Most columns have no `\N` at all and a few have a lot. The big ones are the qualifying times `q2` and `q3` (a driver knocked out in Q1 never drives Q2 or Q3, and before 2006 there was no Q2 or Q3), the fastest-lap columns in old seasons (not recorded), and the newer schedule columns in `races`. The reason differs per column, so in cleaning we only turn `\N` into `NaN` and we do **not** fill anything in. Filling in is done later by the modeling team on the training years only.

**3.2 Field size and podium rate per season.** The podium always has 3 places, so the share of drivers on the podium is about 3 divided by the number of cars (the field size).

In [ ]:
def num(s):
    return pd.to_numeric(s.replace(SENT, np.nan), errors="coerce")

res_raw = RAW["results"][["raceId", "driverId", "positionOrder"]].copy()
res_raw["positionOrder"] = num(res_raw["positionOrder"])
races_raw = RAW["races"][["raceId", "year"]].copy()
races_raw["year"] = num(races_raw["year"])
res_raw = res_raw.merge(races_raw, on="raceId")
res_raw["podium"] = res_raw["positionOrder"] <= 3

season_raw = res_raw.groupby("year").agg(rows=("driverId", "size"), races=("raceId", "nunique"), podium_rate=("podium", "mean"))
season_raw["rows_per_race"] = season_raw["rows"] / season_raw["races"]
season_raw["unique_drivers_per_race"] = res_raw.groupby(["year", "raceId"])["driverId"].nunique().groupby("year").mean()
season_raw["3_divided_by_field"] = 3 / season_raw["rows_per_race"]

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(season_raw.index, season_raw["rows_per_race"], label="result rows per race")
ax[0].plot(season_raw.index, season_raw["unique_drivers_per_race"], "--", label="different drivers per race")
ax[0].set_title("Entrants per race (raw)"); ax[0].set_xlabel("season"); ax[0].legend()
ax[1].plot(season_raw.index, season_raw["podium_rate"], label="observed podium rate")
ax[1].plot(season_raw.index, season_raw["3_divided_by_field"], "--", label="3 / field size")
ax[1].set_title("Podium rate per season (raw)"); ax[1].set_xlabel("season"); ax[1].legend()
save_fig("fig_p1_02_field_size_podium_rate_raw")
plt.show()
print("overall podium rate (raw rows):", round(res_raw["podium"].mean(), 3))

**What we see.** The field size changes a lot over the years (big and uneven fields in the 1950s and in the 1980s-90s, about 20 cars today), so the podium rate changes too: about 15% with 20 cars, much lower with 30+. In the early years there are more result rows per race than different drivers. That gap is the first sign of duplicate rows (section 5). So accuracy is not a good metric for this problem, and the class balance will be different in our train, validation and test years.

**3.3 Qualifying coverage per season.**

In [ ]:
quali_raw = RAW["qualifying"]
flags = quali_raw.assign(has_q1=(quali_raw["q1"] != SENT) & (quali_raw["q1"] != ""),
                         has_q3=(quali_raw["q3"] != SENT) & (quali_raw["q3"] != ""))
flags = flags.groupby(KEYS)[["has_q1", "has_q3"]].max().reset_index()
flags["has_quali_row"] = True

cov = res_raw[KEYS + ["year"]].merge(flags, on=KEYS, how="left")
cov[["has_quali_row", "has_q1", "has_q3"]] = cov[["has_quali_row", "has_q1", "has_q3"]].fillna(False).astype(bool)
coverage = cov.groupby("year")[["has_quali_row", "has_q1", "has_q3"]].mean()
first_year = coverage.index[coverage["has_quali_row"] > 0].min()

ax = coverage.plot(figsize=(11, 4.5), lw=2)
ax.axvline(first_year, color="k", ls=":")
ax.set_title(f"Share of result rows with qualifying data (first season: {int(first_year)})")
ax.set_xlabel("season"); ax.set_ylabel("share of rows")
save_fig("fig_p1_03_qualifying_coverage")
plt.show()
print("first season with qualifying rows:", int(first_year))

**What we see.** There is no qualifying data at all before the first season printed above, then coverage grows. Q1 appears first and Q2/Q3 only in the knockout format (2006 onwards).

**Why we do not fill in old qualifying times:** for the old seasons the data is not "randomly missing", the session is simply not in the dataset (and the formats were different, so a 1962 time is not comparable with a 2015 time). Filling it with an average would invent a feature for half of F1 history. Instead `load_clean()` keeps a `has_qualifying` flag, and the grid position (always available) is the fallback for old seasons. The modeling team can also decide to train only on the seasons with qualifying data.

## 4. Cleaning and types

**Why this choice?**
* `\N` is text. In a number column it turns the whole column into text, so we cannot average it, and comparing text like `"10" < "2"` is wrong. We replace `\N` (and blank cells, which turned up in `q2`/`q3`) by a real missing value, and we convert columns to proper types: ids and counts to integers that can hold missing values (`Int64`), dates to datetime, and lap times like `1:27.452` to seconds (`87.452`) so they can be compared and subtracted.
* We only convert, we do not fill in. Why a value is missing differs by column (see 3.1), so one rule would be wrong, and filling with an average from all years could leak information from the future.
* One check beyond the types: a qualifying lap above 200 s cannot be real. One `q1` value is 1,002.64 s (1995 Japanese Grand Prix), while the longest real laps are the wet Belgian Grand Prix of 1994 (about 2:31 to 2:34). Values above 200 s are set to missing, not replaced by a guess. This affects one value.


In [ ]:
INT_COLS = {
    "races": ["year", "round"], "seasons": ["year"], "drivers": ["number"],
    "constructor_standings": ["position", "wins"], "driver_standings": ["position", "wins"],
    "results": ["number", "grid", "position", "positionOrder", "laps", "milliseconds", "fastestLap", "rank"],
    "qualifying": ["number", "position"],
    "lap_times": ["lap", "position", "milliseconds"],
    "pit_stops": ["stop", "lap", "milliseconds"],
    "sprint_results": ["number", "grid", "position", "positionOrder", "laps", "milliseconds", "fastestLap"],
}
FLOAT_COLS = {
    "circuits": ["lat", "lng", "alt"],
    "constructor_results": ["points"], "constructor_standings": ["points"], "driver_standings": ["points"],
    "results": ["points", "fastestLapSpeed"], "sprint_results": ["points", "fastestLapSpeed"],
}
DATE_COLS = {
    "races": ["date", "fp1_date", "fp2_date", "fp3_date", "quali_date", "sprint_date"],
    "drivers": ["dob"],
}
TIME_COLS = {   # text lap times -> seconds (old column: new column)
    "results": {"fastestLapTime": "fastestLapTime_s"},
    "sprint_results": {"fastestLapTime": "fastestLapTime_s"},
    "qualifying": {"q1": "q1_s", "q2": "q2_s", "q3": "q3_s"},
    "lap_times": {"time": "time_s"},
    "pit_stops": {"duration": "duration_s"},
}


MAX_QUALI_LAP_S = 200   # a qualifying lap above 200 s is a data error (one q1 value is 1,002.64 s)


def to_seconds(s):
    """'1:27.452' -> 87.452 (also handles '59.3' and '1:02:03.5'). Anything else -> NaN."""
    parts = s.astype("string").str.extract(r"^(?:(?:(\d+):)?(\d+):)?(\d+(?:\.\d+)?)$")
    hours = pd.to_numeric(parts[0]).fillna(0)
    minutes = pd.to_numeric(parts[1]).fillna(0)
    seconds = pd.to_numeric(parts[2])
    return (hours * 3600 + minutes * 60 + seconds).astype("float64")


def clean_tables(raw):
    """Return cleaned copies of all tables. The raw tables are not touched."""
    clean = {}
    for name, df in raw.items():
        d = df.replace(SENT, np.nan)                      # \N -> NaN
        d = d.replace(r"^\s*$", np.nan, regex=True)       # blank cells are missing too
        for c in d.columns:                               # ids and counts -> integers (can hold NaN)
            if c.endswith("Id") or c in INT_COLS.get(name, []):
                d[c] = pd.to_numeric(d[c], errors="coerce").round().astype("Int64")
        for c in FLOAT_COLS.get(name, []):
            if c in d.columns:
                d[c] = pd.to_numeric(d[c], errors="coerce")
        for c in DATE_COLS.get(name, []):                 # text -> real dates
            if c in d.columns:
                d[c] = pd.to_datetime(d[c], format="%Y-%m-%d", errors="coerce")
        for old, new in TIME_COLS.get(name, {}).items():  # text lap times -> seconds
            if old in d.columns:
                d[new] = to_seconds(d[old])
                d = d.drop(columns=[old])
        if name == "qualifying":                          # impossible lap times become missing (not filled in)
            for c in ["q1_s", "q2_s", "q3_s"]:
                d[c] = d[c].where(d[c] <= MAX_QUALI_LAP_S)
        clean[name] = d
    return clean

In [ ]:
CLEAN = clean_tables(RAW)

# the lap-time converter on a few examples
test = pd.Series(["1:27.452", "59.3", "1:02:03.5", np.nan, "oops"])
print(dict(zip(test.fillna("NaN"), to_seconds(test).round(3))))

# no \N left anywhere
left = 0
for name, df in CLEAN.items():
    for col in df.columns:
        if df[col].dtype == object or str(df[col].dtype) in ("string", "str"):
            left += (df[col] == SENT).sum()
print("\\N left in the cleaned tables:", left)

pairs = [("results", "grid", "grid"), ("results", "positionOrder", "positionOrder"), ("results", "points", "points"),
         ("results", "fastestLapTime", "fastestLapTime_s"), ("qualifying", "q3", "q3_s"),
         ("races", "date", "date"), ("drivers", "dob", "dob")]
display(pd.DataFrame([(t, f"{a} -> {b}", str(RAW[t][a].dtype), str(CLEAN[t][b].dtype)) for t, a, b in pairs],
                     columns=["table", "column", "type before", "type after"]))

# results joined with the race info and the status text (needed for the next sections)
RES = CLEAN["results"].merge(CLEAN["races"][["raceId", "year", "round", "date", "circuitId"]], on="raceId", how="inner")
RES = RES.merge(CLEAN["status"], on="statusId", how="left")
print("result rows:", len(RES), "(raw:", len(RAW["results"]), ")")

### What does `grid = 0` mean?
The project notes say `0` is probably a pit-lane start. We check two ideas:
* **Pit-lane start:** the driver did race (laps > 0) but had no grid slot.
* **No grid slot because he did not start:** DNQ or withdrawn drivers, mostly in old seasons.

In [ ]:
g = RES.copy()
g["grid_zero"] = (g["grid"] == 0).fillna(False).astype(bool)
g["decade"] = (g["year"] // 10 * 10).astype(int)
g["ran"] = g["laps"].fillna(0) > 0

table = g.groupby("decade").agg(rows=("grid", "size"), grid_zero=("grid_zero", "sum"))
table["grid_zero_and_raced"] = g[g.grid_zero & g.ran].groupby("decade").size()
table["grid_zero_no_laps"] = g[g.grid_zero & ~g.ran].groupby("decade").size()
table = table.fillna(0).astype(int)
display(table)

table[["grid_zero_and_raced", "grid_zero_no_laps"]].plot(kind="bar", stacked=True, figsize=(9, 4), color=["tab:green", "tab:red"])
plt.title("Rows with grid = 0, by decade"); plt.ylabel("rows"); save_fig("fig_p1_04_grid_zero_by_decade")
plt.show()

old, new = g[g.grid_zero & (g.year < 1990)], g[g.grid_zero & (g.year >= 2000)]
print("before 1990:", len(old), "rows | share that completed laps:", round(old["ran"].mean(), 2), "| positionText:", old["positionText"].value_counts().head(4).to_dict())
print("2000+      :", len(new), "rows | share that completed laps:", round(new["ran"].mean(), 2), "| top statuses:", new["status"].value_counts().head(4).to_dict())

**Conclusion.** `grid = 0` means "no grid slot", and there are two kinds of rows behind it. In modern seasons these are drivers who raced from the **pit lane** (after a penalty, a car change, or missing the grid). In old seasons they are mostly drivers who **did not start** (failed to qualify or withdrew) or whose slot was not recorded.

**Why this choice for the table:** `0` must never be read as "better than pole position". We remove the non-starters first (section 6), and what is left with `grid = 0` is flagged as `pitlane_start`. We keep the original `grid` column unchanged.

## 5. One row per driver per race (duplicates)

**Why we need exactly one row per `(raceId, driverId)`:** every row is one question for the model: "will this driver finish on the podium in this race?" If a driver has two rows, they can have different labels and different features (two cars, two grid slots), so the model would be asked the same question twice with different answers. It also counts that driver twice in every rate. Two rows for the same driver also means the same race can end up in two different splits.

**Why duplicates exist:** until 1957 a driver could take over a teammate's car during the race (a *shared drive*) and the points were split. The data has one result row per car, so the same driver gets two rows in one race.

In [ ]:
dup_all = RES.duplicated(KEYS, keep=False)        # every row that belongs to a duplicate group
extra = RES.duplicated(KEYS, keep="first")        # the rows beyond the first one
print("duplicate groups (driver-races with more than one row):", RES[dup_all].groupby(KEYS).ngroups)
print("rows in those groups:", int(dup_all.sum()), "| extra rows to remove:", int(extra.sum()))

per_year = extra.groupby(RES["year"]).sum().reindex(range(1950, 2025), fill_value=0)
print("extra rows in 1950-1957:", int(per_year.loc[1950:1957].sum()), "of", int(per_year.sum()))
plt.figure(figsize=(12, 3.8))
plt.bar(per_year.index, per_year.values, color="tab:red")
plt.title("Duplicate (raceId, driverId) rows per season"); plt.xlabel("season"); plt.ylabel("extra rows"); save_fig("fig_p1_05_duplicates_per_season")
plt.show()

examples = RES[dup_all].merge(CLEAN["drivers"][["driverId", "surname"]], on="driverId")
examples = examples.merge(CLEAN["constructors"][["constructorId", "name"]].rename(columns={"name": "constructor"}), on="constructorId", how="left")
display(examples.sort_values(["year", "raceId", "surname"])[["year", "raceId", "surname", "constructor", "number", "grid", "positionOrder", "status"]].head(10))

### The rule for choosing which row to keep
There are two separate questions, and they follow different rules:

1. **Which row do we keep?** Its `grid`, `constructorId` and `number` become features, so we may only use facts that are known **before the race**: first a car that really started (`grid > 0`), then the lowest grid slot, then `constructorId` and car number as arbitrary but fixed tie-breaks. `resultId` is only the very last tie-break, for rows that look identical before the race. We do **not** use `positionOrder` here, because choosing the row by its finishing position would let the race result decide the features.
2. **What is the label?** The label is the outcome, so it may use the result. `podium = 1` if the driver's **best** finish over all his rows in that race is 1, 2 or 3 (a driver who took over a podium car did get that finish). For a normal driver with one row this is just `positionOrder <= 3`.

**Limitation:** when two cars both started, the pre-race columns cannot tell which one was the driver's own, so the kept `grid` may be the teammate's slot. This only affects shared drives, which are a tiny part of the data (85 driver-races before the Indy 500 and the non-starters are removed, 61 rows in the final table, see section 8) and all of them are from 1950-1964, so the validation and test years are not touched. These rows are flagged with `shared_drive`.

In [ ]:
def flag_indy500(races, circuits):
    """Mark the Indianapolis 500 rounds that counted for the championship (1950-1960).
    The same circuit hosted a real F1 US Grand Prix in 2000-2007, so we also check the year."""
    races = races.copy()
    indy_ids = circuits.loc[circuits["name"].str.contains("Indianapolis", na=False), "circuitId"]
    by_name = races["name"].str.contains("Indianapolis 500", na=False)
    by_place = races["circuitId"].isin(indy_ids) & races["year"].between(1950, 1960)
    races["is_indy500"] = (by_name | by_place).astype(bool)
    return races


def find_non_starters(res):
    """Drivers who did not qualify / withdrew. Only uses facts known before the race."""
    by_code = res["positionText"].isin(["F", "W"])      # F = failed to qualify, W = withdrew
    by_status = res["status"].str.contains("did not (?:pre)?qualify|not qualified|withdrew|107%",
                                           case=False, regex=True, na=False)
    return (by_code | by_status).astype(bool)


def keep_one_row_per_driver(res):
    """One row per (raceId, driverId).
    WHICH row we keep uses only facts known before the race (never the finishing position):
      1) a car that really started (grid > 0), 2) the lowest grid slot, 3) constructorId, car number,
      4) resultId only as a last resort (rows that look identical before the race).
    The LABEL is different: podium = the driver's best finish over all his rows <= 3."""
    res = res.reset_index(drop=True).copy()
    res["n_raw_rows"] = res.groupby(KEYS)["resultId"].transform("size")
    res["best_positionOrder"] = res.groupby(KEYS)["positionOrder"].transform("min")
    res["started"] = res["grid"].fillna(0) > 0
    res["grid_for_sort"] = res["grid"].where(res["started"])
    res = res.sort_values(KEYS + ["started", "grid_for_sort", "constructorId", "number", "resultId"],
                          ascending=[True, True, False, True, True, True, True], kind="mergesort")
    kept = res.drop_duplicates(KEYS, keep="first")
    dropped = res.drop(kept.index).drop(columns=["started", "grid_for_sort"])
    kept = kept.drop(columns=["started", "grid_for_sort"]).copy()
    kept["shared_drive"] = kept["n_raw_rows"] > 1
    kept["podium"] = (kept["best_positionOrder"] <= 3).astype("int8")
    return kept.reset_index(drop=True), dropped

In [ ]:
RES_ONE, DROPPED = keep_one_row_per_driver(RES)
assert not RES_ONE.duplicated(KEYS).any()
assert len(RES_ONE) + len(DROPPED) == len(RES)
print(f"rows: {len(RES)} -> {len(RES_ONE)} (removed {len(DROPPED)}); every (raceId, driverId) pair is still there")

shared = RES_ONE[RES_ONE["shared_drive"]]
own_row_label = (shared["positionOrder"] <= 3).astype(int)
print(f"shared-drive rows (before the Indy 500 and the non-starters are removed): {len(shared)} ({len(shared) / len(RES_ONE):.2%} of all rows), seasons {shared['year'].min()}-{shared['year'].max()}")
print("podium label if we only used the kept row:", int(own_row_label.sum()),
      "| with the driver's best row (used):", int(shared["podium"].sum()),
      "| rows where the two differ:", int((own_row_label != shared["podium"]).sum()))

## 6. Indianapolis 500 and DNQ / non-starters

### Indianapolis 500 (1950-1960)
This race counted for the championship, but almost no F1 drivers entered it, so we **exclude it**.

**Why this choice?** It is a different kind of race (oval, 33 cars, different grid), its drivers are mostly Americans who never raced elsewhere in F1, so it would distort the home-race question (Q2) and the driver form features. We find it by race name and year (not by circuit), because the same Indianapolis circuit hosted a normal F1 US Grand Prix in 2000-2007 that we keep.

In [ ]:
RACES = flag_indy500(CLEAN["races"], CLEAN["circuits"])
print("Indy 500 rounds:", int(RACES["is_indy500"].sum()), "| seasons:", sorted(RACES.loc[RACES.is_indy500, "year"].unique().tolist()))
assert RACES.loc[RACES.year.between(2000, 2007), "is_indy500"].sum() == 0     # the modern US GP stays

RES_ONE = RES_ONE.merge(RACES[["raceId", "is_indy500"]], on="raceId", how="left")
d = RES_ONE.merge(CLEAN["drivers"][["driverId", "nationality"]], on="driverId", how="left")
d["american"] = d["nationality"] == "American"
us_circuits = CLEAN["circuits"].loc[CLEAN["circuits"]["country"].isin(["USA", "United States"]), "circuitId"]
home_rows = d[d["american"] & d["circuitId"].isin(us_circuits)]
print("rows of American drivers at US circuits:", len(home_rows), "| of which Indy 500 rows:", int(home_rows["is_indy500"].sum()),
      f"({home_rows['is_indy500'].mean():.0%})")
print("drivers who only ever appear at the Indy 500:", int(d.groupby("driverId")["is_indy500"].all().sum()))

by_year = d[d.year.between(1950, 1960) & d.american].groupby(["year", "is_indy500"]).size().unstack(fill_value=0)
by_year.plot(kind="bar", stacked=True, figsize=(9, 3.8))
plt.title("American driver entries per season, 1950-1960"); plt.legend(["other F1 races", "Indy 500"]); plt.ylabel("rows"); save_fig("fig_p1_06_american_entries_indy500")
plt.show()

### DNQ / non-starters
In the years with more cars than grid places, slow drivers *failed to (pre)qualify* but still have a row in `results`. Withdrawn drivers are the same. We **separate them from the modeling table**.

**Why this choice?** The model predicts after the grid is set, so a driver who is not on the grid is not part of the question, and their label is always 0. They also make the podium rate artificially low in the big-field years. They never have a podium finish, so removing them does not remove any positives. We find them only with facts known before the race (failed to qualify, withdrew, 107% rule). We do not use the number of laps or the race status of drivers who raced, because dropping rows based on how the race went would use the outcome.

In [ ]:
RES_ONE["is_non_starter"] = find_non_starters(RES_ONE)
ns = RES_ONE[RES_ONE["is_non_starter"] & ~RES_ONE["is_indy500"]]
print("non-starter rows (outside Indy 500):", len(ns), "of", int((~RES_ONE["is_indy500"]).sum()))
print("by decade:", {int(k): int(v) for k, v in ns.groupby(ns["year"] // 10 * 10).size().items()})
print("positionText:", ns["positionText"].value_counts().head(4).to_dict(), "| status:", ns["status"].value_counts().head(3).to_dict())
print("non-starters with a podium finish:", int((ns["podium"] == 1).sum()), "(should be 0)")

RES_FINAL = RES_ONE[~RES_ONE["is_indy500"] & ~RES_ONE["is_non_starter"]].reset_index(drop=True)
print("modeling population:", len(RES_FINAL), "rows")

dec = RES_ONE[~RES_ONE["is_indy500"]].copy()
dec["decade"] = dec["year"] // 10 * 10
rate = pd.DataFrame({"with non-starters": dec.groupby("decade")["podium"].mean(),
                     "starters only": dec[~dec["is_non_starter"]].groupby("decade")["podium"].mean()})
rate.plot(kind="bar", figsize=(9, 3.8)); plt.title("Podium rate per decade"); plt.ylabel("podium rate"); save_fig("fig_p1_07_podium_rate_per_decade")
plt.show()

## 7. EDA after cleaning and before/after comparison

We repeat the metrics from section 3 on the cleaned data (`RES_FINAL`) and put them next to the raw ones.

**First check: did turning `\N` into `NaN` lose or invent anything?** For every column that had missing values we compare the number of missing cells before (`\N` or blank) with the number of `NaN` after. They must be equal.

In [ ]:
rows = []
for table, df in RAW.items():
    for col in df.columns:
        before = ((df[col] == SENT) | (df[col].str.strip() == "")).sum()
        if before == 0:
            continue
        new_col = TIME_COLS.get(table, {}).get(col, col)
        after = CLEAN[table][new_col].isna().sum()
        rows.append({"table": table, "column": col, "missing before": before, "NaN after": after, "difference": after - before})
check = pd.DataFrame(rows)
print("columns checked:", len(check), "| columns with a difference:", int((check["difference"] != 0).sum()))
display(check[check["difference"] != 0])

top = check.sort_values("missing before", ascending=False).head(12)
x = np.arange(len(top))
plt.figure(figsize=(11, 4))
plt.bar(x - 0.2, top["missing before"], 0.4, label="missing before (\\N or blank)")
plt.bar(x + 0.2, top["NaN after"], 0.4, label="NaN after")
plt.xticks(x, top["table"] + "." + top["column"], rotation=70, ha="right"); plt.yscale("log"); plt.legend()
plt.title("Missing values before and after cleaning"); save_fig("fig_p1_08_missing_before_after")
plt.show()

In [ ]:
# field size and podium rate per season: raw vs cleaned
season_clean = RES_FINAL.groupby("year").agg(rows=("driverId", "size"), races=("raceId", "nunique"), podium_rate=("podium", "mean"))
season_clean["rows_per_race"] = season_clean["rows"] / season_clean["races"]

fig, ax = plt.subplots(2, 2, figsize=(14, 8))
ax[0, 0].plot(season_raw.index, season_raw["rows_per_race"], label="raw rows per race")
ax[0, 0].plot(season_clean.index, season_clean["rows_per_race"], label="clean rows per race (starters)")
ax[0, 0].set_title("Field size per season"); ax[0, 0].legend()
ax[0, 1].plot(season_raw.index, season_raw["podium_rate"], label="raw")
ax[0, 1].plot(season_clean.index, season_clean["podium_rate"], label="clean")
ax[0, 1].plot(season_clean.index, 3 / season_clean["rows_per_race"], "k:", label="3 / clean field size")
ax[0, 1].set_title("Podium rate per season"); ax[0, 1].legend()

podium_rows_raw = res_raw.groupby("raceId")["podium"].sum().value_counts().sort_index()
podium_rows_clean = RES_FINAL.groupby("raceId")["podium"].sum().value_counts().sort_index()
pd.concat([podium_rows_raw.rename("raw"), podium_rows_clean.rename("clean")], axis=1).fillna(0).plot(kind="bar", ax=ax[1, 0])
ax[1, 0].set_title("Podium rows per race (3 expected)"); ax[1, 0].set_xlabel("podium rows in a race"); ax[1, 0].set_ylabel("races")

lap = CLEAN["results"]["fastestLapTime_s"].dropna()
ax[1, 1].hist(lap.clip(30, 200), bins=60, color="tab:green")
ax[1, 1].set_title("fastestLapTime_s after conversion (seconds)")
save_fig("fig_p1_09_before_after_field_podium_laptimes")
plt.show()

print("races without exactly 3 podium rows after cleaning:", int((podium_rows_clean.drop(3, errors="ignore")).sum()), "of", int(podium_rows_clean.sum()))
for col, df in [("fastestLapTime_s", CLEAN["results"]), ("q1_s", CLEAN["qualifying"]), ("q3_s", CLEAN["qualifying"])]:
    s = df[col].dropna()
    print(f"{col:17s} n={len(s):6d}  min={s.min():7.2f}  median={s.median():7.2f}  max={s.max():8.2f}")

In [ ]:
# what each cleaning rule removed
def podium_rows(df):
    return int((df["positionOrder"] <= 3).sum())

removed_indy = RES_ONE[RES_ONE["is_indy500"]]
removed_ns = RES_ONE[~RES_ONE["is_indy500"] & RES_ONE["is_non_starter"]]
summary = pd.DataFrame([
    ("typed results (start)", len(RES), podium_rows(RES)),
    ("removed: duplicate rows", len(DROPPED), podium_rows(DROPPED)),
    ("removed: Indy 500", len(removed_indy), podium_rows(removed_indy)),
    ("removed: non-starters", len(removed_ns), podium_rows(removed_ns)),
    ("modeling population", len(RES_FINAL), podium_rows(RES_FINAL)),
], columns=["step", "rows", "rows with positionOrder <= 3"])
display(summary)
print("podium label = 1 in the final table:", int(RES_FINAL["podium"].sum()))

**What the cleaning achieved**
* **Missing values:** the number of missing cells before and after is the same in every column except `q1` (one more `NaN` after: the impossible 1,002.64 s lap set to missing on purpose, see section 4), so nothing else was lost or invented. The lap times are now numbers in a sensible range (tens of seconds, no conversion errors).
* **Duplicates:** the rows per race now match the number of different drivers, and almost every race has exactly 3 podium rows. The few races that do not are shared-drive cases where two drivers were credited with the same podium car.
* **Podium rate:** the change is concentrated in the early decades (duplicates, the Indy 500 and non-starters). Recent seasons, which are our validation and test years, are practically unchanged. The non-starter rule removes no podium rows, and the duplicates are only removed as extra rows, never as a driver-race.

## 8. `load_clean()` and the data dictionary

**Why one function?** If every teammate cleaned the data in their own way (one keeps the Indy 500, another drops DNQs), the row counts and scores would no longer be comparable. `load_clean()` runs all the steps above in the same order, so everyone starts from identical tables.

It returns:
* `model_base`: one row per driver per race with **pre-race columns only** plus the target `podium`.
* `outcomes`: the **post-race** columns of the same rows (status, points, positions, laps...). They are used for the data-engineering questions and must **not** be used as features.
* `tables`: all other cleaned tables. The standings tables are post-race snapshots, so use the previous round before using them as features.
* `meta`: how many rows each step removed.

The function stops with an error if a driver-race has more than one row or if a post-race column ends up in `model_base`. It does not fill in or scale anything, so nothing can be fitted on the test years.

In [ ]:
PRE_RACE_COLS = ["raceId", "driverId", "constructorId", "year", "round", "date", "circuitId", "number",
                 "grid", "pitlane_start", "has_qualifying", "qual_position", "q1_s", "q2_s", "q3_s"]
OUTCOME_COLS = ["raceId", "driverId", "resultId", "positionOrder", "podium", "positionText", "position",
                "points", "laps", "time", "milliseconds", "fastestLap", "rank", "fastestLapTime_s",
                "fastestLapSpeed", "statusId", "status", "n_raw_rows", "shared_drive", "best_positionOrder"]
POST_RACE_COLS = {"positionOrder", "positionText", "position", "points", "laps", "time", "milliseconds",
                  "fastestLap", "rank", "fastestLapTime_s", "fastestLapSpeed", "statusId", "status",
                  "resultId", "n_raw_rows", "shared_drive", "best_positionOrder"}


def load_clean(data_dir=None, exclude_indy500=True, drop_non_starters=True, exclude_shared_drives=False):
    """Raw CSVs -> clean tables. Everybody on the team should start from this function.
    model_base = pre-race columns + target `podium` (one row per driver per race)
    outcomes   = post-race columns (status, points, ...) - never use these as features."""
    random.seed(SEED)
    np.random.seed(SEED)
    tables = clean_tables(load_raw_tables(data_dir or find_data_dir()))
    tables["races"] = flag_indy500(tables["races"], tables["circuits"])

    res = tables["results"].merge(tables["races"][["raceId", "year", "round", "date", "circuitId", "is_indy500"]],
                                  on="raceId", how="inner")
    res = res.merge(tables["status"], on="statusId", how="left")
    meta = {"raw_result_rows": len(tables["results"]), "after_race_join": len(res)}

    if exclude_indy500:
        meta["indy500_rows_removed"] = int(res["is_indy500"].sum())
        res = res[~res["is_indy500"]]
    res, dropped = keep_one_row_per_driver(res)
    meta["duplicate_rows_removed"] = len(dropped)
    res["is_non_starter"] = find_non_starters(res)
    if drop_non_starters:
        meta["non_starter_rows_removed"] = int(res["is_non_starter"].sum())
        res = res[~res["is_non_starter"]]
    if exclude_shared_drives:
        meta["shared_drive_rows_removed"] = int(res["shared_drive"].sum())
        res = res[~res["shared_drive"]]
    res = res.reset_index(drop=True)
    meta["final_rows"] = len(res)

    quali = tables["qualifying"].sort_values("qualifyId")
    meta["qualifying_duplicates_dropped"] = int(quali.duplicated(KEYS, keep="last").sum())
    quali = quali.drop_duplicates(KEYS, keep="last").rename(columns={"position": "qual_position"})
    quali = quali[KEYS + ["qual_position", "q1_s", "q2_s", "q3_s"]]

    res["pitlane_start"] = (res["grid"] == 0).fillna(False).astype(bool)
    base = res.merge(quali, on=KEYS, how="left", indicator=True)
    base["has_qualifying"] = base["_merge"] == "both"

    model_base = base[PRE_RACE_COLS].copy()
    model_base["podium"] = base["podium"].values
    outcomes = base[OUTCOME_COLS].copy()

    # safety checks
    assert not model_base.duplicated(KEYS).any(), "more than one row for a driver in a race"
    assert model_base[KEYS].notna().all().all(), "missing key"
    assert not (POST_RACE_COLS & set(model_base.columns)), "post-race column in the base table"
    return {"model_base": model_base, "outcomes": outcomes, "tables": tables, "meta": meta}

In [ ]:
CL = load_clean(DATA_DIR)
model_base, outcomes = CL["model_base"], CL["outcomes"]
print(CL["meta"])
display(model_base.head())

# same result as the step-by-step version above
assert len(model_base) == len(RES_FINAL) and model_base["podium"].sum() == RES_FINAL["podium"].sum()
print(f"{len(model_base)} rows, {model_base['raceId'].nunique()} races, seasons {model_base['year'].min()}-{model_base['year'].max()}, podium rate {model_base['podium'].mean():.1%}")

split = np.select([model_base["year"] <= 2019, model_base["year"].between(2020, 2021)], ["train (<=2019)", "validation (2020-21)"], "test (>=2022)")
display(model_base.groupby(split)["podium"].agg(rows="size", podium_rate="mean").round(3))

**Counts for the final table.** Shared-drive rows, how many podium labels the best-row rule changed, and how much of each split has qualifying data (these numbers are used in the report).


In [ ]:
sd = CL["outcomes"][CL["outcomes"]["shared_drive"]].merge(model_base[["raceId", "driverId", "year"]], on=KEYS)
print(f"shared-drive rows in the final table: {len(sd)} ({len(sd) / len(model_base):.2%}), seasons {sd['year'].min()}-{sd['year'].max()}, after 1957: {int((sd['year'] > 1957).sum())}")

changed = CL["outcomes"][(CL["outcomes"]["positionOrder"] <= 3).astype(int) != CL["outcomes"]["podium"]]
print("podium labels that differ from positionOrder <= 3:", len(changed), "| all of them shared-drive rows:", bool(changed["shared_drive"].all()))

split_name = np.select([model_base["year"] <= 2019, model_base["year"].between(2020, 2021)], ["train", "validation"], "test")
print("share of rows with qualifying data by split:", model_base.groupby(split_name)["has_qualifying"].mean().round(3).to_dict())
print("share of rows with qualifying data: before 1994 =", round(model_base.loc[model_base["year"] < 1994, "has_qualifying"].mean(), 3),
      "| 2003 onwards =", round(model_base.loc[model_base["year"] >= 2003, "has_qualifying"].mean(), 3))

In [ ]:
descriptions = {
    "raceId": ("key", "race id (links to races)"), "driverId": ("key", "driver id (links to drivers)"),
    "constructorId": ("pre-race", "team of the kept row"), "year": ("pre-race", "season"), "round": ("pre-race", "round number in the season"),
    "date": ("pre-race", "race date"), "circuitId": ("pre-race", "circuit (links to circuits)"), "number": ("pre-race", "car number"),
    "grid": ("pre-race", "starting slot after penalties; 0 = no grid slot (pit-lane start for drivers who started)"),
    "pitlane_start": ("pre-race", "True if grid = 0 (non-starters were already removed)"),
    "has_qualifying": ("pre-race", "True if the driver has a qualifying row"),
    "qual_position": ("pre-race", "qualifying position (NaN when there is no qualifying data)"),
    "q1_s": ("pre-race", "Q1 best lap in seconds"), "q2_s": ("pre-race", "Q2 best lap in seconds (NaN if knocked out in Q1 or before 2006)"),
    "q3_s": ("pre-race", "Q3 best lap in seconds (NaN if knocked out earlier or before 2006)"),
    "podium": ("TARGET", "1 if the driver's best finish in the race is top 3"),
    "resultId": ("outcome", "id of the kept result row"), "positionOrder": ("outcome", "final ranking of every entrant"),
    "positionText": ("outcome", "position or letter code (R, D, E, W, F, N)"), "position": ("outcome", "classified position"),
    "points": ("outcome", "championship points"), "laps": ("outcome", "laps completed"), "time": ("outcome", "race time text"),
    "milliseconds": ("outcome", "race time in ms"), "fastestLap": ("outcome", "lap of the fastest lap"), "rank": ("outcome", "fastest-lap rank"),
    "fastestLapTime_s": ("outcome", "fastest lap in seconds"), "fastestLapSpeed": ("outcome", "fastest lap speed"),
    "statusId": ("outcome", "why the race ended (links to status)"), "status": ("outcome", "status text"),
    "n_raw_rows": ("audit", "number of raw result rows of this driver in this race"),
    "shared_drive": ("audit", "True if the driver had more than one raw row (car swap, 1950-1964)"),
    "best_positionOrder": ("audit", "best positionOrder over the driver's rows (used for the label)"),
}
dictionary = pd.DataFrame([(tbl, col, str(df[col].dtype), *descriptions[col], round(100 * df[col].isna().mean(), 1))
                           for tbl, df in [("model_base", model_base), ("outcomes", outcomes)] for col in df.columns],
                          columns=["table", "column", "type", "role", "meaning", "% missing"])
display(dictionary)

dictionary.to_csv(OUTPUT_DIR / "data_dictionary.csv", index=False)
model_base.to_csv(OUTPUT_DIR / "model_base.csv", index=False)
outcomes.to_csv(OUTPUT_DIR / "outcomes.csv", index=False)

### Notes for the team
* Features come from `model_base` (and from earlier races). Columns marked `outcome` or `audit` are never features.
* Standings tables are after-the-race: shift them to the previous round. Rolling form uses `shift(1)`.
* Fit imputers, scalers and encoders on the training years only (<= 2019).
* Shared-drive rows (61 rows, 0.25% of rows, all from 1950-1964) are kept and flagged. `load_clean(exclude_shared_drives=True)` removes them if the modeling team prefers.

---
# >>> Section from `notebooks/p2_de.ipynb`


## Part 2: Data-engineering questions (P2)

Three questions answered with multi-table joins. Each one ends with the answer, the plots, and short notes on
**grain** (what one row is), **missingness** (what is empty and why) and the **pre-race cutoff** (what is known before the race starts).

In [ ]:
import pandas as pd               # tables
import matplotlib.pyplot as plt   # charts
import seaborn as sns             # heatmap (backup section)

clean = load_clean()
base = clean["model_base"]    # pre-race columns + the target `podium`
outcomes = clean["outcomes"]  # post-race columns (status, points...). Never features
tables = clean["tables"]      # all other cleaned tables (races, circuits, drivers...)

### Q1: Which circuits convert a front-row start into a podium most often?

**In plain words:** when a driver starts 1st or 2nd (the front row), how often do they finish in the top 3, at each circuit?
Does that ranking change after 2014? We answer it twice: once with **grid** (where the car really started) and once with
**qualifying position** (where it qualified).

**Why 2014?** In 2014 F1 switched to **hybrid engines** (a small turbo engine plus batteries and electric motors).
One team (Mercedes) became much faster than the rest, and every engine part became limited, which caused many more grid penalties.

**Which years count as "before 2014"?** We make three comparisons, each one for grid and for qualifying (6 in total):

| Before | After | Why |
|---|---|---|
| **2004-2013** | 2014-2024 | **Main, the fairest:** similar length (10 vs 11 seasons), grid penalties exist in both, full qualifying data, similar car reliability |
| 1994-2013 | 2014-2024 | from the first season with qualifying data |
| 1950-2013 | 2014-2024 | all of F1 history |

**Tables and joins (multi-hop):**

```
results ──raceId──► races ──circuitId──► circuits        (which circuit was the race at?)
   │
   └──(raceId, driverId)──► qualifying                     (where did the driver qualify?)
```

- `results ⋈ races ⋈ qualifying` is already done inside `load_clean()` → `model_base`
  (`grid`, `podium`, `year`, `circuitId`, `qual_position`). `load_clean()` also used `status` to remove drivers who did not start,
  removed the Indianapolis 500 and kept one row per driver per race.
- Here we add the last hop: `model_base ⋈ circuits` on `circuitId` (circuit name and country).
- `drivers` and `races` are joined only to show names in an example table; `outcomes` only to explain the result (never as a feature).
- Not used: `sprint_results` (a different, shorter race with its own grid), standings, lap times, pit stops.

In [ ]:
# 1) take only the 3 columns we need from the circuits table
circuits = tables["circuits"][["circuitId", "name", "country"]]
# 2) rename "name" to "circuit" so the column name is clear
circuits = circuits.rename(columns={"name": "circuit"})

# 3) join: every row of base gets the name and country of its circuit
q1 = base.merge(circuits, on="circuitId", how="left", validate="many_to_one")

# 4) safety checks: the join must not add or lose rows, and every race must find its circuit
assert len(q1) == len(base), "the join changed the number of rows"
assert q1["circuit"].notna().all(), "a race has no circuit name"

print("rows (one per driver per race):", len(q1))
print("seasons:", q1["year"].min(), "-", q1["year"].max(), "| races:", q1["raceId"].nunique(), "| circuits:", q1["circuit"].nunique())

#### Grid vs qualifying position

- **Qualifying position** = the result of Saturday's timed session (fastest lap = 1st).
- **Grid** = where the car really started on Sunday.
- **Before 2004 they were the same:** the grid was simply the qualifying order.
- **From 2004** the rules limit engines (and from 2008 gearboxes). A driver who changes one early is moved back on the grid
  (a **penalty**), and the drivers behind move up. A car changed after qualifying starts from the **pit lane** (`grid = 0`).
- **After 2014** every part of the hybrid engine is limited, so penalties became about **twice as common**.

The chart shows this per season (from 1994, the first season with qualifying data, so we can see the "before").

In [ ]:
# keep only rows that have a qualifying position (1994 and later)
with_quali = q1[q1["has_qualifying"]].copy()

# places lost between qualifying and the start (positive = started further back)
with_quali["places_lost"] = with_quali["grid"] - with_quali["qual_position"]

# pushed back = lost 3 or more places, or started from the pit lane (grid 0)
with_quali["pushed_back"] = (with_quali["places_lost"] >= 3) | (with_quali["grid"] == 0)

# per season: number of races, number of pushed-back drivers, and drivers per race
per_year = with_quali.groupby("year").agg(races=("raceId", "nunique"), pushed_back=("pushed_back", "sum"))
per_year["per_race"] = per_year["pushed_back"] / per_year["races"]

plt.figure(figsize=(11, 4))
plt.bar(per_year.index, per_year["per_race"], color="tab:red", label="drivers pushed back per race")
plt.axvline(2003.5, color="grey", linestyle="--", label="2004: first engine rule")
plt.axvline(2013.5, color="black", linestyle="--", label="2014: hybrid engines")
plt.title("Drivers who started 3+ places behind their qualifying position (or from the pit lane)")
plt.xlabel("season")
plt.ylabel("drivers per race")
plt.legend()
plt.show()

**Why 3 places and not 1?** The table counts, per race, the drivers who started 1, 2, 3-4 or 5+ places behind their
qualifying position. Losing **1** place happens about equally often in every span, even before 2004 when there were no
penalties, so it is not a penalty (small recording differences, or someone moved ahead after qualifying). Losing **3 or more**
is zero before 2004 and then grows with the penalty rules. 3 places is also the smallest common real penalty
(for example, blocking another driver in qualifying).

In [ ]:
# three time spans: before penalties, first engine/gearbox rules, hybrid era
with_quali["span"] = "1994-2003"
with_quali.loc[with_quali["year"] >= 2004, "span"] = "2004-13"
with_quali.loc[with_quali["year"] >= 2014, "span"] = "2014-24"

# only drivers who started behind their qualifying position (pit-lane starts have a negative value, so they are left out)
lost = with_quali[with_quali["places_lost"] >= 1].copy()

# put every driver in a group by how many places they lost
lost["places"] = "1"
lost.loc[lost["places_lost"] == 2, "places"] = "2"
lost.loc[lost["places_lost"] >= 3, "places"] = "3-4"
lost.loc[lost["places_lost"] >= 5, "places"] = "5+"

# count the drivers per span and group, then divide each row by the number of races in that span
counts = pd.crosstab(lost["span"], lost["places"])
races_per_span = with_quali.groupby("span")["raceId"].nunique()
places_lost_per_race = counts.div(races_per_span, axis=0)

print("drivers per race who started 1, 2, 3-4 or 5+ places behind their qualifying position:")
places_lost_per_race.round(2)

On the **front row**, every driver who qualified 1st/2nd or started 1st/2nd is one of three cases:
**same** (qualified and started on the front row), **penalty** (qualified on the front row but started further back),
or **moved up** (qualified 3rd or lower but started on the front row because someone ahead was penalised).

In [ ]:
# True / False for every row: qualified 1st or 2nd? started 1st or 2nd?
qualified_front = (with_quali["qual_position"] == 1) | (with_quali["qual_position"] == 2)
started_front = (with_quali["grid"] == 1) | (with_quali["grid"] == 2)

# give every row a readable name for its case
SAME = "qualified 1st/2nd and started 1st/2nd (same)"
PENALTY = "qualified 1st/2nd but started further back (penalty)"
MOVED_UP = "qualified 3rd or lower but started 1st/2nd (moved up)"
with_quali["case"] = "not on the front row"
with_quali.loc[qualified_front & started_front, "case"] = SAME
with_quali.loc[qualified_front & ~started_front, "case"] = PENALTY
with_quali.loc[~qualified_front & started_front, "case"] = MOVED_UP

# only drivers on the front row in at least one of the two, counted per span
front_cases = with_quali[qualified_front | started_front]
case_table = pd.crosstab(front_cases["case"], front_cases["span"])
case_table

In [ ]:
# a few real examples, with driver and race names
drivers = tables["drivers"][["driverId", "forename", "surname"]]
races = tables["races"][["raceId", "name"]].rename(columns={"name": "race"})

examples = front_cases[front_cases["case"] != SAME]
examples = examples.merge(drivers, on="driverId", how="left")
examples = examples.merge(races, on="raceId", how="left")
examples = examples.sort_values("year", ascending=False)
examples[["year", "race", "forename", "surname", "qual_position", "grid", "case"]].head(6)

In [ ]:
# missing data: % of drivers with a known qualifying position, per season
coverage = q1.groupby("year")["has_qualifying"].mean() * 100

plt.figure(figsize=(11, 3.5))
plt.bar(coverage.index, coverage.values, color="tab:blue", label="% of drivers with a known qualifying position")
plt.title("Qualifying data per season: none before 1994, patchy 1996-2002, complete from 2003")
plt.xlabel("season")
plt.ylabel("% of drivers")
plt.legend(loc="upper left")
plt.show()

#### Filling the qualifying gaps with the grid (only before 2004)

Qualifying positions are missing before 1994 and for many races in 1996-2002. Qualifying **did** happen in those years and
**decided the grid**; the dataset just does not record it. Before 2004 there were no grid penalties, so the grid is the
qualifying order. The check below proves it on the years where we have both. So we make a new column `qual_or_grid`:
the qualifying position, and where it is missing **before 2004**, the grid. Missing values from 2004 on are **not** filled,
because from then on the grid can differ from qualifying (penalties).

In [ ]:
# check: in 1994-2003 (before penalties), is the grid the same as the qualifying position?
before_penalties = q1[q1["has_qualifying"] & (q1["year"] <= 2003)]
same_position = before_penalties["grid"] == before_penalties["qual_position"]
print("1994-2003: grid = qualifying position for", round(same_position.mean() * 100, 1), "% of drivers")

qualified_front_old = (before_penalties["qual_position"] == 1) | (before_penalties["qual_position"] == 2)
started_front_old = (before_penalties["grid"] == 1) | (before_penalties["grid"] == 2)
print("front row: qualified 1st/2nd:", qualified_front_old.sum(),
      "| of these, started 1st/2nd:", (qualified_front_old & started_front_old).sum())

# new column: the qualifying position; where it is missing before 2004, use the grid instead
q1["qual_or_grid"] = q1["qual_position"]
fill = q1["qual_or_grid"].isna() & (q1["year"] < 2004)
q1.loc[fill, "qual_or_grid"] = q1.loc[fill, "grid"]

print("rows filled with the grid:", fill.sum())
print("still missing (2004 and later, not filled):", q1["qual_or_grid"].isna().sum())

#### Helper functions (our toolbox)

| Function | What it does |
|---|---|
| `add_era(df, start_year)` | keeps the seasons from `start_year` on and labels them "start_year-2013" or "2014-24" |
| `front_row_rate(df, position_column)` | per circuit and era: **starts** (drivers in position 1 or 2), **podiums**, **rate** = podiums ÷ starts, **enough_starts** (at least `MIN_STARTS`) |
| `overall_rate(df, position_column)` | the same, but all circuits together |
| `plot_top10(ax, table, era, title)` | bar chart of the 10 circuits with the highest rate in one era |
| `compare_eras(table, before)` | circuits with enough starts in **both** eras, and the **Spearman** correlation of their two rankings |
| `slope_chart(ax, rates, spearman, title)` | the rank-change chart: one line per circuit from its rank before to its rank after |

**Spearman** compares two rankings: **1** = exactly the same order, **0** = the old order says nothing about the new order,
**−1** = the order is reversed.

In [ ]:
MIN_STARTS = 10   # a circuit needs at least 10 front-row starts (about 5 races) in an era to be ranked


def add_era(df, start_year):
    """Keep the seasons from start_year on and label each row with its era."""
    part = df[df["year"] >= start_year].copy()
    part["era"] = str(start_year) + "-2013"
    part.loc[part["year"] >= 2014, "era"] = "2014-24"
    return part


def front_row_rate(df, position_column):
    """For every circuit and era: front-row starts, podiums and the rate."""
    # keep only drivers in position 1 or 2 (a missing position is not front row)
    is_front = ((df[position_column] == 1) | (df[position_column] == 2)).fillna(False)
    front = df[is_front]
    # count the starts and add up the podiums (podium is 1 or 0) per circuit and era
    table = front.groupby(["circuit", "era"])["podium"].agg(starts="count", podiums="sum")
    table = table.reset_index()
    # rate = podiums / starts
    table["rate"] = table["podiums"] / table["starts"]
    # enough starts to be ranked?
    table["enough_starts"] = table["starts"] >= MIN_STARTS
    return table


def overall_rate(df, position_column):
    """Front-row starts, podiums and rate per era, all circuits together."""
    is_front = ((df[position_column] == 1) | (df[position_column] == 2)).fillna(False)
    front = df[is_front]
    table = front.groupby("era")["podium"].agg(starts="count", podiums="sum")
    table["rate"] = table["podiums"] / table["starts"]
    return table


def plot_top10(ax, table, era, title):
    """Bar chart of the 10 circuits with the highest rate in one era (only circuits with enough starts)."""
    rows = table[(table["era"] == era) & table["enough_starts"]]
    # sort from low to high and keep the last 10, so the best circuit is drawn at the top
    top10 = rows.sort_values(["rate", "starts"]).tail(10)
    ax.barh(top10["circuit"], top10["rate"] * 100, color="tab:green", label="% of front-row starters on the podium")
    # write "podiums of starts" on each bar
    for i in range(len(top10)):
        ax.text(1, i, str(top10["podiums"].iloc[i]) + " of " + str(top10["starts"].iloc[i]),
                va="center", color="white", fontsize=8)
    ax.set_xlim(0, 100)
    ax.set_title(title)
    ax.set_xlabel("% of front-row starters who finished on the podium")
    ax.legend(loc="lower right", fontsize=8)


def compare_eras(table, before, after="2014-24", min_starts=MIN_STARTS):
    """Rates of the circuits that have enough starts in both eras, and the Spearman correlation of their ranks."""
    rates = table.pivot(index="circuit", columns="era", values="rate")[[before, after]]
    starts = table.pivot(index="circuit", columns="era", values="starts")[[before, after]]
    in_both = (starts[before] >= min_starts) & (starts[after] >= min_starts)
    rates = rates[in_both]
    ranks = rates.rank(ascending=False)   # 1 = highest rate
    spearman = ranks[before].corr(ranks[after])
    return rates, spearman


def slope_chart(ax, rates, spearman, title):
    """Left column = order before, right column = order after, one line per circuit."""
    before = rates.columns[0]
    after = rates.columns[1]
    # position of every circuit in each column (0 = top)
    left = pd.Series(range(len(rates)), index=rates.sort_values(before, ascending=False).index)
    right = pd.Series(range(len(rates)), index=rates.sort_values(after, ascending=False).index)
    for circuit in rates.index:
        if right[circuit] < left[circuit]:
            color = "tab:green"
        elif right[circuit] > left[circuit]:
            color = "tab:red"
        else:
            color = "grey"
        ax.plot([0, 1], [left[circuit], right[circuit]], marker="o", color=color)
        ax.text(-0.03, left[circuit], circuit + " (" + str(round(rates.loc[circuit, before] * 100)) + "%)",
                ha="right", va="center", fontsize=8)
        ax.text(1.03, right[circuit], "(" + str(round(rates.loc[circuit, after] * 100)) + "%) " + circuit,
                ha="left", va="center", fontsize=8)
    ax.plot([], [], color="tab:green", label="better rank after")
    ax.plot([], [], color="tab:red", label="worse rank after")
    ax.plot([], [], color="grey", label="same rank")
    ax.invert_yaxis()                       # best at the top
    ax.set_xlim(-1.2, 2.2)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([before, after])
    ax.set_yticks([])
    ax.set_ylabel("rank (best at the top)")
    ax.set_title(title + "\n" + str(len(rates)) + " circuits, Spearman = " + str(round(spearman, 2)))
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=3, fontsize=8)

#### Minimum number of starts

Some circuits hosted only a few races in an era. A circuit with 2 front-row starts and 2 podiums shows 100%, but that is
2 cars, not a pattern, and it would wrongly top the ranking. So a circuit is ranked only if it has **at least 10 front-row
starts in that era (about 5 races)**: then one car can change the rate by at most 10 points. A later cell checks that
choosing 6, 8 or 15 instead does not change the answer.

In [ ]:
all_seasons = add_era(q1, 1950)
all_grid_table = front_row_rate(all_seasons, "grid")

plt.figure(figsize=(8, 4))
plt.hist(all_grid_table["starts"], bins=30, color="tab:blue", edgecolor="white", label="circuit-eras")
plt.axvline(MIN_STARTS, color="red", linestyle="--", label="minimum = 10 starts")
plt.title("Front-row starts per circuit and era (grid 1st/2nd, 1950-2013 and 2014-24)")
plt.xlabel("front-row starts")
plt.ylabel("number of circuit-eras")
plt.legend()
plt.show()

small = all_grid_table[~all_grid_table["enough_starts"]]
print("circuit-eras with fewer than", MIN_STARTS, "starts:", len(small), "of", len(all_grid_table))
print("examples (these are left out of the ranking):")
small.sort_values("rate", ascending=False).head(5)

#### Main comparison: 2004-2013 vs 2014-2024

The fairest comparison: grid penalties exist in both eras (so grid and qualifying mean the same thing on both sides),
the eras have a similar length, and qualifying data is complete. Top row = grid, bottom row = qualifying.

In [ ]:
fair = add_era(q1, 2004)
grid_fair = front_row_rate(fair, "grid")
qual_fair = front_row_rate(fair, "qual_or_grid")

print("Front row -> podium, all circuits together:")
print("grid:")
display(overall_rate(fair, "grid").loc[["2004-2013", "2014-24"]].round(3))
print("qualifying:")
display(overall_rate(fair, "qual_or_grid").loc[["2004-2013", "2014-24"]].round(3))

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
plot_top10(axes[0, 0], grid_fair, "2004-2013", "Top 10 circuits, started 1st/2nd (grid), 2004-2013")
plot_top10(axes[0, 1], grid_fair, "2014-24", "Top 10 circuits, started 1st/2nd (grid), 2014-2024")
plot_top10(axes[1, 0], qual_fair, "2004-2013", "Top 10 circuits, qualified 1st/2nd, 2004-2013")
plot_top10(axes[1, 1], qual_fair, "2014-24", "Top 10 circuits, qualified 1st/2nd, 2014-2024")
plt.tight_layout()
plt.show()

**Does the ranking change?** We take the circuits with at least 10 front-row starts in **both** eras and rank them in each era.
Each line is one circuit: going up (green) = better rank after 2014, going down (red) = worse.

In [ ]:
grid_rates, grid_spearman = compare_eras(grid_fair, "2004-2013")
qual_rates, qual_spearman = compare_eras(qual_fair, "2004-2013")

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
slope_chart(axes[0], grid_rates, grid_spearman, "Rank change 2004-13 -> 2014-24, started 1st/2nd (grid)")
slope_chart(axes[1], qual_rates, qual_spearman, "Rank change 2004-13 -> 2014-24, qualified 1st/2nd")
plt.tight_layout()
plt.show()

#### All six comparisons

The same calculation for the three "before" windows (1950-2013, 1994-2013, 2004-2013), each with grid and with
qualifying (`qual_or_grid`), and each compared with **2014-24**.

In [ ]:
# compute the 6 comparisons: 3 "before" windows x 2 positions
rows = []
for start_year in [1950, 1994, 2004]:
    data = add_era(q1, start_year)
    before = str(start_year) + "-2013"
    for position_column in ["grid", "qual_or_grid"]:
        overall = overall_rate(data, position_column)
        rates, spearman = compare_eras(front_row_rate(data, position_column), before)
        rows.append([before, position_column, overall.loc[before, "rate"] * 100,
                     overall.loc["2014-24", "rate"] * 100, spearman])

six = pd.DataFrame(rows, columns=["before", "position", "rate before", "rate 2014-24", "spearman"])
six["position"] = six["position"].replace("qual_or_grid", "qualifying")   # nicer name for the charts

**1) The podium rate: each "before" window next to 2014-24.** One chart per window. In each chart the grey bar is the
window before 2014 and the green bar is 2014-24, for grid and for qualifying.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, before in zip(axes, ["1950-2013", "1994-2013", "2004-2013"]):
    # the 2 rows of this window (grid and qualifying), with the position as row name
    window = six[six["before"] == before].set_index("position")
    # keep the two rates and name them after their years
    pair = window[["rate before", "rate 2014-24"]]
    pair.columns = [before, "2014-24"]

    pair.plot(kind="bar", ax=ax, color=["tab:grey", "tab:green"])
    for bars in ax.containers:
        ax.bar_label(bars, fmt="%.0f", fontsize=9)   # write the value on top of each bar

    # difference between the two grid bars, using the same rounded numbers that are written on the bars
    gain = round(pair.loc["grid", "2014-24"]) - round(pair.loc["grid", before])
    ax.set_title(before + " vs 2014-24\n(grid: +" + str(gain) + " points after 2014)")
    ax.set_xlabel("front row by")
    ax.set_ylabel("% of front-row starters on the podium")
    ax.set_ylim(0, 100)
    ax.tick_params(axis="x", rotation=0)
    ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

**2) The circuit ranking: how similar is each "before" window to 2014-24?** Spearman: 1 = same order, 0 = unrelated.

In [ ]:
spearman_table = six.pivot(index="before", columns="position", values="spearman")

ax = spearman_table.plot(kind="bar", color=["tab:blue", "tab:orange"], figsize=(8, 4.5))
ax.axhline(0, color="black", linewidth=1)
for bars in ax.containers:
    ax.bar_label(bars, fmt="%.2f", fontsize=9)   # write the value on top of each bar
ax.set_title("How similar is the circuit ranking to 2014-24?")
ax.set_xlabel('"before 2014" window')
ax.set_ylabel("Spearman (1 = same order, 0 = unrelated)")
ax.set_ylim(-0.5, 1)
ax.tick_params(axis="x", rotation=0)
plt.show()

#### Why did the front row convert more often?

Every front-row starter ends in one of three ways: **podium**, **did not finish** (the car broke down, crashed or was
disqualified), or **finished 4th or lower**. We also count how often both front-row cars came from the same team.
`outcomes` is used here only to explain the result, never as a model feature.

In [ ]:
# add positionText from outcomes: a number ("1", "2", ...) if the driver was classified, a letter (R = retired...) if not
why = q1.merge(outcomes[["raceId", "driverId", "positionText"]], on=["raceId", "driverId"], how="left")
why = why[(why["grid"] == 1) | (why["grid"] == 2)].copy()

# three spans of years
why["span"] = "1950-2003"
why.loc[why["year"] >= 2004, "span"] = "2004-13"
why.loc[why["year"] >= 2014, "span"] = "2014-24"

# put every front-row starter into one of three groups
why["result"] = "finished 4th or lower"
why.loc[~why["positionText"].str.isnumeric(), "result"] = "did not finish"
why.loc[why["podium"] == 1, "result"] = "podium"

# % of each group per span (normalize="index" makes every row add up to 100%)
result_share = pd.crosstab(why["span"], why["result"], normalize="index") * 100
result_share = result_share[["podium", "finished 4th or lower", "did not finish"]]
display(result_share.round(1))

result_share.plot(kind="bar", stacked=True, color=["tab:green", "tab:orange", "tab:red"], figsize=(7, 4))
plt.title("What happened to the front-row starters (grid 1st/2nd)?")
plt.xlabel("years")
plt.ylabel("% of front-row starters")
plt.xticks(rotation=0)
plt.legend(title="result", bbox_to_anchor=(1, 1))
plt.show()

# how often were both front-row cars from the same team?
per_race = why.groupby(["raceId", "span"]).agg(cars=("constructorId", "count"), teams=("constructorId", "nunique"))
per_race = per_race.reset_index()
per_race = per_race[per_race["cars"] == 2]
per_race["same_team"] = per_race["teams"] == 1
print("% of races where both front-row cars were from the same team:")
print((per_race.groupby("span")["same_team"].mean() * 100).round(1))

#### Check: does our choice of 10 starts change the answer?

The minimum of 10 starts is our own decision. We repeat the main comparison (2004-13 vs 2014-24) with 6, 8 and 15
to see whether the answer (how similar the rankings are, and which circuits are on top after 2014) stays the same.

In [ ]:
rows = []
for min_starts in [6, 8, 10, 15]:
    for position_column, table in [("grid", grid_fair), ("qual_or_grid", qual_fair)]:
        rates, spearman = compare_eras(table, "2004-2013", min_starts=min_starts)
        top5 = rates.sort_values("2014-24", ascending=False).head(5).index
        rows.append([position_column, min_starts, len(rates), round(spearman, 2), ", ".join(top5)])

check = pd.DataFrame(rows, columns=["position", "min starts", "circuits", "spearman", "top 5 in 2014-24"])
with pd.option_context("display.max_colwidth", None):
    display(check)

#### Answer to Q1

**1. Best circuits in the hybrid era (2014-24): Monaco and Suzuka.** About 9 in 10 front-row starters finished on the podium
there: Monaco 17 of 19, Suzuka 16 of 18 (89%). They are the **top 2 in all six comparisons**, with grid and with qualifying.
Next come Yas Marina, Barcelona and Spa (82-86%).

**2. Main comparison, 2004-13 vs 2014-24 (the fairest).** The front row converts more often after 2014:

| Front row by | 2004-13 | 2014-24 |
|---|---|---|
| grid (started 1st/2nd) | 69% (254 of 366) | **76%** (345 of 453) |
| qualifying (qualified 1st/2nd) | 69% (251 of 366) | **74%** (336 of 454) |

The ranking is **only partly the same**. Spearman is **0.28 (grid)** and **0.45 (qualifying)**, for 14 circuits.
- **Moved up:** Yas Marina (60% → 86%), Barcelona, Interlagos, Canada.
- **Moved down:** Monza, Silverstone, Bahrain, Hungaroring.
- Monza and Silverstone barely changed their own rate (75% → 73%, 75% → 75% on grid). They dropped because other circuits rose.
- With a minimum of 6, 8 or 15 starts, Spearman stays positive (0.09-0.39 grid, 0.24-0.54 qualifying), and Monaco and Suzuka
  stay the top 2.

**3. The further back "before" goes, the more different the ranking looks.**

| Before | Spearman grid | Spearman qualifying |
|---|---|---|
| 2004-2013 | 0.28 | 0.45 |
| 1994-2013 | -0.04 | -0.00 |
| 1950-2013 | -0.08 | -0.11 |

Compared with the last decade, part of the order is kept. Compared with all of F1 history, the old order says nothing about the new one.

**4. Why the front row converts more often.**
- Over the long run, front-row cars stopped breaking down: **31%** did not finish in 1950-2003, **10%** in 2004-13, **7%** in 2014-24.
- Between 2004-13 and 2014-24, the share that finished but lost the podium on track also fell (20% → 17%).
- Both front-row cars came from the same team more often (32% → 44% of races). One dominant team kept both spots.

**5. Grid vs qualifying.**
- In 1994-2003 the grid was the qualifying order. 98.5% of drivers started where they qualified, and all 164 front-row
  qualifiers started on the front row.
- Grid penalties start in 2004. Drivers per race who started 3+ places behind their qualifying spot: **0.01** (1994-2003),
  **0.76** (2004-13), **1.05** (2014-24)
  (the 3-4 and 5+ columns of the places-lost table added together).
- On the front row, 0 → 9 → 28 qualifiers were pushed back. Example: Verstappen, Belgium 2024, qualified 1st and started 11th.
- This is why the grid and qualifying answers are identical before 2004 and differ a little after.
- Qualifying gives a slightly lower rate after 2014 (74% vs 76%), because a fast car pushed off the front row still counts
  in qualifying.

**Grain**
- Input: one row per driver per race (`model_base`).
- Front-row subset: 2 rows per race (position 1 and 2).
- Result tables: one row per circuit and era.
- The circuits join is many-to-one, and the row count is checked to stay the same.

**Missingness**
- `qual_position` is empty before 1994 and patchy in 1996-2002 (see the coverage chart).
- `qual_or_grid` fills these gaps with the grid, **only before 2004**, when grid = qualifying order (checked: 164 of 164
  front-row qualifiers started on the front row).
- 21 missing values from 2004 on are not filled.
- `grid` has no gaps. `grid = 0` (pit-lane start) is not counted as front row.

**Pre-race cutoff**
- `grid` and `qual_position` are both known before the race starts.
- `podium` is the outcome we measure, not an input.
- `outcomes` (positionText) is used only to explain why the rate rose, never as an input.

**Limitations**
- Some circuits hosted only a few races in an era (fewer than 10 front-row starts), so they are left out of the ranking.
- One `circuitId` covers all layout changes of a track (e.g. Silverstone, Spa).
- Circuits come and go. Magny-Cours and Istanbul topped 2004-13 but are not on the calendar after 2014.
- Shared drives (two drivers, one car) exist only before 1965 and are counted once, with the best finish.

#### Backup: longer "before" windows and four periods

Extra detail, not needed for the answer above.

In [ ]:
# rank change for the two longer windows: 1994-2013 and 1950-2013 (top = grid, bottom = qualifying)
fig, axes = plt.subplots(2, 2, figsize=(15, 14))
for column, start_year in enumerate([1994, 1950]):
    data = add_era(q1, start_year)
    before = str(start_year) + "-2013"
    rates, spearman = compare_eras(front_row_rate(data, "grid"), before)
    slope_chart(axes[0, column], rates, spearman, "Rank change " + before + " -> 2014-24, grid")
    rates, spearman = compare_eras(front_row_rate(data, "qual_or_grid"), before)
    slope_chart(axes[1, column], rates, spearman, "Rank change " + before + " -> 2014-24, qualifying")
plt.tight_layout()
plt.show()

In [ ]:
# four periods: the rate for all circuits together (grid)
q1["period"] = "1950-93"
q1.loc[q1["year"] >= 1994, "period"] = "1994-2003"
q1.loc[q1["year"] >= 2004, "period"] = "2004-13"
q1.loc[q1["year"] >= 2014, "period"] = "2014-24"

grid_front = q1[(q1["grid"] == 1) | (q1["grid"] == 2)]
by_period = grid_front.groupby("period")["podium"].agg(starts="count", podiums="sum")
by_period["rate"] = by_period["podiums"] / by_period["starts"]
display(by_period.round(3))

plt.figure(figsize=(7, 4))
plt.bar(by_period.index, by_period["rate"] * 100, color="tab:blue", label="all circuits together")
plt.ylim(0, 100)
plt.title("Front row (grid 1st/2nd) -> podium, per period")
plt.xlabel("period")
plt.ylabel("% of front-row starters on the podium")
plt.legend(loc="upper left")
plt.show()

In [ ]:
# heatmap: rate per circuit and period, only where the circuit had at least 10 front-row starts
grid_front = q1[(q1["grid"] == 1) | (q1["grid"] == 2)]
period_table = grid_front.groupby(["circuit", "period"])["podium"].agg(starts="count", podiums="sum").reset_index()
period_table["rate"] = period_table["podiums"] / period_table["starts"]
period_table = period_table[period_table["starts"] >= MIN_STARTS]
heat = period_table.pivot(index="circuit", columns="period", values="rate") * 100

# keep circuits that appear in at least 2 periods, best in 2014-24 at the top
heat = heat[heat.notna().sum(axis=1) >= 2]
heat = heat.sort_values("2014-24", ascending=False)

plt.figure(figsize=(8, 9))
sns.heatmap(heat, annot=True, fmt=".0f", cmap="Greens", vmin=0, vmax=100,
            cbar_kws={"label": "% of front-row starters on the podium"})
plt.title("Front row (grid 1st/2nd) -> podium per circuit and period\n(blank = fewer than 10 starts)")
plt.xlabel("period")
plt.ylabel("circuit")
plt.show()

### Q2: Do drivers get more podiums at their home circuit?

**In plain words:** does a British driver racing in Britain finish on the podium more often than when racing elsewhere?

**The catch:** if home drivers simply start further forward, they will of course finish better. So we compare
**drivers who started from the same grid row**: a home driver who started 3rd-4th is compared with away drivers who
also started 3rd-4th.

**Tables and joins (multi-hop):**

```
results ──driverId──► drivers                              (nationality of the driver)
   │
   └──raceId──► races ──circuitId──► circuits              (country of the circuit)
```

- `results ⋈ races` is already done inside `load_clean()` → `model_base` (`grid`, `podium`, `year`, `circuitId`).
- Here we add `drivers` (on `driverId`) for the nationality and `circuits` (on `circuitId`) for the country.
- Nationality ("British") and country ("UK") are written differently, so we connect them with a mapping written by hand.
- Not used: qualifying (we control with `grid`, which exists for every race since 1950), standings, lap times, pit stops.
- All seasons, **1950-2024**: nationality, country and grid are never missing.

In [ ]:
# driver nationality (from drivers.csv) -> country name as written in circuits.csv, written by hand
NATIONALITY_TO_COUNTRY = {
    "American": "USA",
    "American-Italian": "USA", 
    "Argentine": "Argentina", 
    "Argentine-Italian": "Argentina",
    "Argentinian": "Argentina", 
    "Australian": "Australia",
    "Austrian": "Austria", 
    "Belgian": "Belgium",
    "Brazilian": "Brazil",
    "British": "UK",
    "Canadian": "Canada", 
    "Chilean": "Chile", 
    "Chinese": "China",
    "Colombian": "Colombia", 
    "Czech": "Czech Republic", 
    "Danish": "Denmark", "Dutch": "Netherlands",
    "East German": "Germany",
    "Finnish": "Finland", 
    "French": "France",
    "German": "Germany",
    "Hungarian": "Hungary",
    "Indian": "India",
    "Indonesian": "Indonesia",
    "Irish": "Ireland", 
    "Italian": "Italy", 
    "Japanese": "Japan",
    "Liechtensteiner": "Liechtenstein",
    "Malaysian": "Malaysia", 
    "Mexican": "Mexico", 
    "Monegasque": "Monaco",
    "New Zealander": "New Zealand", 
    "Polish": "Poland", 
    "Portuguese": "Portugal",
    "Rhodesian": "Rhodesia",
    "Russian": "Russia", 
    "South African": "South Africa",
    "Spanish": "Spain",
    "Swedish": "Sweden",
    "Swiss": "Switzerland",
    "Thai": "Thailand",
    "Uruguayan": "Uruguay",
    "Venezuelan": "Venezuela",
}

# drivers with two nationalities have a second home country
SECOND_COUNTRY = {"American-Italian": "Italy", 
                  "Argentine-Italian": "Italy"}

# show the mapping as a table (shared with P3 for the home-race feature)
mapping = pd.DataFrame(list(NATIONALITY_TO_COUNTRY.items()), columns=["nationality", "home_country"])
mapping["second_home_country"] = mapping["nationality"].map(SECOND_COUNTRY)
mapping

In [ ]:
# 1) driver nationality; .str.strip() removes spaces at the ends ("Argentinian " -> "Argentinian")
drivers_nat = tables["drivers"][["driverId", "nationality"]].copy()
drivers_nat["nationality"] = drivers_nat["nationality"].str.strip() #  `strip()` removes spaces at the start and end

# 2) circuit country; "United States" and "USA" are the same country written two ways
circuit_country = tables["circuits"][["circuitId", "name", "country"]].rename(columns={"name": "circuit"})
circuit_country["country"] = circuit_country["country"].replace("United States", "USA")

# 3) joins: every row of base gets the driver's nationality and the circuit's country
q2 = base.merge(drivers_nat, on="driverId", how="left", validate="many_to_one")
q2 = q2.merge(circuit_country, on="circuitId", how="left", validate="many_to_one")

# 4) safety checks
assert len(q2) == len(base), "a join changed the number of rows"
assert q2["nationality"].notna().all(), "a driver has no nationality"
assert q2["country"].notna().all(), "a circuit has no country"
unmapped = set(q2["nationality"]) - set(NATIONALITY_TO_COUNTRY)
assert len(unmapped) == 0, "nationalities missing from the mapping: " + str(unmapped)

# 5) home = the circuit is in the driver's (first or second) home country
q2["home_country"] = q2["nationality"].map(NATIONALITY_TO_COUNTRY) # the mapping is happening here, not in the join, 
q2["second_home_country"] = q2["nationality"].map(SECOND_COUNTRY) # because some drivers have two nationalities and a second home country
q2["is_home"] = (q2["country"] == q2["home_country"]) | (q2["country"] == q2["second_home_country"])

print("rows:", len(q2), "| home rows:", q2["is_home"].sum(), "| away rows:", (~q2["is_home"]).sum())

#### Demo: how the Q2 joins work, step by step

Six real rows, one for each case: a normal home race, a normal away race, two drivers with two nationalities,
a circuit written "United States", and a nationality with a hidden space. Then two fake rows show what would happen
if a driver had no nationality or a circuit had no country. These cells do not change `q2`.

In [ ]:
# DEMO (only to understand the joins): 6 real rows, one for each case. Nothing here changes q2.

# the 6 example rows: (raceId, driverId)
examples = [
    (1132, 1),     # Hamilton (British) at the British GP 2024        -> home
    (1136, 1),     # Hamilton (British) at the Italian GP 2024        -> away
    (754, 491),    # Thiele (American-Italian) at the Italian GP 1960 -> home by his SECOND country
    (776, 573),    # de Tomaso (Argentine-Italian) at the Argentine GP 1957 -> home by his FIRST country
    (1119, 858),   # Sargeant (American) at Las Vegas 2023: circuit written "United States"
    (1136, 861),   # Colapinto ("Argentinian " with a space) at the Italian GP 2024 -> away
]
example_pairs = pd.DataFrame(examples, columns=["raceId", "driverId"])

# STEP 0: the three tables alone (only the rows we need)
demo_base = base.merge(example_pairs, on=["raceId", "driverId"])[["raceId", "driverId", "circuitId", "year", "grid", "podium"]]
print("STEP 0a: BASE (model_base), our 6 rows")
display(demo_base)

demo_drivers = tables["drivers"][tables["drivers"]["driverId"].isin(demo_base["driverId"])][["driverId", "forename", "surname", "nationality"]].copy()
demo_drivers["nationality in quotes"] = '"' + demo_drivers["nationality"] + '"'   # quotes make the hidden space visible
print("STEP 0b: DRIVERS table, only these drivers (before .str.strip())")
display(demo_drivers)

demo_circuits = tables["circuits"][tables["circuits"]["circuitId"].isin(demo_base["circuitId"])][["circuitId", "name", "country"]]
print("STEP 0c: CIRCUITS table, only these circuits (before the 'United States' -> 'USA' fix)")
display(demo_circuits)

# STEP 1: base + drivers, matched on driverId -> every row gets the nationality (already stripped)
step1 = demo_base.merge(drivers_nat, on="driverId", how="left", validate="many_to_one")
print("STEP 1: base + drivers (matched on driverId)")
display(step1)

# STEP 2: + circuits, matched on circuitId -> every row gets the circuit name and its country (already fixed)
step2 = step1.merge(circuit_country, on="circuitId", how="left", validate="many_to_one")
print("STEP 2: + circuits (matched on circuitId)")
display(step2)

# STEP 3: home countries and is_home, exactly like the real cell
step2["home_country"] = step2["nationality"].map(NATIONALITY_TO_COUNTRY)
step2["second_home_country"] = step2["nationality"].map(SECOND_COUNTRY)
step2["is_home"] = (step2["country"] == step2["home_country"]) | (step2["country"] == step2["second_home_country"])
print("STEP 3: home country (first and second) and is_home")
display(step2[["driverId", "nationality", "circuit", "country", "home_country", "second_home_country", "is_home"]])

In [ ]:
# mapping check: which countries never meet?
race_countries = set(q2["country"])
driver_countries = set(q2["home_country"]) | set(q2["second_home_country"].dropna())

print("driver countries that never hosted a race (their drivers are always away):")
print(sorted(driver_countries - race_countries))
print("race countries with no driver from that country (nobody is ever at home there):")
print(sorted(race_countries - driver_countries))
print()
print("home starts per country (top 10):")
q2[q2["is_home"]]["country"].value_counts().head(10)

#### Grid groups

We group the starting positions by **grid row**. On an F1 grid the cars stand in rows of two (1st and 2nd side by side,
then 3rd and 4th, ...), so each group is one row: **1-2, 3-4, 5-6, ... 19-20**. Everything further back, and pit-lane
starts (`grid = 0`), go into the last group **"21+ / pit lane"**. Each group has about 150 home starts, enough to compare.
(The work split suggested 1 / 2-3 / 4-6 / 7-10 / 11+; grid rows give the same answer with a finer view.)

In [ ]:
# the groups in grid order (used to sort tables and charts)
GRID_GROUPS = ["1-2", "3-4", "5-6", "7-8", "9-10", "11-12", "13-14", "15-16", "17-18", "19-20", "21+ / pit lane"]

# every row starts in the last group (grid 21 or further back, or pit lane = grid 0)
q2["grid_group"] = "21+ / pit lane"

# then each pair of grid positions gets its own group
q2.loc[q2["grid"].between(1, 2), "grid_group"] = "1-2"
q2.loc[q2["grid"].between(3, 4), "grid_group"] = "3-4"
q2.loc[q2["grid"].between(5, 6), "grid_group"] = "5-6"
q2.loc[q2["grid"].between(7, 8), "grid_group"] = "7-8"
q2.loc[q2["grid"].between(9, 10), "grid_group"] = "9-10"
q2.loc[q2["grid"].between(11, 12), "grid_group"] = "11-12"
q2.loc[q2["grid"].between(13, 14), "grid_group"] = "13-14"
q2.loc[q2["grid"].between(15, 16), "grid_group"] = "15-16"
q2.loc[q2["grid"].between(17, 18), "grid_group"] = "17-18"
q2.loc[q2["grid"].between(19, 20), "grid_group"] = "19-20"

# count home and away starts in each group
group_counts = pd.crosstab(q2["grid_group"], q2["is_home"])                              # count each (group, home/away) pair
group_counts = group_counts.loc[GRID_GROUPS]                                             # put the groups in grid order
group_counts = group_counts.rename(columns={False: "away starts", True: "home starts"})  # readable column names
group_counts

#### Without grid groups vs with grid groups

First the plain comparison (all home starts vs all away starts). Then the fair comparison inside each grid row.
**"difference"** = home % − away %: how many percentage points higher the home rate is in that row
(for example 45% home and 40% away gives +5; a negative number means away is higher).

In [ ]:
# plain comparison, no grid groups

# split the rows into two groups
home_rows = q2[q2["is_home"]]       # only home races
away_rows = q2[~q2["is_home"]]      # not home races      # only away races (~ means NOT)

# build a small table: one row for home, one row for away
plain = pd.DataFrame({
    "starts": [len(home_rows), len(away_rows)],                          # how many races
    "podiums": [home_rows["podium"].sum(), away_rows["podium"].sum()],   # how many podiums (podium is 1 or 0)
}, index=["home", "away"])

# podium % = podiums / starts * 100
plain["podium %"] = plain["podiums"] / plain["starts"] * 100
plain.round(1)

In [ ]:
# fair comparison: podium rate for home and away inside each grid group
# (home_rows and away_rows come from the plain-comparison cell above)

# count starts and podiums per grid group, separately for home and away
home_away = pd.DataFrame({
    "home starts": home_rows.groupby("grid_group")["podium"].count(),    # how many home races in each group
    "home podiums": home_rows.groupby("grid_group")["podium"].sum(),     # how many of them ended on the podium
    "away starts": away_rows.groupby("grid_group")["podium"].count(),
    "away podiums": away_rows.groupby("grid_group")["podium"].sum(),
})

# put the groups in grid order (1-2, 3-4, ... instead of alphabetical)
home_away = home_away.loc[GRID_GROUPS]

# % = podiums / starts * 100, then difference = home % minus away %
home_away["home %"] = home_away["home podiums"] / home_away["home starts"] * 100
home_away["away %"] = home_away["away podiums"] / home_away["away starts"] * 100
home_away["difference"] = home_away["home %"] - home_away["away %"]
home_away.round(1)

In [ ]:
# grouped bar chart: home (orange) next to away (blue) for each grid row
home_away[["home %", "away %"]].plot(kind="bar", color=["tab:orange", "tab:blue"], figsize=(14, 5))

plt.title("Podium rate at home vs away, for drivers who started from the same grid row (1950-2024)")
plt.xlabel("starting position (grid row)")
plt.ylabel("% of drivers who finished on the podium")
plt.ylim(0, 100)                                       # y axis from 0 to 100%
plt.xticks(rotation=0)                                 # keep the group names horizontal
plt.legend(["home (racing in own country)", "away"])   # legend names, same order as the columns
plt.show()

#### One number for all grid groups: 3 methods

Each method turns the 11 differences into one answer. If all three say "about 0", there is no home advantage.

1. **Simple average:** the average of the 11 differences in the table (every grid row counts the same).
2. **Same driver:** each driver at home vs the same driver away, from the same grid row.
3. **Exact grid position:** 1st vs 1st, 2nd vs 2nd ... 20th vs 20th, instead of grid rows.

**How the numbers are calculated, step by step (numbers from the table above):**

**Step 1: the % in each row** = podiums ÷ starts × 100. A sample of three rows:

| grid row | home starts | home podiums | home % | away starts | away podiums | away % |
|---|---|---|---|---|---|---|
| 1-2 | 160 | 96 | 96 ÷ 160 × 100 = **60.0** | 2067 | 1254 | 1254 ÷ 2067 × 100 = **60.7** |
| 3-4 | 144 | 65 | 65 ÷ 144 × 100 = **45.1** | 2080 | 838 | 838 ÷ 2080 × 100 = **40.3** |
| 9-10 | 180 | 8 | 8 ÷ 180 × 100 = **4.4** | 2033 | 145 | 145 ÷ 2033 × 100 = **7.1** |

**Step 2: the difference in each row** = home % − away %:

| grid row | home % − away % | difference | meaning |
|---|---|---|---|
| 1-2 | 60.0 − 60.7 | **−0.7** | away slightly higher |
| 3-4 | 45.1 − 40.3 | **+4.9** | home higher |
| 9-10 | 4.4 − 7.1 | **−2.7** | away higher |

In [ ]:
# METHOD 1: simple average of the 11 differences (every grid row counts the same)
average_difference = home_away["difference"].mean()
print("average difference over the 11 grid rows:", round(average_difference, 2), "percentage points")

In [ ]:
# METHOD 2: same driver - each driver at home vs the SAME driver away, from the same grid row

# for every driver in every grid row: home starts, home podium %, away podium %
driver_home_starts = home_rows.groupby(["driverId", "grid_group"])["podium"].count()
driver_home = home_rows.groupby(["driverId", "grid_group"])["podium"].mean() * 100
driver_away = away_rows.groupby(["driverId", "grid_group"])["podium"].mean() * 100

# put them side by side; keep only rows where the driver has BOTH a home and an away start
same_driver = pd.DataFrame({"home starts": driver_home_starts, "home %": driver_home, "away %": driver_away})
same_driver = same_driver.dropna()
same_driver["difference"] = same_driver["home %"] - same_driver["away %"]

print("driver + grid row pairs compared:", len(same_driver))
print("average difference (same driver):", round(same_driver["difference"].mean(), 2), "percentage points")

# example: Lewis Hamilton (driverId 1)
same_driver.loc[1].round(1)

In [ ]:
# METHOD 3: exact grid position (1st vs 1st, 2nd vs 2nd, ... 20th vs 20th) instead of grid rows

# only grid positions 1 to 20
home_1_20 = home_rows[home_rows["grid"].between(1, 20)]
away_1_20 = away_rows[away_rows["grid"].between(1, 20)]

# podium % for every grid position, at home and away
by_position = pd.DataFrame({
    "home %": home_1_20.groupby("grid")["podium"].mean() * 100,
    "away %": away_1_20.groupby("grid")["podium"].mean() * 100,
})
by_position["difference"] = by_position["home %"] - by_position["away %"]

print("average difference over grid positions 1-20:", round(by_position["difference"].mean(), 2), "percentage points")
by_position.round(1)

#### Answer to Q2

**No, drivers do not get more podiums at home** once we compare drivers who started from the same grid row.

- **Without grid groups**, home even looks worse: 11.9% podiums at home vs 13.8% away. But home drivers start further back
  more often: 15% of home starts are from 21st or further back / the pit lane, vs 11% of away starts
  (291 of 1932 vs 2542 of 22634, see the grid-group table). Many local drivers entered only their home race.
- **With grid groups** (the fair comparison), home and away are almost the same in every row:
  - front row (1-2): 60.0% home vs 60.7% away
  - 5-6: 21.7% vs 21.9%
  - home is clearly higher in only 2 of 11 rows (3-4: +4.9, 11-12: +1.8)
- **One number, 3 methods** (difference = home % − away %, in percentage points):

  | Method | Result |
  |---|---|
  | 1. Simple average of the 11 grid rows | **−0.24** |
  | 2. Same driver (each driver at home vs himself away, same grid row) | **+0.66** |
  | 3. Exact grid position (1st vs 1st ... 20th vs 20th) | **−0.20** |

  All three are less than 1 percentage point, so home makes practically no difference, even when each driver is
  compared with himself.
- The +4.9 in row 3-4 comes from only 144 home starts (65 podiums). It is alone among the other rows, so it is not a home advantage.

**Grain**
- Input: one row per driver per race (`model_base`).
- Result: one row per grid row (11 groups), with home and away side by side.
- Method 2: one row per driver and grid row (999 pairs). Method 3: one row per grid position (1-20).
- Both joins (`drivers`, `circuits`) are many-to-one, and the row count is checked to stay the same.

**Missingness**
- Nationality, circuit country and grid are never missing, so all seasons 1950-2024 are used.
- Every nationality is in the mapping (checked with an assert).
- 14 driver countries never hosted a race (e.g. Finland, New Zealand), so their drivers are always "away".
- 9 race countries never had a driver from there (e.g. Bahrain, Singapore), so nobody is "home" there.
- Method 2 keeps only drivers who started from the same grid row both at home and away (`dropna`).
  Method 3 uses grid 1-20 only.

**Pre-race cutoff**
- Nationality, circuit country and grid are all known before the race starts.
- `podium` is the outcome we measure, not an input.
- No column from `outcomes` is used.

**Limitations**
- Nationality is not the same as where a driver lives or grew up.
- Only 2 dual nationalities exist in the data. Both of their countries count as home.
- Most home starts are British (446) and Italian (400), so the answer mostly describes those drivers.
- In method 2 most driver + grid row pairs have only 1-2 home starts, so one race changes their % a lot.

### Q3: Which teams' cars break down the most?

**In plain words:** for each team, what share of its race starts ended because the car broke? We compare **2014-2021**
with **2022-2024**.

**Mechanical retirement** = a race start that ended early because **a part of the car failed** (engine, gearbox, brakes,
hydraulics, ...). Not counted: crashes, driver mistakes, and rule decisions (for example a disqualification).

**Where the data comes from:** everything comes from `load_clean()`.
- `model_base`: who started which race, for which team, in which year.
- `outcomes`: how each race ended (`statusId`). The team rule says `outcomes` is *only for looking at the data, never as
  model input*. Q3 only counts how races ended and builds no model, so this is exactly that allowed use.
- `tables["status"]`: the 139 status names, which we label by hand.
- `tables["constructors"]`: the team names.

**Tables and joins (multi-hop):**

```
model_base ──(raceId, driverId)──► outcomes ──statusId──► status        (how did the race end?)
     │
     └──constructorId──► constructors                                     (which team?)
```

`results → races` (for the year) is already done inside `load_clean()`. Not used: sprint results (a different, shorter
race), lap times, pit stops, standings.

In [ ]:
# how each status ended the race, written by hand from the status table (139 values)
ACCIDENT = ["Accident", "Collision", "Collision damage", "Damage", "Debris", "Fatal accident",
            "Puncture", "Tyre puncture", "Front wing", "Rear wing", "Broken wing", "Undertray"]
DRIVER = ["Spun off", "Stalled", "Illness", "Injured", "Injury", "Physical", "Driver unwell", "Eye injury"]
OTHER = ["Disqualified", "Excluded", "Underweight", "Retired", "Not classified", "Withdrew", "Did not qualify",
         "Did not prequalify", "107% Rule", "Not restarted", "Safety", "Safety concerns", "Out of fuel",
         "Refuelling", "Fuel rig", "Wheel nut"]

status_labels = tables["status"][["statusId", "status"]].copy()

# every status starts as "mechanical" (a part of the car failed), then the lists above overwrite it
status_labels["label"] = "mechanical"
status_labels.loc[status_labels["status"].isin(ACCIDENT), "label"] = "accident"
status_labels.loc[status_labels["status"].isin(DRIVER), "label"] = "driver"
status_labels.loc[status_labels["status"].isin(OTHER), "label"] = "other"

# finished the race: "Finished", or "+1 Lap", "+2 Laps", ... (classified, but some laps behind the winner)
finished = (status_labels["status"] == "Finished") | status_labels["status"].str.startswith("+")
status_labels.loc[finished, "label"] = "finished"

# check: every name in the lists really exists in the status table (catches typos)
names_in_lists = set(ACCIDENT + DRIVER + OTHER)
names_in_table = set(status_labels["status"])
assert names_in_lists <= names_in_table, "not in the status table: " + str(names_in_lists - names_in_table)

print("statuses labelled:", len(status_labels))
status_labels["label"].value_counts()

In [ ]:
# the full label table, one line per label (this is the documented mapping)
for label in ["mechanical", "accident", "driver", "other", "finished"]:
    names = status_labels[status_labels["label"] == label]["status"].tolist()
    print(label.upper(), "(" + str(len(names)) + "):", ", ".join(names))
    print()

#### Team renames

Some teams changed their name but stayed the same team (same factory, same staff). We put each name under one
**team history**, so a team's breakdowns are not split over two or three names.

In [ ]:
# team name in constructors.csv -> team history, written by hand (teams not listed keep their own name)
TEAM_HISTORY = {
    "Toro Rosso": "Toro Rosso / AlphaTauri / RB",
    "AlphaTauri": "Toro Rosso / AlphaTauri / RB",
    "RB F1 Team": "Toro Rosso / AlphaTauri / RB",
    "Force India": "Force India / Racing Point / Aston Martin",
    "Racing Point": "Force India / Racing Point / Aston Martin",
    "Aston Martin": "Force India / Racing Point / Aston Martin",
    "Lotus F1": "Lotus / Renault / Alpine",
    "Renault": "Lotus / Renault / Alpine",
    "Alpine F1 Team": "Lotus / Renault / Alpine",
    "Sauber": "Sauber / Alfa Romeo",
    "Alfa Romeo": "Sauber / Alfa Romeo",
    "Marussia": "Marussia / Manor",
    "Manor Marussia": "Marussia / Manor",
}
pd.DataFrame(list(TEAM_HISTORY.items()), columns=["name in the data", "team history"])

**Check: are these really the same team?** For every renamed team we print the seasons and the drivers. The old name
stops exactly when the new one starts, and the same drivers appear under both names (for example Gasly at Toro Rosso
and AlphaTauri, Pérez at Force India and Racing Point, Ocon at Renault and Alpine, Bottas and Zhou at Alfa Romeo and Sauber).

In [ ]:
# seasons and drivers of every renamed team, 2014-2024
check = base[base["year"] >= 2014]
check = check.merge(tables["constructors"][["constructorId", "name"]], on="constructorId", how="left")
check = check.merge(tables["drivers"][["driverId", "surname"]], on="driverId", how="left")

for name in TEAM_HISTORY:
    rows = check[check["name"] == name]
    seasons = sorted(set(rows["year"].tolist()))     # .tolist() gives plain numbers (2014 instead of np.int64(2014))
    drivers = sorted(set(rows["surname"].tolist()))
    print(TEAM_HISTORY[name], "|", name, "| seasons:", seasons, "| drivers:", ", ".join(drivers))

#### Build the Q3 table (2014-2024)

In [ ]:
# 1) who started which race for which team (2014-2024)
q3 = base[base["year"] >= 2014][["raceId", "driverId", "constructorId", "year"]]

# 2) joins: how the race ended (statusId) -> its status name and label -> the team name
q3 = q3.merge(outcomes[["raceId", "driverId", "statusId"]], on=["raceId", "driverId"], how="left", validate="one_to_one")
q3 = q3.merge(status_labels, on="statusId", how="left", validate="many_to_one")
q3 = q3.merge(tables["constructors"][["constructorId", "name"]], on="constructorId", how="left", validate="many_to_one")

# 3) safety checks
assert len(q3) == (base["year"] >= 2014).sum(), "a join changed the number of rows"
assert q3["label"].notna().all(), "a race end has no label"
assert q3["name"].notna().all(), "a team has no name"

# 4) team history (renames merged) and period
q3["team"] = q3["name"].replace(TEAM_HISTORY)
q3["period"] = "2014-21"
q3.loc[q3["year"] >= 2022, "period"] = "2022-24"

print("race starts:", len(q3), "| teams:", q3["team"].nunique())
print(q3.groupby("period")["raceId"].nunique())

#### Demo: how the Q3 joins work, step by step

Six real rows, one for each kind of race end, including two renamed teams (Aston Martin, AlphaTauri). Then two fake
rows show what would happen if a status or a team could not be found. These cells do not change `q3`.

In [ ]:
# DEMO (only to understand the joins): 6 real rows. Nothing here changes q3.
examples = [
    (963, 1),      # Hamilton, Malaysian GP 2016       -> Engine        -> mechanical
    (952, 3),      # Rosberg, Spanish GP 2016          -> Collision     -> accident
    (1076, 832),   # Sainz, Australian GP 2022         -> Spun off      -> driver
    (1062, 20),    # Vettel (Aston Martin), Hungary 2021 -> Disqualified -> other
    (1038, 842),   # Gasly (AlphaTauri), Italian GP 2020 -> Finished    -> finished (he won)
    (1137, 815),   # Pérez, Azerbaijan GP 2024         -> Retired       -> other (no reason given)
]
example_pairs = pd.DataFrame(examples, columns=["raceId", "driverId"])

# STEP 0: the tables alone (only the rows we need)
demo_base = base.merge(example_pairs, on=["raceId", "driverId"])[["raceId", "driverId", "constructorId", "year"]]
print("STEP 0a: BASE (model_base), our 6 rows")
display(demo_base)

demo_outcomes = outcomes.merge(example_pairs, on=["raceId", "driverId"])[["raceId", "driverId", "statusId"]]
print("STEP 0b: OUTCOMES, the same 6 rows (only statusId)")
display(demo_outcomes)

print("STEP 0c: STATUS_LABELS, only these statuses")
display(status_labels[status_labels["statusId"].isin(demo_outcomes["statusId"])])

print("STEP 0d: CONSTRUCTORS, only these teams")
display(tables["constructors"][tables["constructors"]["constructorId"].isin(demo_base["constructorId"])][["constructorId", "name"]])

# STEP 1: base + outcomes, matched on raceId AND driverId -> every row gets its statusId
step1 = demo_base.merge(outcomes[["raceId", "driverId", "statusId"]], on=["raceId", "driverId"], how="left")
print("STEP 1: base + outcomes (matched on raceId and driverId)")
display(step1)

# STEP 2: + status_labels, matched on statusId -> every row gets the status name and its label
step2 = step1.merge(status_labels, on="statusId", how="left")
print("STEP 2: + status_labels (matched on statusId)")
display(step2)

# STEP 3: + constructors, matched on constructorId -> team name, then team history and period
step3 = step2.merge(tables["constructors"][["constructorId", "name"]], on="constructorId", how="left")
step3["team"] = step3["name"].replace(TEAM_HISTORY)
step3["period"] = "2014-21"
step3.loc[step3["year"] >= 2022, "period"] = "2022-24"
print("STEP 3: + constructors (matched on constructorId), team history and period")
display(step3)

In [ ]:
# DEMO: what if a status or a team cannot be found? The real data has no such rows, so we make 2 fake ones.
fake = pd.DataFrame({
    "raceId": [1, 2],
    "constructorId": [131, 99999],   # row 1: Mercedes (exists) | row 2: a team that does NOT exist
    "statusId": [99999, 5],          # row 1: a status that does NOT exist | row 2: Engine (exists)
})
fake = fake.merge(status_labels, on="statusId", how="left")
fake = fake.merge(tables["constructors"][["constructorId", "name"]], on="constructorId", how="left")
display(fake)

print("row 1: the status is not found -> label is NaN (empty), so this start would never count as mechanical")
print("row 2: the team is not found -> name is NaN, so this breakdown would belong to no team")
print("how='left' keeps both rows, but they would silently give wrong counts.")
print("That is why the real cell has the asserts: they stop the notebook instead.")

#### How did the races end?

Before the team rates, the big picture: how many starts in each period ended in each way.

In [ ]:
# count of starts per period and label
ended = pd.crosstab(q3["period"], q3["label"])
ended = ended[["finished", "mechanical", "accident", "driver", "other"]]
display(ended)

# the same as % of starts (normalize="index" makes every row add up to 100%)
ended_pct = pd.crosstab(q3["period"], q3["label"], normalize="index") * 100
ended_pct = ended_pct[["finished", "mechanical", "accident", "driver", "other"]]
ended_pct.round(1)

#### Mechanical retirement rate per team

**Rate** = mechanical retirements ÷ race starts × 100. Example: 46 breakdowns in 319 starts = 14.4%.

**Minimum 40 starts per team and period** (about one full season with 2 cars). A team with fewer starts raced only
part of a season, so a few breakdowns would give it an extreme rate. Teams below the minimum are shown but not ranked.

In [ ]:
MIN_TEAM_STARTS = 40   # about one full season with 2 cars

# True / False for every start: did it end with a car failure?
q3["mechanical"] = q3["label"] == "mechanical"

# per period and team: starts (count of rows) and breakdowns (sum of True = 1)
starts = q3.groupby(["period", "team"])["mechanical"].count()
breakdowns = q3.groupby(["period", "team"])["mechanical"].sum()

rates = pd.DataFrame({"starts": starts, "mechanical": breakdowns})
rates["rate %"] = rates["mechanical"] / rates["starts"] * 100
rates["enough starts"] = rates["starts"] >= MIN_TEAM_STARTS

print("left out (fewer than", MIN_TEAM_STARTS, "starts):")
display(rates[~rates["enough starts"]].round(1))
rates.round(1)

In [ ]:
# one bar chart per period, teams sorted by rate (the highest at the top)
for period in ["2014-21", "2022-24"]:
    one_period = rates.loc[period]                               # only this period (rows = teams)
    one_period = one_period[one_period["enough starts"]]         # only teams with enough starts
    one_period = one_period.sort_values("rate %")                # low to high, so barh draws the highest at the top

    plt.figure(figsize=(9, 5))
    plt.barh(one_period.index, one_period["rate %"], color="tab:red", label="mechanical retirements per 100 starts")
    plt.title("Mechanical retirement rate per team, " + period)
    plt.xlabel("% of race starts that ended with a car failure")
    plt.ylabel("team")
    plt.xlim(0, 16)                                              # same scale in both charts, so they can be compared
    plt.legend(loc="lower right")
    plt.show()

#### Answer to Q3

**2014-2021, highest mechanical-retirement rate:**
1. Toro Rosso / AlphaTauri / RB: **14.4%** (46 of 319 starts)
2. Lotus / Renault / Alpine: 13.8% (44 of 318)
3. McLaren: 13.7% (43 of 315)

Lowest: Mercedes, **3.8%** (12 of 320).

**2022-2024, highest mechanical-retirement rate:**
1. Lotus / Renault / Alpine: **10.4%** (14 of 135)
2. Sauber / Alfa Romeo: 8.8% (12 of 136)
3. Toro Rosso / AlphaTauri / RB: 5.9% (8 of 136)

Lowest: Red Bull and McLaren, **3.7%** (5 of 136 each).

**What changed:**
- Cars broke down less in 2022-24. Mechanical retirements fell from 9.4% to 5.5% of all starts, while crashes stayed the
  same (6.9% → 6.7%). More cars finished (82.0% → 86.2%).
- Lotus / Renault / Alpine is near the top in both periods (2nd → 1st).
- McLaren dropped the most, from 13.7% (3rd) to 3.7% (lowest).

**Grain**
- Input: one row per driver per race start (`model_base`, 2014-2024: 4595 starts).
- Result: one row per team and period.
- All joins are checked: `outcomes` is one-to-one, `status` and `constructors` are many-to-one, and the row count stays the same.

**Missingness**
- Every start has a status, and all 139 status values have a label (checked with asserts).
- Teams with fewer than 40 starts in a period are shown but not ranked (only Caterham 2014: 7 of 33 = 21%).

**Pre-race cutoff**
- The status (how the race ended) is the *outcome* we measure. It comes from `outcomes` and is never used as a model input.

**Limitations**
- "Retired" without a reason counts as other, so a few breakdowns may be missed. It is a similar share in both periods
  (1.5% vs 1.2% of starts, inside "other"), so the comparison stays fair.
- Borderline labels are a choice: punctures and wing damage count as accidents (usually contact), a failed tyre counts as mechanical.
- Team renames are merged by hand (checked: same seasons in a row, same drivers).
- 2022-24 has only 3 seasons (about 135 starts per team), so one or two breakdowns move a team's rate by about 1 point.
- A breakdown is counted for the team, even when the failing part (for example the engine) comes from another manufacturer.

---
# >>> Section from `notebooks/p3_features.ipynb`


## Part 3: Features, split and preprocessing (P3)

### Section 3.1: Setup

In [ ]:
clean = load_clean()
base     = clean["model_base"]   # pre-race columns + target 'podium'
outcomes = clean["outcomes"]     # post-race columns NEVER use as features
tables   = clean["tables"]       # all other cleaned tables

base.shape, outcomes.shape, list(tables.keys())

### Section 3.2: Base table sanity check

the datatset has been cleaned with 0 duplicated rows for (raceId, driverId)

our target 'podium' is binary int8 with values [0,1]. the overall podium rate is around 13.7% which matches the expected 3/field-size ratio.

missing values are present in the qualifying-related columns q1_s, q2_S, q3_s, qual_position.

this is what we want to target when choosing the start year since the f1 knockout started after 2006.

In [ ]:
import pandas as pd

# Missingness across the full base table (all columns, all years)
miss = base.isna().mean().sort_values(ascending=False)

summary = pd.DataFrame({
    "missing_count": base.isna().sum(),
    "missing_share": miss,
    "dtype": base.dtypes,
})
summary["missing_share"] = summary["missing_share"].round(4)

print("=" * 65)
print(f"base shape: {base.shape[0]:,} rows × {base.shape[1]} columns")
print(f"Columns with ANY missing:  {(summary['missing_count'] > 0).sum()}")
print(f"Columns with NO  missing:  {(summary['missing_count'] == 0).sum()}")
print("=" * 65)

print("\nColumns with missing values (sorted):")
print(summary[summary["missing_count"] > 0].to_string())

In [ ]:
base.head()

In [ ]:
base.dtypes

In [ ]:
# Shape of the cleaned base table
print("base shape:", base.shape)

# Grain: one row per (raceId, driverId)
dupes = base.duplicated(["raceId", "driverId"]).sum()
print("duplicated (raceId, driverId):", dupes)
assert dupes == 0, "grain broken"

# Target column present
assert "podium" in base.columns, "target 'podium' missing"
print("target dtype:", base["podium"].dtype)
print("target values:", base["podium"].unique())

# Target rate on the raw base
print("\nOverall podium rate:", base["podium"].mean().round(4))

In [ ]:
yearly = base.groupby("year")["podium"].agg(["mean", "count"])
print("Podium rate, first 10 years:")
print(yearly.head(10))

print("\nPodium rate, last 10 years:")
print(yearly.tail(10))

In [ ]:
miss = base.isna().mean().sort_values(ascending=False)
print("Top 15 columns by missing share:")
print(miss.head(15).round(4))

print("\nColumns with zero missing:")
print(miss[miss == 0].index.tolist())

### Section 3.3: Training start year

The ml table uses qualifying features (qual_position, q1_s, q2_s, q3_s,
gap_to_pole_pct). 

Qualifying data does not exist for most of F1 history (e.g. 1950-1993),
so we need to choose the earliest stable year to include in the ML table.

We look at qualifying coverage per season and pick the earliest year where it's stable.

In [ ]:
# Calendar table needed to bring year/round into standings and results
races = tables["races"][["raceId", "year", "round", "circuitId", "date"]].copy()
races.head()

In [ ]:
# Find the first year with any qualifying data and the first year with high coverage
q_cov_exact = base.groupby("year")["has_qualifying"].mean()

# Years before this have no qualifying data at all
years_with_any = q_cov_exact[q_cov_exact > 0].index.min()
# Years before this have full or near-full coverage
years_with_full = q_cov_exact[q_cov_exact >= 0.90].index.min()

print("=" * 60)
print("First year with ANY qualifying data (has_qualifying > 0):", years_with_any)
print("First year with >=90% coverage:                      ", years_with_full)
print("=" * 60)

print("\nCoverage 1990–1997 (the transition years):")
print(q_cov_exact.loc[1990:1997].round(4))

In [ ]:
# Qualifying coverage per season: share of rows with a q1_s time
cov = base.groupby("year")["has_qualifying"].mean()

# Also look at q1_s, q2_s, q3_s separately
q_cov = base.groupby("year")[["q1_s", "q2_s", "q3_s"]].apply(lambda d: d.notna().mean())

print("has_qualifying per year (first 10 and last 10):")
print(cov.head(10))
print(cov.tail(10))

print("\nq1_s / q2_s / q3_s coverage for 1994-2010:")
print(q_cov.loc[1993:2010])

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cov.index, cov.values, marker="o", label="has_qualifying")
ax.axvline(2003, color="red", linestyle="--", label="START_YEAR = 2003")

ax.set_title("Qualifying coverage per season")
ax.set_xlabel("Year")
ax.set_ylabel("Share of rows with qualifying data")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

We chose 2003 as our start year since it defines the beginning of a stable inclusion of qualifying data. After the year 2003, the data value for the qualifying did not plummet nor vary greatly, overall.
Despite Q2 and Q3 missing for some of the years after and including 2003, we can handle it with reached_q2, reached_q3, and qual_format rather than imputation. This will help us distinguish if the values aof Q2 and Q3 are missing because of the yeat (before 2006) or if it's because they were eliminated in Q1.

In [ ]:
START_YEAR = 2003
model = base[base["year"] >= START_YEAR].copy()
model = model.sort_values(["year", "round", "raceId", "driverId"]).reset_index(drop=True)

print("Rows kept:", model.shape[0])
print("Years:", model["year"].min(), "-", model["year"].max())
print("Rows dropped:", base.shape[0] - model.shape[0])

**`qual_format` is 0 for the years before the knockout format was introduced (before 2006) and is 1 for the years after 2006** 

In [ ]:
model["reached_q2"]  = model["q2_s"].notna().astype(int)
model["reached_q3"]  = model["q3_s"].notna().astype(int)
model["qual_format"] = (model["year"] >= 2006).astype(int)

model[["year", "q1_s", "q2_s", "q3_s", "reached_q2", "reached_q3", "qual_format"]]
#qual_format tells us if that year was before the knockout era where the q2 and q3 
#were getting recorded starting from 2006
#we're gonna use it to see the reason q2 and q3 are missing within the data

In [ ]:
print("model shape:", model.shape)
print("model years:", model["year"].min(), "-", model["year"].max())

### Section 3.4: Grid and qualifying features

The prediction cutoff is after qualifying, and before the race. This means all the following columns are safe for our predicitons.

**Grid features**:
- `grid` is the actual starting position after any penalties, (0 = no grid slot, handled by pit_lane_flag).
- `pit_lane_flag` is 1 if the driver started from the pit lane where `grid==0`, it is 0 otherwise.

**Qualifying features**
- `qual_position` is the driver's rank in the qualifying session
- `best_qual_s` is the driver's best qualifying lap in seconds
  (`min(q1_s, q2_s, q3_s)`).
- `gap_to_pole_pct` is the percentage of how much slower a driver was than the pole sitter
  in the same race (the fastest driver). Per-race normalised, so comparable across eras.

**Missingness.** Qualifying data does not exist for most of F1 history, so
we cut the ML table at 2003 (see 3.3). From 2006 onward, `q2_s` and `q3_s`
are also structurally missing for drivers eliminated in earlier rounds;
we handle this with `reached_q2`, `reached_q3`, and `qual_format` rather
than imputing.  

All are set during or before grid formation. None come
from the race itself.

In [ ]:
# Grid features
model["pit_lane_flag"] = model["pitlane_start"].astype(int)

print("Grid feature summary:")
print(model[["grid", "pit_lane_flag"]].describe())
print("\nPit-lane starts share:", model["pit_lane_flag"].mean().round(4))
print("Rows with grid == 0:", (model["grid"] == 0).sum())

In [ ]:
model["best_qual_s"] = model[["q1_s", "q2_s", "q3_s"]].min(axis=1)

pole = model.groupby("raceId")["best_qual_s"].transform("min")
model["gap_to_pole_pct"] = (model["best_qual_s"] - pole) / pole * 100

print(model[["grid", "pit_lane_flag", "qual_position",
             "best_qual_s", "gap_to_pole_pct"]].describe())

In [ ]:
FORBIDDEN = {"positionOrder", "position", "positionText", "points", "laps",
             "time", "milliseconds", "fastestLap", "rank", "status"}
used = {"grid", "pit_lane_flag", "qual_position", "best_qual_s", "gap_to_pole_pct"}
assert not (used & FORBIDDEN), "post-race column used as feature"
print("No grid + qualifying leakage check: OK")

### Section 3.5: Pre-race standings ###

The standings tables (`driver_standings`, `constructor_standings`) record
cumulative points and rank **after** each race. Using them directly would
leak the current race's result into the features.

Fix: merge `races` to bring `year` and `round` into the standings, sort
by season+round, then `groupby(id).shift(1)` so each row holds the
standings **after the previous round** = standings going into this race.

**Leakage assertion.** Round 1 of every season must have NaN pre-race
standings (there is no previous race). The assertion fails loudly if
this is violated.

**Missingness.** ~5–7% of rows (round 1 of each season). We do NOT impute as we do not have/know the value since there is no previous race.

In [ ]:
# Bring year/round in needed to sort standings chronologically
races = tables["races"][["raceId", "year", "round"]].copy()
print(races.shape)
races.head()

In [ ]:
ds = tables["driver_standings"].copy()
ds = ds.merge(races, on="raceId", how="left")
ds = ds.sort_values(["year", "round", "driverId"])

ds["pts_before"] = ds.groupby(["year", "driverId"])["points"].shift(1)
ds["pos_before"] = ds.groupby(["year", "driverId"])["position"].shift(1)

ds_pre = ds[["raceId", "driverId", "pts_before", "pos_before"]].copy()
ds_pre.head()

In [ ]:
# What year/round is raceId 833?
print(races[races["raceId"] == 833])

# Look at more rows to see the shift working
# (filter to one driver across a season)
sample_driver = ds_pre["driverId"].iloc[0]
print("\nSample driver:", sample_driver)
sample = ds_pre.merge(races, on="raceId").query("driverId == @sample_driver")
print(sample.sort_values(["year", "round"]).head(10))

In [ ]:
# Merge pre-race driver standings into the modeling table
model = model.merge(ds_pre, on=["raceId", "driverId"], how="left")

print("model shape:", model.shape)
print(model[["pts_before", "pos_before"]].isna().mean())

# Assertion: first race of each season must have NaN pre-race standings
first_race = model.groupby("year")["round"].transform("min") == model["round"]
assert model.loc[first_race, "pts_before"].isna().all(), "LEAK: standings before round 1"
assert model.loc[first_race, "pos_before"].isna().all(), "LEAK: standings before round 1"
print("No driver standings leakage check: OK")

In [ ]:
# --- Constructor standings: standings going INTO each race ---
cs = tables["constructor_standings"].copy()
cs = cs.merge(races, on="raceId", how="left")
cs = cs.sort_values(["year", "round", "constructorId"])

cs["cons_pts_before"] = cs.groupby(["year", "constructorId"])["points"].shift(1)
cs["cons_pos_before"] = cs.groupby(["year", "constructorId"])["position"].shift(1)

cs_pre = cs[["raceId", "constructorId", "cons_pts_before", "cons_pos_before"]].copy()
cs_pre.head()

In [ ]:
print(races[races["raceId"].isin([765, 766])])

**Reading the constructor standings output.** 

Constructor standings are built the same way as driver standings: for each 
`(year, constructorId)` group, `shift(1)` moves each row's standings down 
by one race, so round N holds the standings after round N−1.

Two cases produce NaN:

1. **Round 1 of every season**: no previous race, so no standings exist. 
   Example: race 765 (1958 round 1) has NaN for every constructor.
2. **A constructor's first appearance in the data**: if a team enters 
   mid-season or a new `constructorId` is created (renames like 
   AlphaTauri → RB F1 Team in 2024), its first race has NaN because there 
   is no prior history under that ID. Example: constructor 32 in race 766 
   (1958 round 2), the team did not race in round 1.

In [ ]:
# Merge into model
model = model.merge(cs_pre, on=["raceId", "constructorId"], how="left")

print("model shape:", model.shape)
print(model[["cons_pts_before", "cons_pos_before"]].isna().mean())

# Assertion: round 1 of each season must have NaN
assert model.loc[first_race, "cons_pts_before"].isna().all(), "LEAK: constructor standings"
assert model.loc[first_race, "cons_pos_before"].isna().all(), "LEAK: constructor standings"
print("No constructor standings leakage check: OK")

### 3.6 Rolling form ###

We build a 5-race rolling podium rate for each driver and each constructor:
"how often did this driver/team podium in the last 5 races?"

The rule is that rolling windows must use only past races. We enforce this
with `shift(1)` before `.rolling()`: row N never sees its own race's result.

For constructors, we first collapse to one row per team-race, because two
drivers share the same constructor. We do not se `groupby("constructorId").shift(1)` as it 
would leak the teammate's same-race result into the form value.

Leakage assertions:
- First career race per driver = NaN
- First race per constructor = NaN
- Both drivers of a team-race share the same constructor_form5 value

In [ ]:
# Clean slate: drop the old form columns if they exist
model = model.drop(columns=["driver_form5", "constructor_form5"], errors="ignore")
print("model columns now:", model.shape)

In [ ]:
# Compute form on ALL years so pre-2003 history is available
base_sorted = base.sort_values(["driverId", "year", "round"]).reset_index(drop=True)

base_sorted["driver_form5"] = (
    base_sorted.groupby("driverId")["podium"]
               .transform(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean())
)

# Merge onto model (2003+)
model = model.merge(
    base_sorted[["raceId", "driverId", "driver_form5"]],
    on=["raceId", "driverId"],
    how="left"
)

model[["driverId", "year", "round", "podium", "driver_form5"]].head(10)

In [ ]:
# First career race ever (from base) must be NaN check by looking at base_sorted
first_driver_career = base_sorted.groupby("driverId").cumcount() == 0
assert base_sorted.loc[first_driver_career, "driver_form5"].isna().all(), "LEAK: driver form"

# Also check: within model, no NaN except genuinely-new drivers
print("driver_form5 NaN share in model:", model["driver_form5"].isna().mean().round(4))
print("No driver form leakage check: OK")

In [ ]:
# Collapse to one row per (constructor, race) on ALL years
base_cons = (
    base.groupby(["constructorId", "raceId", "year", "round"], as_index=False)["podium"]
        .mean()
)

base_cons = base_cons.sort_values(["constructorId", "year", "round"])
base_cons["constructor_form5"] = (
    base_cons.groupby("constructorId")["podium"]
             .transform(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean())
)

# Merge onto model
model = model.merge(
    base_cons[["raceId", "constructorId", "constructor_form5"]],
    on=["raceId", "constructorId"],
    how="left"
)

model[["constructorId", "year", "round", "podium", "constructor_form5"]].head(10)

In [ ]:
# 1. First race per constructor (chronologically) must be NaN
first_per_constructor = (
    base_cons.sort_values(["constructorId", "year", "round"])
            .groupby("constructorId")
            .head(1)
)
assert first_per_constructor["constructor_form5"].isna().all(), \
    "LEAK: constructor form on first race"

# 2. Both drivers of a team-race must share the same value
grouped = model.groupby(["raceId", "constructorId"])["constructor_form5"].nunique(dropna=False)
assert (grouped <= 1).all(), \
    "constructor form differs within the same team-race"

print("No constructor form leakage checks: OK")
print("constructor_form5 NaN share in model:", model["constructor_form5"].isna().mean().round(4))

In [ ]:
# Put rows back in chronological race order for downstream sections
model = model.sort_values(["year", "round", "raceId", "driverId"]).reset_index(drop=True)

print("model shape:", model.shape)
model[["raceId", "driverId", "year", "round",
       "driver_form5", "constructor_form5", "podium"]].head(5)

### Section 3.7: Circuit history + home flag + era ###

We also need to add these to our features: 

**Circuit history.** Some drivers are specialists at certain tracks (e.g. Senna
at Monaco, Hamilton at Silverstone). We capture this with two features per
(driver, circuit):
- `circuit_starts_before`: number of previous races by this driver at this circuit
- `circuit_podiums_before`: number of previous podiums by this driver at this circuit

Both use only past races. The current race is excluded by `shift(1).cumsum()`.

We compute on the full `base` table (all years), then merge onto `model`
(2003+) so 2003 rows have their pre-2003 circuit history.

**Home flag.** `home_flag = 1` when the driver's nationality matches the
circuit's country (e.g. British driver at Silverstone).

**Era.** F1 rules and car performance shift by technical era. We bucket
years: `pre_2014` (2003–2013), `hybrid` (2014–2021), `ground_effect` (2022+).
Year is known before the race.

In [ ]:
base_sorted_circuit = base.sort_values(["driverId", "circuitId", "year", "round"]).reset_index(drop=True)

base_sorted_circuit["circuit_starts_before"] = (
    base_sorted_circuit.groupby(["driverId", "circuitId"]).cumcount()
)

base_sorted_circuit["circuit_podiums_before"] = (
    base_sorted_circuit.groupby(["driverId", "circuitId"])["podium"]
                       .transform(lambda s: s.shift(1).fillna(0).cumsum())
)

# Re-merge onto model (drop old columns first to avoid suffix mess)
model = model.drop(columns=["circuit_starts_before", "circuit_podiums_before"], errors="ignore")
model = model.merge(
    base_sorted_circuit[["raceId", "driverId",
                         "circuit_starts_before", "circuit_podiums_before"]],
    on=["raceId", "driverId"],
    how="left"
)

model[["driverId", "circuitId", "year", "round",
       "circuit_starts_before", "circuit_podiums_before", "podium"]].head(10)

In [ ]:
# Assertion on base_sorted_circuit (all years), not on model
first_at_circuit_base = (
    base_sorted_circuit.sort_values(["driverId", "circuitId", "year", "round"])
                      .groupby(["driverId", "circuitId"])
                      .cumcount() == 0
)
assert (base_sorted_circuit.loc[first_at_circuit_base, "circuit_starts_before"] == 0).all(), \
    "LEAK: starts before"
assert (base_sorted_circuit.loc[first_at_circuit_base, "circuit_podiums_before"] == 0).all(), \
    "LEAK: podiums before"
print("No circuit history leakage check: OK")

In [ ]:
first = base_sorted_circuit.loc[first_at_circuit_base]
print("Total first-occurrence rows:", len(first))
print("Non-zero podiums:", (first["circuit_podiums_before"] != 0).sum())
print("NaN podiums:", first["circuit_podiums_before"].isna().sum())
print("\nSample:")
print(first[["driverId", "circuitId", "year", "circuit_podiums_before"]].head(10))

In [ ]:
NATIONALITY_TO_COUNTRY = {
    "American": "USA", "American-Italian": "USA", "Argentine": "Argentina",
    "Argentine-Italian": "Argentina", "Argentinian": "Argentina", "Australian": "Australia",
    "Austrian": "Austria", "Belgian": "Belgium", "Brazilian": "Brazil", "British": "UK",
    "Canadian": "Canada", "Chilean": "Chile", "Chinese": "China", "Colombian": "Colombia",
    "Czech": "Czech Republic", "Danish": "Denmark", "Dutch": "Netherlands",
    "East German": "Germany", "Finnish": "Finland", "French": "France", "German": "Germany",
    "Hungarian": "Hungary", "Indian": "India", "Indonesian": "Indonesia", "Irish": "Ireland",
    "Italian": "Italy", "Japanese": "Japan", "Liechtensteiner": "Liechtenstein",
    "Malaysian": "Malaysia", "Mexican": "Mexico", "Monegasque": "Monaco",
    "New Zealander": "New Zealand", "Polish": "Poland", "Portuguese": "Portugal",
    "Rhodesian": "Rhodesia", "Russian": "Russia", "South African": "South Africa",
    "Spanish": "Spain", "Swedish": "Sweden", "Swiss": "Switzerland", "Thai": "Thailand",
    "Uruguayan": "Uruguay", "Venezuelan": "Venezuela",
}
SECOND_COUNTRY = {"American-Italian": "Italy", "Argentine-Italian": "Italy"}

# --- Drivers: nationality, stripped ---
drivers_nat = tables["drivers"][["driverId", "nationality"]].copy()
drivers_nat["nationality"] = drivers_nat["nationality"].str.strip()

# --- Circuits: country, normalized ---
circuits_country = tables["circuits"][["circuitId", "country"]].copy()
circuits_country["country"] = circuits_country["country"].replace("United States", "USA")

# --- Merge onto model ---
model = model.merge(drivers_nat, on="driverId", how="left", validate="many_to_one")
model = model.merge(circuits_country, on="circuitId", how="left", validate="many_to_one")

# --- home_flag: circuit country matches driver's primary OR secondary home country ---
model["home_country"] = model["nationality"].map(NATIONALITY_TO_COUNTRY)
model["second_home_country"] = model["nationality"].map(SECOND_COUNTRY)

model["home_flag"] = (
    (model["country"] == model["home_country"]) |
    (model["country"] == model["second_home_country"])
).astype(int)

assert model["nationality"].notna().all(), "driver missing nationality"
assert model["country"].notna().all(), "circuit missing country"
unmapped = set(model["nationality"]) - set(NATIONALITY_TO_COUNTRY)
assert len(unmapped) == 0, f"nationalities missing from mapping: {unmapped}"

print("home_flag value counts:")
print(model["home_flag"].value_counts())
print("\nHome starts per country (top 10):")
print(model[model["home_flag"] == 1]["country"].value_counts().head(10))

In [ ]:
# Keep only home_flag , drop helper columns from the join
model = model.drop(columns=["nationality", "country", "home_country", "second_home_country"],
                   errors="ignore")
print("model shape:", model.shape)

In [ ]:
import pandas as pd
model["era"] = pd.cut(
    model["year"],
    bins=[2002, 2013, 2021, 2024],
    labels=["v8", "hybrid", "ground_effect"]
)
print(model["era"].value_counts())
print("\nmodel shape:", model.shape)

In [ ]:
print("Columns in model:")
for i, col in enumerate(sorted(model.columns), 1):
    print(f"{i:2d}. {col}")
print("\nTotal:", len(model.columns))

### Section 3.8: Data dictionary

The modeling table has the columns: 16 (keys + target + grid/qual raw), 
and 16 features grouped into 6 categories for the ablation, and `split` column.

**Note on identifiers.** Four columns in the table are identifiers, not 
features: `driverId`, `constructorId`, `circuitId`, and `number`. They 
are deliberately excluded from `FEATURES` for three reasons.

**1. They are not predictive on their own.** A raw ID is just a number 
assigned by the dataset. `driverId = 830` does not mean "better" or 
"worse" than `driverId = 815`. If we fed IDs directly to the model, it 
would treat them as ordered numeric values and learn spurious patterns 
(e.g. "higher IDs tend to podium") that would not generalise to new data.

**2. One-hot encoding them would explode the feature space.** There are 
~860 drivers, ~210 constructors, and ~77 circuits in the full dataset. 
One-hot encoding these would add hundreds of columns, most of which 
would have very few non-zero rows. That overfits, especially with only 
6,781 training rows.

**3. Their signal is captured better by historical aggregates.** Each 
identifier is useful only insofar as it points to past performance. We 
extract that signal into numerically comparable features:

| Identifier | Replaced by | What it captures |
|---|---|---|
| `driverId` | `driver_form5` | Driver's recent podium rate (last 5 races) |
| `constructorId` | `constructor_form5` | Team's recent podium rate (last 5 races) |
| `driverId` + `circuitId` | `circuit_starts_before`, `circuit_podiums_before` | Driver's history at this specific track |
| `constructorId` + `circuitId` | `constructor_circuit_podium_rate` (if used) | Team's history at this specific track |
| `number` | (dropped entirely) | Car number has no consistent meaning across eras |

**Note on `number`.** Car numbers changed every season in early F1 and 
were only made permanent per-driver from 2014 onward. A number like "1" 
historically indicated the reigning champion in some eras and meant 
nothing in others. It is high-cardinality, era-inconsistent, and adds 
noise. We exclude it entirely.

**Note on `circuitId`.** The circuit's own identity is not used directly, 
but its context (country, past races) is captured by `home_flag` and 
`circuit_*_before`. If we ever wanted circuit-level fixed effects (e.g. 
Monaco is harder to overtake), we would need to encode them as 
target-encoded features or add them as a tree model's categorical 
input — but that is out of scope for this milestone.

**What stays.** `raceId`, `year`, `round`, `date` are keys/context, they 
appear in the table but are also excluded from `FEATURES`. `split` is a 
model-control label, not a feature.

**Note on pit_lane_flag.** This is the standardised int version of the raw 
`pitlane_start`. Only `pit_lane_flag` is in FEATURES; the raw column is not used.

**Note on `era`.** `era` was built in 3.7 and tested its effect in 3.12. It is 
excluded from FEATURES because the training set (2003–2019) contains only 
`v8` and `hybrid`, so a one-hot encoder cannot represent the `ground_effect` 
era that appears only in test. Experiment 3 in 3.12 confirms that dropping 
it does not hurt validation performance. `era` remains a column in the 
table for EDA and reporting only.

In [ ]:
FEATURE_GROUPS = {
    "grid":       ["grid", "pit_lane_flag"],
    "qualifying": ["qual_position", "best_qual_s", "gap_to_pole_pct",
                   "reached_q2", "reached_q3", "qual_format"],
    "standings":  ["pts_before", "pos_before",
                   "cons_pts_before", "cons_pos_before"],
    "form":       ["driver_form5", "constructor_form5"],
    "circuit":    ["circuit_starts_before", "circuit_podiums_before"],
    "context":    ["home_flag"],
}
FEATURES = [f for group in FEATURE_GROUPS.values() for f in group]

print(f"Total features: {len(FEATURES)}")
for group, cols in FEATURE_GROUPS.items():
    print(f"  {group}: {len(cols)}")

**Note on `era`.** `era` was removed from FEATURES after the ablation
showed the context group adds ~0 to validation PR-AUC, and their
recommended dropping it. The ground-effect test era (2022+) never appears 
in training, so its one-hot encoding is uninformative. `era` stays in the 
table for EDA and reporting.

**Note on the `v8` label.** The `v8` era bucket spans 2003–2013, but F1 used 
V10 engines 2003–2005 and V8 engines 2006–2013. We label the whole window 
`v8` for simplicity.

In [ ]:
FORBIDDEN = {"positionOrder", "position", "positionText", "points", "laps",
             "time", "milliseconds", "fastestLap", "rank", "status"}
assert not (set(FEATURES) & FORBIDDEN), f"forbidden in FEATURES: {set(FEATURES) & FORBIDDEN}"
print("all features are leakage-safe")

In [ ]:
import json
import os
os.makedirs("reports", exist_ok=True)

with open("reports/p3_feature_groups.json", "w") as f:
    json.dump(FEATURE_GROUPS, f, indent=2)

# Save as CSV instead of parquet
#.to_csv("reports/p3_modeling_table.csv", index=False)

print("saved:")
print(" - reports/p3_feature_groups.json")
#print(" - reports/p3_modeling_table.csv")
print(" - shape:", model.shape)

### Section 3.9: Evidence plots

We produce four plots to justify our features:

1. **Podium rate vs grid bucket** shows that starting position predicts podium.
2. **Podium rate vs qualifying bucket** shows that qualifying position predicts podium.
3. **Feature distributions by target** shows form features separate podium vs. non-podium.
4. **Correlation heatmap** checks for redundancy and confirms features relate to `podium`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

fig, ax = plt.subplots(figsize=(10, 5))

# Bucket grid into sensible groups
bins = [-1, 0, 2, 4, 6, 10, 15, 25]
labels = ["pit lane", "1-2", "3-4", "5-6", "7-10", "11-15", "16+"]
model["grid_bucket"] = pd.cut(model["grid"], bins=bins, labels=labels)

grid_stats = model.groupby("grid_bucket", observed=True)["podium"].agg(["mean", "count"])
grid_stats["mean_pct"] = grid_stats["mean"] * 100

bars = ax.bar(grid_stats.index.astype(str), grid_stats["mean_pct"], color="tab:blue")
ax.bar_label(bars, fmt="%.1f%%", fontsize=9)
ax.set_title("Podium rate by grid position bucket")
ax.set_xlabel("Grid bucket")
ax.set_ylabel("Podium rate (%)")
ax.set_ylim(0, 80)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("reports/fig_p3_podium_vs_grid.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

bins_q = [0, 2, 4, 6, 10, 15, 25]
labels_q = ["1-2", "3-4", "5-6", "7-10", "11-15", "16+"]
model["qual_bucket"] = pd.cut(model["qual_position"], bins=bins_q, labels=labels_q)

qual_stats = model.groupby("qual_bucket", observed=True)["podium"].agg(["mean", "count"])
qual_stats["mean_pct"] = qual_stats["mean"] * 100

bars = ax.bar(qual_stats.index.astype(str), qual_stats["mean_pct"], color="tab:orange")
ax.bar_label(bars, fmt="%.1f%%", fontsize=9)
ax.set_title("Podium rate by qualifying position bucket")
ax.set_xlabel("Qualifying bucket")
ax.set_ylabel("Podium rate (%)")
ax.set_ylim(0, 80)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("reports/fig_p3_podium_vs_qual.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot form by target
axes[0].hist(model.loc[model["podium"] == 0, "driver_form5"].dropna(),
             bins=20, alpha=0.6, label="not podium", color="tab:red", density=True)
axes[0].hist(model.loc[model["podium"] == 1, "driver_form5"].dropna(),
             bins=20, alpha=0.6, label="podium", color="tab:green", density=True)
axes[0].set_title("driver_form5 by target")
axes[0].set_xlabel("driver_form5")
axes[0].set_ylabel("Density")
axes[0].legend()

# Constructor form by target
axes[1].hist(model.loc[model["podium"] == 0, "constructor_form5"].dropna(),
             bins=20, alpha=0.6, label="not podium", color="tab:red", density=True)
axes[1].hist(model.loc[model["podium"] == 1, "constructor_form5"].dropna(),
             bins=20, alpha=0.6, label="podium", color="tab:green", density=True)
axes[1].set_title("constructor_form5 by target")
axes[1].set_xlabel("constructor_form5")
axes[1].legend()

# Pre-race driver points by target
axes[2].hist(model.loc[model["podium"] == 0, "pts_before"].dropna(),
             bins=20, alpha=0.6, label="not podium", color="tab:red", density=True)
axes[2].hist(model.loc[model["podium"] == 1, "pts_before"].dropna(),
             bins=20, alpha=0.6, label="podium", color="tab:green", density=True)
axes[2].set_title("pts_before by target")
axes[2].set_xlabel("pts_before")
axes[2].legend()

plt.tight_layout()
plt.savefig("reports/fig_p3_distributions_by_target.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
import seaborn as sns

# Numeric features only
numeric_features = [f for f in FEATURES if f != "era"]

corr = model[numeric_features + ["podium"]].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, cbar_kws={"shrink": 0.8}, annot_kws={"size": 8})
ax.set_title("Feature correlation heatmap")
plt.tight_layout()
plt.savefig("reports/fig_p3_correlation_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()

# Print correlations with podium, sorted
print("\nCorrelation with podium (sorted):")
print(corr["podium"].drop("podium").sort_values(ascending=False).round(3))

### 3.9 findings

**Plot 1 (grid):** podium rate drops monotonically from 72.5% (grid 1–2) 
to 1% (grid 16+). `grid` is a strong predictor.

**Plot 2 (qualifying):** same shape as grid. `qual_position` matters.

**Plot 3 (distributions):** `driver_form5`, `constructor_form5`, and 
`pts_before` all show clear class separation.

**Plot 4 (correlations with podium):**
- Strongest: `driver_form5` (0.55), `constructor_form5` (0.55), `qual_position` (-0.50), `grid` (-0.48)
- Moderate: `pts_before` (0.44), `circuit_podiums_before` (0.24)
- Weak: `home_flag` (-0.001), `qual_format` (-0.009), `pit_lane_flag` (-0.037)

**Notes:**
- `best_qual_s` (-0.055) is weak because absolute lap times vary by track; 
  `gap_to_pole_pct` (-0.28) is the normalized alternative and carries more signal.
- `home_flag` near-zero which is consistent with Q2 finding.
- No feature pair exceeds 0.95 correlation (see heatmap), so no severe redundancy.

### Section 3.10: Time split ###

We will split the 2003-2024 rows into: 
- **train**: year ≤ 2019
- **validation**: year 2020–2021
- **test**: year ≥ 2022

**Why these years.** 2020–2021 gives us two full seasons (with one shortened 
by COVID) for hyperparameter tuning. 2022+ is a clean, larger test set 
(3 seasons) that includes the new ground-effect era. Test set is used once.

In [ ]:
import numpy as np

model["split"] = np.select(
    [model["year"] <= 2019,
     model["year"].between(2020, 2021),
     model["year"] >= 2022],
    ["train", "validation", "test"],
    default="unknown"
)

print(model["split"].value_counts())
print("\nmodel shape:", model.shape)

In [ ]:
# Class balance per split
balance = model.groupby("split")["podium"].agg(["mean", "count"])
print("Class balance per split:")
print(balance.round(4))

# Plot
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
balance_ordered = balance.reindex(["train", "validation", "test"])
bars = ax.bar(balance_ordered.index, balance_ordered["mean"] * 100,
              color=["tab:blue", "tab:orange", "tab:green"])
ax.bar_label(bars, fmt="%.1f%%")
ax.set_title("Podium rate per split (class balance)")
ax.set_xlabel("Split")
ax.set_ylabel("Podium rate (%)")
ax.set_ylim(0, 20)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("reports/fig_p3_split_balance.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
model.to_csv("reports/p3_modeling_table.csv", index=False)
print("saved with split, shape:", model.shape)
print("columns:", len(model.columns))

### Section 3.11: Preprocessing pipeline

We build a scikit-learn `ColumnTransformer`:

- **Numeric features**: `SimpleImputer(strategy="median")` → `StandardScaler()

The pipeline is **fit on train only**, then applied to validation and test

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# All features are numeric now
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, FEATURES),
])

print(f"preprocessor built for {len(FEATURES)} features")
print(FEATURES)

In [ ]:
train_df = model[model["split"] == "train"]
val_df   = model[model["split"] == "validation"]
test_df  = model[model["split"] == "test"]

X_train = train_df[FEATURES]
X_val   = val_df[FEATURES]
X_test  = test_df[FEATURES]

y_train = train_df["podium"].values
y_val   = val_df["podium"].values
y_test  = test_df["podium"].values

# Fit on TRAIN ONLY, then transform all three
preprocessor.fit(X_train)

X_train_t = preprocessor.transform(X_train)
X_val_t   = preprocessor.transform(X_val)
X_test_t  = preprocessor.transform(X_test)

print("Shapes:")
print(f"  X_train: {X_train.shape} -> {X_train_t.shape}")
print(f"  X_val:   {X_val.shape} -> {X_val_t.shape}")
print(f"  X_test:  {X_test.shape} -> {X_test_t.shape}")

In [ ]:
import numpy as np

# Sanity: no NaN after transform, values are roughly standardized
print("NaN after transform (should be 0):")
print(f"  X_train: {np.isnan(X_train_t).sum()}")
print(f"  X_val:   {np.isnan(X_val_t).sum()}")
print(f"  X_test:  {np.isnan(X_test_t).sum()}")

print("\nTrain mean ~0, std ~1 per feature:")
print(f"  mean range: {X_train_t.mean(axis=0).min():.4f} to {X_train_t.mean(axis=0).max():.4f}")
print(f"  std range:  {X_train_t.std(axis=0).min():.4f} to {X_train_t.std(axis=0).max():.4f}")

# Val/test means can be slightly off 0 (different distribution), but no NaN
print("\nVal mean range (may differ from 0):")
print(f"  {X_val_t.mean(axis=0).min():.4f} to {X_val_t.mean(axis=0).max():.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Three representative features covering different scales and missingness patterns
sample_features = ["gap_to_pole_pct", "pts_before", "driver_form5"]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))

for i, feat in enumerate(sample_features):
    # Get the column index in FEATURES (matches the transform order)
    col_idx = FEATURES.index(feat)

    # Top row: raw values (before impute + scale), NaN dropped for plotting
    axes[0, i].hist(X_train[feat].dropna(), bins=30, alpha=0.7, color="tab:blue")
    axes[0, i].set_title(f"{feat} — raw")
    axes[0, i].set_ylabel("Count")
    axes[0, i].set_xlabel(feat)

    # Bottom row: after impute + scale
    axes[1, i].hist(X_train_t[:, col_idx], bins=30, alpha=0.7, color="tab:orange")
    axes[1, i].set_title(f"{feat} — after impute + scale")
    axes[1, i].set_ylabel("Count")
    axes[1, i].set_xlabel("standardized")

fig.suptitle("Effect of preprocessing on feature distributions (train set)", y=1.00)
plt.tight_layout()
plt.savefig("reports/fig_p3_preprocessing_effect.png", dpi=120, bbox_inches="tight")
plt.show()

**Visual effect of preprocessing.** The top row shows the raw feature 
distributions on the training set; the bottom row shows the same features 
after median imputation and standardization. Raw features have very 
different units and ranges (%, points, rate), which would distort 
distance-based models. After preprocessing, all features share a comparable 
scale (mean ≈ 0, std ≈ 1) while their underlying shape is preserved. This 
is what linear models and the neural network require.

### 3.11 wrap-up

Pipeline: `SimpleImputer(strategy="median")` → `StandardScaler()`, fit on
the 6,781 training rows only, then applied to validation (774 rows) and
test (1,351 rows).

**Evidence of correctness:**
- No NaN after transform (imputer filled all missing values)
- Train mean ≈ 0, std ≈ 1 (scaler normalized correctly)
- Validation mean shifted to -0.43 to 0.43 (different year distribution, 
  as expected; the scaler was not fit on validation data, which would have 
  been leakage).

### Section 3.12: Preprocessing experiments

The following experiments justify our preprocessing choices. Both use a small logistic 
regression only to measure the effect of the preprocessing.

**Experiment 1: scaling vs no scaling (validation)**
Does standardizing the features change model performance or 
training behaviour?

**Setup.** Two pipelines, identical except for the scaler:

- **impute → scale**: median imputation, then `StandardScaler()`, then 
  logistic regression.
- **impute-only**: median imputation, then logistic regression (features 
  stay in their raw units).

Both use the same train/validation split, the same `LogisticRegression` 
(`max_iter=2000, random_state=SEED`), and are scored on validation PR-AUC.

**Why it matters.** The features span very different units: `grid` 
ranges 0–24, `gap_to_pole_pct` ranges 0–10%, `pts_before` ranges 0–400. 
Distance-based optimizers like `lbfgs` (used by logistic regression) 
struggle when features differ by orders of magnitude. Standardizing to 
mean 0, std 1 puts every feature on the same scale and should make the 
optimizer's job easier.

We check two things: validation PR-AUC (does accuracy change?) and 
convergence warnings (does the optimizer finish cleanly?).

**Experiment 2: fit-on-train vs fit-on-all (leakage demo)**
Demonstrates why we fit preprocessing on train only. The "fit on all" version 
sees validation/test data during fitting; its test score is optimistic, and 
the correct version is the honest estimate.

**Experiment 3: does `era` contribute to the model's prediction?**
During feature engineering, `era` was added as a candidate feature 
(`v8`, `hybrid`, `ground_effect`). After the train/validation/test split, 
we noticed a structural issue: the training set (2003–2019) contains only 
`v8` and `hybrid`. The `ground_effect` era appears only in the test set 
(2022–2024). A one-hot encoder fit on training data would therefore learn 
only two categories, and every test row (all `ground_effect`) would be 
encoded as an unseen category, effectively all-zeros.

The feature `era` was therefore dropped from the ML table. To verify that dropping it 
does not hurt, we fit two pipelines with the same preprocessing and model, 
differing only by whether `era` is included as a one-hot feature:

In [ ]:
#Experiment 1a:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

# Pull splits
train_df = model[model["split"] == "train"]
val_df   = model[model["split"] == "validation"]
test_df  = model[model["split"] == "test"]

def lr_pipeline(scale: bool):
    """Logistic regression with median imputation; optionally scale."""
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("lr", LogisticRegression(max_iter=2000, random_state=SEED)))
    return Pipeline(steps)

def evaluate_pipeline(pipe, X_train, y_train, X_eval, y_eval, fit_on):
    """Fit pipe on fit_on data, evaluate on X_eval / y_eval."""
    pipe.fit(fit_on[0], fit_on[1])
    proba = pipe.predict_proba(X_eval)[:, 1]
    return {
        "PR-AUC": round(average_precision_score(y_eval, proba), 4),
        "ROC-AUC": round(roc_auc_score(y_eval, proba), 4),
    }

print("setup ready")

In [ ]:
import warnings
import pandas as pd
from sklearn.exceptions import ConvergenceWarning

X_tr = train_df[FEATURES];  y_tr = train_df["podium"].values
X_va = val_df[FEATURES];    y_va = val_df["podium"].values

def run_and_count_warnings(pipe, X_fit, y_fit, X_eval, y_eval):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        pipe.fit(X_fit, y_fit)
        proba = pipe.predict_proba(X_eval)[:, 1]
    n_warn = sum(1 for x in w if issubclass(x.category, ConvergenceWarning))
    return {
        "PR-AUC": round(average_precision_score(y_eval, proba), 4),
        "ROC-AUC": round(roc_auc_score(y_eval, proba), 4),
        "convergence warnings": n_warn,
    }

rows = []
for name, scale in [("impute→scale", True), ("impute-only", False)]:
    r = run_and_count_warnings(lr_pipeline(scale=scale), X_tr, y_tr, X_va, y_va)
    rows.append({"pipeline": name, **r})

exp1 = pd.DataFrame(rows)
print("=" * 60)
print("Experiment 1a: scaling vs no scaling (validation)")
print("=" * 60)
print(exp1.to_string(index=False))

In [ ]:
# Experiment 1b order: impute→scale vs scale→impute
pipe_imp_sc = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc",  StandardScaler()),
    ("lr",  LogisticRegression(max_iter=2000, random_state=SEED)),
])
pipe_sc_imp = Pipeline([
    ("sc",  StandardScaler()),
    ("imp", SimpleImputer(strategy="median")),
    ("lr",  LogisticRegression(max_iter=2000, random_state=SEED)),
])
print("Comparison between the order of the imputation and scaling:")
for name, pipe in [("impute→scale", pipe_imp_sc), ("scale→impute", pipe_sc_imp)]:
    pipe.fit(X_tr, y_tr)
    pr = average_precision_score(y_va, pipe.predict_proba(X_va)[:, 1])
    print(f"{name}: validation PR-AUC = {pr:.4f}")

In [ ]:
from sklearn.base import clone

# Isolate preprocessing leakage: classifier always trained on TRAIN labels,
# only the preprocessing (imputer + scaler) sees different data.
prep_template = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

def lr_on(prep, X, y):
    return LogisticRegression(max_iter=2000, random_state=SEED).fit(prep.transform(X), y)

# Correct: preprocessing fit on TRAIN features only
prep_correct = clone(prep_template).fit(X_tr)
clf_correct = lr_on(prep_correct, X_tr, y_tr)
proba_correct = clf_correct.predict_proba(prep_correct.transform(X_va))[:, 1]

# Leaky: preprocessing fit on TRAIN + VALIDATION features
# (no labels involved, only the median/mean/std statistics)
X_tr_va_features = pd.concat([X_tr, X_va])
prep_leaky = clone(prep_template).fit(X_tr_va_features)
clf_leaky = lr_on(prep_leaky, X_tr, y_tr)
proba_leaky = clf_leaky.predict_proba(prep_leaky.transform(X_va))[:, 1]

# Metrics
pr_correct = average_precision_score(y_va, proba_correct)
pr_leaky   = average_precision_score(y_va, proba_leaky)

print("Experiment 2: preprocessing fit on train vs train+val (evaluated on VALIDATION)")
print(f"  Correct (preprocessing fit on train only) : PR-AUC = {pr_correct:.4f}")
print(f"  Leaky   (preprocessing fit on train+val)  : PR-AUC = {pr_leaky:.4f}")
print(f"  PR-AUC difference                         : {pr_leaky - pr_correct:+.4f}")

# Evidence that the leak is REAL, not seed noise:
# the scaler learned different statistics from the two fits.
scaler_correct = prep_correct.named_steps["scaler"]
scaler_leaky   = prep_leaky.named_steps["scaler"]

# Scaler drift expressed in standard-deviation units per feature
mean_shift_std = np.abs(scaler_correct.mean_ - scaler_leaky.mean_) / scaler_correct.scale_
scale_shift_pct = np.abs(scaler_correct.scale_ - scaler_leaky.scale_) / scaler_correct.scale_ * 100

print()
print("Scaler parameter drift (per-feature, std units):")
print(f"  Mean shift  : {mean_shift_std.mean():.4f} std (max {mean_shift_std.max():.4f})")
print(f"  Std change  : {scale_shift_pct.mean():.2f}%    (max {scale_shift_pct.max():.2f}%)")

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

FEATURES_WITH_ERA = FEATURES + ["era"]
X_tr_era = train_df[FEATURES_WITH_ERA]
X_va_era = val_df[FEATURES_WITH_ERA]
num_cols_with_era = [f for f in FEATURES_WITH_ERA if f != "era"]

# Pipeline A, with era (one-hot)
pipe_with_era = Pipeline([
    ("prep", ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
        ]), num_cols_with_era),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["era"]),
    ])),
    ("lr", LogisticRegression(max_iter=2000, random_state=SEED)),
])

# Pipeline B without era
pipe_no_era = lr_pipeline(scale=True)

# Fit and evaluate on validation
pipe_with_era.fit(X_tr_era, y_tr)
pr_with = average_precision_score(y_va, pipe_with_era.predict_proba(X_va_era)[:, 1])

pipe_no_era.fit(X_tr, y_tr)
pr_without = average_precision_score(y_va, pipe_no_era.predict_proba(X_va)[:, 1])

print("Experiment 3: era vs no era (validation)")
print(f"  With era:    PR-AUC = {pr_with:.4f}")
print(f"  Without era: PR-AUC = {pr_without:.4f}")
print(f"  Difference:  {pr_with - pr_without:+.4f}")

# Structural check: which categories does the train-fitted encoder know?
encoder = pipe_with_era.named_steps["prep"].named_transformers_["cat"]
print(f"\nOne-hot categories learned from train: {encoder.categories_[0].tolist()}")
print(f"Era values present in test: {sorted(test_df['era'].dropna().unique().tolist())}")
print("→ Any test row in a category not seen in train would be encoded as all-zeros.")

### 3.12 findings

**Experiment 1: impute→scale vs impute-only (validation)**

| Pipeline | PR-AUC | ROC-AUC | Convergence warnings |
|---|---|---|---|
| impute→scale | 0.7325 | 0.9298 | 0 |
| impute-only  | 0.7321 | 0.9284 | 1 |

Metrics are nearly identical, but the **unscaled version fails to converge**
(lbfgs hits the 2000-iteration limit). Scaling is not a nicety, it's required
for the optimizer to work on this feature set. We use `SimpleImputer(strategy=
"median")` followed by `StandardScaler()` in the pipeline.

**Experiment 1b: does the order matter? (impute→scale vs. scale→impute)**

Both orders converge and produce nearly the same validation PR-AUC 
(see the output above). We choose **impute→scale** because:
- It is the conventional order in scikit-learn pipelines.
- It produces a standardized final distribution (mean 0, std 1); the 
  scale→impute variant fills NaN with the median of scaled values, which 
  is not exactly 0.

The difference is expected to be small because our missingness is 
structural and small in volume (~7% of rows, mostly round-1 standings).

**Experiment 2: preprocessing fit on train vs train+val (leakage demo)**

The classifier is trained on train labels in both versions. Only the
preprocessing (imputer + scaler) sees different data during fitting.
Both are evaluated on **validation**, the test set is left untouched.

| Setup | Validation PR-AUC |
|---|---|
| Preprocessing fit on train only (correct) | 0.7325 |
| Preprocessing fit on train+val (leaky) | 0.7328 |
| Difference | **+0.0003** |

**Is the leak real, or is it noise?**

We compare the scaler's learned parameters between the two fits. Per
feature, the mean shifts by ~0.02 standard deviations on average
(max ~0.04), and the scale changes by a few percent on average, with a
max of ~27% on one feature (see output above). That max is on a feature
with high missingness, the round-1 rows contribute meaningfully to the
leaky fit. The leak is a real, measurable parameter drift.

**Why doesn't the metric move much?**

The PR-AUC impact is small (+0.0003): imputation and scaling are mild
transformations, so the leak's effect on a linear model's ranking is
small even though the scaler parameters moved.

**Why we still fit on train only:**

Even when the metric impact is small, the leak is a real violation: the
"leaky" pipeline's preprocessing statistics were influenced by data it
wasn't supposed to see. In a scenario with a larger distribution shift
(e.g. the ground-effect test era), the same violation could inflate the
score more. Fitting on train only is the honest, consistent rule.

**Takeaways:** StandardScaler is required (convergence), and fit-on-train-only
is the honest choice (leakage demo).

**Experiment 3: does `era` affect the model's prediction?**
`era` was built and tested as a categorical feature 
(v8, hybrid, ground_effect).

1. On validation, including `era` as a one-hot feature produces **lower** 
   PR-AUC (0.7312) than omitting it (0.7325); a difference of -0.0013. 
   The model does not benefit from the era label; on this split it is 
   slightly worse.
2. The one-hot encoder is fit on train (2003–2019), which contains only 
   `v8` and `hybrid`. The ground-effect era (2022+) appears only in test. 
   An encoder fit on train cannot represent an unseen category; every 
   test row would be encoded as all-zeros, which is an artifact rather 
   than a signal.

`era` remains a column in the table for EDA and reporting, just not as 
a model input.

### Section 3.13: Feature-group ablation

We measure each feature group's contribution by dropping one group at a time, 
retraining the same model, and recording the drop in validation PR-AUC.

**Setup:** same model as the baseline (logistic regression), same 
preprocessing (impute median + scale), same train/val split. Only the feature 
set changes.

**Uses `train_eval()`** (from `src/f1_eval.py`) so the metrics are 
directly comparable to the baseline model's comparison table.

**Reading the results:** a large drop means the group is important. A near-zero 
or negative drop means the group adds nothing (or its signal is redundant with 
other groups).

In [ ]:
import sys
# copied from src/f1_eval.py
import pandas as pd
from sklearn.metrics import (average_precision_score, f1_score, precision_score,
                             recall_score, roc_auc_score)


def evaluate(model, X, y, threshold=0.5):
    """Score an already-fitted model on one split and return the 5 metrics.
    ROC-AUC and PR-AUC use the probabilities (no threshold needed);
    F1, precision and recall use the yes/no prediction at `threshold`."""
    proba = model.predict_proba(X)[:, 1]          # predicted probability of a podium
    pred = (proba >= threshold).astype(int)       # 1 = "podium" at this threshold
    return {
        "ROC-AUC": roc_auc_score(y, proba),
        "PR-AUC": average_precision_score(y, proba),
        "F1": f1_score(y, pred, zero_division=0),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
    }


def train_eval(model, X_train, y_train, X_val, y_val, threshold=0.5, name="model"):
    """Fit `model` on the training years only, then score it on train and validation.
    The test set is not used here on purpose: it is scored once, at the end, with evaluate()."""
    model.fit(X_train, y_train)
    rows = [
        {"model": name, "split": "train", **evaluate(model, X_train, y_train, threshold)},
        {"model": name, "split": "validation", **evaluate(model, X_val, y_val, threshold)},
    ]
    return pd.DataFrame(rows).round(4)


from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Same preprocessing + model for every run (impute, scale, logistic regression)
def make_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=2000, random_state=SEED)),
    ])

print("setup ready")

In [ ]:
results = {}

# Baseline: all 17 features
df = train_eval(
    make_pipeline(),
    train_df[FEATURES], y_train,
    val_df[FEATURES],   y_val,
    name="baseline (all 17)",
)
baseline_pr = df[df["split"] == "validation"]["PR-AUC"].iloc[0]
results["baseline"] = baseline_pr

print(f"Baseline validation PR-AUC (all 17 features): {baseline_pr:.4f}")
print(df.round(4))

In [ ]:
results = {"baseline": baseline_pr}

for group, cols in FEATURE_GROUPS.items():
    dropped_features = [f for f in FEATURES if f not in cols]

    df = train_eval(
        make_pipeline(),
        train_df[dropped_features], y_train,
        val_df[dropped_features],   y_val,
        name=f"without {group}",
    )
    val_pr = df[df["split"] == "validation"]["PR-AUC"].iloc[0]
    results[f"without {group}"] = val_pr
    print(f"without {group:12s} (dropped {len(cols)} features): PR-AUC = {val_pr:.4f}  drop = {baseline_pr - val_pr:+.4f}")

In [ ]:
import pandas as pd

rows = []
for name, pr in results.items():
    if name == "baseline":
        continue
    rows.append({
        "dropped group": name.replace("without ", ""),
        "PR-AUC": round(pr, 4),
        "drop vs baseline": round(baseline_pr - pr, 4),
    })

ablation_table = pd.DataFrame(rows).sort_values("drop vs baseline", ascending=False)
print(f"Baseline PR-AUC: {baseline_pr:.4f}")
print()
print(ablation_table.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
sorted_table = ablation_table.sort_values("drop vs baseline")
values = sorted_table["drop vs baseline"]
labels = sorted_table["dropped group"]

bars = ax.barh(labels, values, color="tab:blue")

# Place labels with proper alignment based on sign
for bar, val in zip(bars, values):
    if val >= 0:
        ax.text(val + 0.0004, bar.get_y() + bar.get_height()/2,
                f"{val:+.4f}", va="center", ha="left", fontsize=9)
    else:
        ax.text(val - 0.0004, bar.get_y() + bar.get_height()/2,
                f"{val:+.4f}", va="center", ha="right", fontsize=9)

ax.axvline(0, color="black", linestyle="-", linewidth=0.8)

# Give some padding so labels don't touch the plot edges
x_min, x_max = values.min(), values.max()
padding = (x_max - x_min) * 0.15
ax.set_xlim(x_min - padding, x_max + padding)

ax.set_title("Feature-group ablation: validation PR-AUC drop when group is removed")
ax.set_xlabel("PR-AUC drop vs baseline (larger = more important)")
ax.set_ylabel("")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("reports/fig_p3_ablation.png", dpi=120, bbox_inches="tight")
plt.show()

### 3.13 findings

Baseline validation PR-AUC (all 17 features): **0.7325**

| Dropped group | PR-AUC | Drop vs baseline |
|---|---|---|
| grid | 0.7183 | +0.0142 |
| qualifying | 0.7196 | +0.0129 |
| standings | 0.7209 | +0.0116 |
| circuit | 0.7275 | +0.0050 |
| form | 0.7303 | +0.0022 |
| context | 0.7355 | **-0.0030** |

**Interpretation**

*Ranking:* grid > qualifying > standings > circuit > form > context.

*All drops are small* (max 0.014). This is expected because features are 
correlated within and across groups: removing `grid` still leaves 
`qual_position` (~0.9 correlated), so the model compensates. The sum of the 
drops (0.043) is far smaller than the total signal, confirming that groups 
share information.

*Context is a negative drop* (-0.0030): removing `home_flag` slightly 
*improves* PR-AUC. This is within noise, but consistent with Q2 finding 
that home advantage disappears once you control for grid. `home_flag` is 
weak; keeping it is harmless, dropping it is also defensible.

**Takeaway.** Grid, qualifying, and standings are the core predictors. 
Circuit and form add modest value. Context (`home_flag`) adds nothing 
measurable, retained for completeness, flagged as weak.

---
# >>> Section from `notebooks/p4_models.ipynb`


## Part 4: Models, metrics and XAI (P4)

This part trains and compares the models on P3's modeling table, chooses the metrics, the hyperparameters and the decision threshold **without touching the test set**, evaluates the final choice once on the test set, explains the final model and lists its limitations.

**Inputs from Part 3:** the modeling table `model` (one row per driver per race, 2003-2024, with the `split` column), the 17 features `FEATURES` and their 6 groups `FEATURE_GROUPS`.
**Outputs for P1's inference function:** `FINAL_MODEL` (preprocessing + model in one pipeline), `FINAL_THRESHOLD` and `FINAL_NAME`.

Sections: 4.1) evaluation function `train_eval()`, 4.2) data and preprocessing, 4.3) sanity check of the metrics, 4.4) baselines, 4.5) statistical models, 4.6) shallow neural network, 4.7) hyperparameter tuning, 4.8) metric choice and threshold, 4.9) final model, 4.10) test set evaluation, 4.11) explainability (XAI), 4.12) model limitations, 4.13) hand-off to the inference function.

In [ ]:
clean = load_clean()
outcomes = clean["outcomes"]    # post-race columns (status, finishing position...). Never features: only used to tell the stories in 4.11-4.12
tables = clean["tables"]        # cleaned tables (driver and race names for the examples)

### 4.1 The evaluation function `train_eval()`

One function scores every model the same way, so all results (our models and P3's ablation) are directly comparable.

**Why these metrics?** The brief asks for ROC-AUC, PR-AUC and F1 for every model; we also report precision and recall, because F1 alone hides which of the two is weak.
* **ROC-AUC:** how well the model separates podium from non-podium overall (uses probabilities, no threshold).
* **PR-AUC:** how well the model ranks the real podium finishers at the top (uses probabilities). Computed as *average precision*, the standard summary of the precision-recall curve. It is more informative than ROC-AUC when the positive class is rare, and podiums are rare (about 14% of rows).
* **F1, precision, recall:** quality of the yes/no decision at a probability threshold. The threshold is 0.5 until it is tuned on validation in 4.8.

**Leakage safety:** the model is fitted on the training years only. The test set is deliberately *not* an input: it is scored once, at the very end, with `evaluate()`.

The same code is in `src/f1_eval.py`, so P3 can import it for the ablation.

In [ ]:
import pandas as pd
from sklearn.metrics import (average_precision_score, f1_score, precision_score,
                             recall_score, roc_auc_score)


def evaluate(model, X, y, threshold=0.5):
    """Score an already-fitted model on one split and return the 5 metrics.
    ROC-AUC and PR-AUC use the probabilities (no threshold needed);
    F1, precision and recall use the yes/no prediction at `threshold`."""
    proba = model.predict_proba(X)[:, 1]          # predicted probability of a podium
    pred = (proba >= threshold).astype(int)       # 1 = "podium" at this threshold
    return {
        "ROC-AUC": roc_auc_score(y, proba),
        "PR-AUC": average_precision_score(y, proba),
        "F1": f1_score(y, pred, zero_division=0),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
    }


def train_eval(model, X_train, y_train, X_val, y_val, threshold=0.5, name="model"):
    """Fit `model` on the training years only, then score it on train and validation.
    The test set is not used here on purpose: it is scored once, at the end, with evaluate()."""
    model.fit(X_train, y_train)
    rows = [
        {"model": name, "split": "train", **evaluate(model, X_train, y_train, threshold)},
        {"model": name, "split": "validation", **evaluate(model, X_val, y_val, threshold)},
    ]
    return pd.DataFrame(rows).round(4)

### 4.2 Data and preprocessing

**Data:** P3's modeling table and P3's time split (train 2003-2019, validation 2020-2021, test 2022-2024). All models use the same 17 features, all numeric. The test rows are separated only in 4.10.

**Preprocessing (one pipeline, the same for every model):**
* **Missing values are structural, not errors:** standings are empty in round 1 of every season (no race before it), form is empty for a driver's first race, and qualifying times are missing for about 1-2% of rows. P3 left them as `NaN` on purpose. We fill them with the **training median** and add a **missing-indicator column** for every feature that has gaps, so the model still knows that the value was missing (for example "first race of the season").
* **Scaling:** every feature is standardized (mean 0, standard deviation 1). Logistic regression and the network need it; for the tree models it changes nothing (trees only compare values within one feature), so all models can share one pipeline.
* **Fitted on the training rows only:** the imputer and the scaler are steps of a `Pipeline`, so `fit()` learns the medians, means and standard deviations from the training years only, and the validation and test rows are only transformed.
* **Difference from P3's experiments:** P3's preprocessing experiments and ablation (3.12-3.13) use plain median imputation without missing-value flags. Our models add the flags (`add_indicator=True`), so a missing value stays visible to the model. The flags exist only for the 9 features that have gaps in the training years; a feature that is never missing in training gets no flag (its flag column would be all zeros, so the model could not learn anything from it). This is why P3's logistic-regression baseline (validation PR-AUC 0.7325) and ours (4.4) differ slightly.
* **`era` is not a model input** (P3, 3.8 and 3.12): the training years contain only the `v8` and `hybrid` eras and the test years only `ground_effect`, so no model can learn anything about the test era from this column.

In [ ]:
import numpy as np
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler

table = model                     # P3's modeling table (read only; the name `model` is kept for Part 3)
assert not set(FEATURES) & set(outcomes.columns), "a post-race column is used as a feature"

train = table[table["split"] == "train"]
val = table[table["split"] == "validation"]
X_train, y_train = train[FEATURES], train["podium"]
X_val, y_val = val[FEATURES], val["podium"]

assert all(pd.api.types.is_numeric_dtype(table[f]) for f in FEATURES), "all features must be numeric"
preprocess = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler())


def with_preprocessing(estimator):
    """The shared preprocessing followed by `estimator`, as one pipeline (fitted on the training rows only)."""
    return Pipeline([("prep", clone(preprocess)), ("model", estimator)])


for name, part in [("train", train), ("validation", val)]:
    print(f"{name}: {len(part)} rows ({part['year'].min()}-{part['year'].max()}), "
          f"{part['podium'].sum()} podiums, podium rate {part['podium'].mean():.3f}")
display(X_train.isna().mean().loc[lambda s: s > 0].round(3).to_frame("share missing (train)"))

### 4.3 Sanity check of the metrics

Before trusting the function, we test it on a model that knows nothing: `DummyClassifier` always predicts the training podium rate. A correct implementation must give **ROC-AUC = 0.5** (no separation), **PR-AUC = the podium rate** (no better than random ranking) and **F1 = 0** at threshold 0.5 (it never says "podium"). We also print its accuracy, to show why accuracy is not one of our metrics.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy = DummyClassifier(strategy="prior", random_state=SEED)
dummy_results = train_eval(dummy, X_train, y_train, X_val, y_val, name="dummy (always the base rate)")
display(dummy_results)

val_row = dummy_results[dummy_results["split"] == "validation"].iloc[0]
assert abs(val_row["ROC-AUC"] - 0.5) < 1e-3
assert abs(val_row["PR-AUC"] - y_val.mean()) < 1e-3
assert val_row["F1"] == 0
print(f"Dummy accuracy on validation: {accuracy_score(y_val, dummy.predict(X_val)):.1%} "
      f"(looks good, but it finds 0 podiums)")

**What we see.** The dummy model gets ROC-AUC 0.5, PR-AUC equal to the podium rate (0.151 on validation) and F1 = 0, exactly as expected, so the function computes the metrics correctly. Its accuracy is still 84.9%, although it never finds a single podium. That is why accuracy is not one of our metrics, and why **PR-AUC is the natural primary metric** for this rare positive class. These numbers are the floor every real model must beat.

### 4.4 Baselines

A baseline is the simplest reasonable approach. Every later model must beat it, otherwise its extra complexity is not earned.
* **Rule `grid <= 3`:** predict a podium for the three cars that start at the front. It learns nothing; it encodes the domain fact that the starting position is the strongest single predictor.
* **Logistic regression:** the simplest statistical model. It multiplies each feature by a learned weight, adds them up, and turns the sum into a probability with the sigmoid function. *Limitation:* it is linear, so it cannot learn interactions on its own (for example "a good grid slot matters more for a top team").

#### Attempt 1: rule `grid <= 3`
The rule gives a hard yes/no, so its ROC-AUC and PR-AUC are those of a single operating point. `grid = 0` (pit-lane start) is not a front-row start.

In [ ]:
class GridRule:
    """Rule baseline: 'podium' if the car starts in grid slot 1, 2 or 3. Learns nothing."""
    def fit(self, X, y):
        return self

    def predict_proba(self, X):
        p = X["grid"].between(1, 3).astype(float).to_numpy()
        return np.column_stack([1 - p, p])

rule_results = train_eval(GridRule(), X_train, y_train, X_val, y_val, name="rule grid <= 3")
display(rule_results)

#### Attempt 2: logistic regression (all 17 features)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = with_preprocessing(LogisticRegression(max_iter=2000, random_state=SEED))
lr_results = train_eval(lr, X_train, y_train, X_val, y_val, name="logistic regression")
display(lr_results)

#### Attempt 3: logistic regression with `class_weight="balanced"`
Podiums are only about 14% of the training rows. `class_weight="balanced"` gives the podium rows a larger weight, so both classes count equally in training. We expect the ranking metrics (ROC-AUC, PR-AUC) to stay about the same and the yes/no metrics at threshold 0.5 to change, because the weighting mostly moves the predicted probabilities up.

In [ ]:
lr_bal = with_preprocessing(LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
lr_bal_results = train_eval(lr_bal, X_train, y_train, X_val, y_val, name="logistic regression (balanced)")
display(lr_bal_results)

#### Attempt 4: logistic regression on the 4 draft features only
Before P3's table was ready, the models were built on four raw pre-race columns (`grid`, `pitlane_start`, `has_qualifying`, `qual_position`). Same model, same rows: does the feature engineering add anything?

In [ ]:
DRAFT_FEATURES = ["grid", "pitlane_start", "has_qualifying", "qual_position"]
lr_draft = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(max_iter=2000, random_state=SEED))
lr_draft_results = train_eval(lr_draft, train[DRAFT_FEATURES].astype(float), y_train,
                              val[DRAFT_FEATURES].astype(float), y_val, name="logistic regression (4 draft features)")
display(lr_draft_results)

#### Comparison of the baselines (validation)

In [ ]:
import matplotlib.pyplot as plt

baselines = pd.concat([dummy_results, rule_results, lr_results, lr_bal_results, lr_draft_results], ignore_index=True)
val_table = baselines[baselines["split"] == "validation"].set_index("model").drop(columns="split")
display(val_table)

ax = val_table[["ROC-AUC", "PR-AUC", "F1"]].plot.barh(figsize=(8, 4))
ax.set_title("Baselines on validation (2020-2021), threshold 0.5")
ax.set_xlabel("score (higher is better)")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.legend(title="metric", loc="lower right")
plt.tight_layout()
plt.show()

# what the logistic regression learned: weights of the standardized inputs, largest first
weights = pd.Series(lr.named_steps["model"].coef_[0], index=lr.named_steps["prep"].get_feature_names_out())
display(weights.sort_values(key=abs, ascending=False).head(10).round(3).to_frame("weight (standardized input)"))

**What we see (validation 2020-2021).**
* **Logistic regression is far above both references:** PR-AUC 0.738 vs 0.530 for the rule and 0.151 for the dummy; ROC-AUC 0.933 vs 0.820.
* **P3's features add real information:** the same logistic regression on the same rows reaches PR-AUC 0.672 with the 4 draft features and **0.738 with the 17 features (+0.066)**. Which group brings how much is measured by P3's ablation (3.13) and by the XAI in 4.11.
* **Same model as P3's baseline, slightly different number:** P3's logistic regression (3.13) gets 0.7325, ours 0.7380. The only difference is the missing-value flags our pipeline adds (4.2).
* **At threshold 0.5 the rule still has the best F1** (0.695 vs 0.688): plain logistic regression is cautious at 0.5 (precision 0.73, recall 0.65). The threshold matters as much as the model, so it is tuned in 4.8.
* **`class_weight="balanced"` behaves as expected:** PR-AUC barely changes (0.733 vs 0.738) because the ranking stays the same, while at 0.5 recall rises to 0.85 and precision falls to 0.53. Weighting mostly moves the probabilities, which threshold tuning can also do.
* **Weights:** the largest are `qual_position` (-1.18) and `grid` (-0.91): a position nearer the front means a higher podium probability; then the championship position before the race, `pos_before` (-0.47). Several features carry the same information (`grid` and `qual_position`; `reached_q2`, `reached_q3` and `qual_position`; points and positions), so the sign of a single weight should not be over-read. These are associations the model uses, not causes.
* **Train PR-AUC is a little lower than validation** (0.691 vs 0.738). The training years 2003-2019 include seasons with less predictable results; with P3's start year (2003) this gap is much smaller than with the draft data that started in 1950 (about -0.14).

**Baseline to beat:** logistic regression, validation PR-AUC **0.738**.

### 4.5 Statistical models: random forest and gradient boosting

Two tree-based models, chosen because they can learn what logistic regression cannot: non-linear effects and interactions between features (for example "a good grid slot matters more for a team in good form"). Both are in scikit-learn, so no extra library is needed.
* **Random forest:** many decision trees, each trained on a random sample of rows and features; the prediction is the average vote. Averaging many different trees reduces overfitting. *Limitation:* its probabilities are averages of votes (not calibrated), and it cannot extrapolate beyond the values seen in training.
* **Gradient boosting** (`HistGradientBoostingClassifier`): small trees built one after another, each one correcting the errors of the trees before it. Usually the strongest model family on tables like ours. *Limitation:* more hyperparameters, and it can overfit if it runs too long (scikit-learn stops early on a 10% slice of the training rows, never on our validation years).

**Class imbalance:** both use `class_weight="balanced"` (podiums are about 14% of rows). Section 4.4 showed this mostly moves the probabilities; the threshold is tuned in 4.8. Hyperparameters are tuned in 4.7.

#### Attempt 5: random forest
300 trees; `min_samples_leaf=20` so every leaf is based on at least 20 races, which keeps the probabilities smooth instead of memorizing single rows.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = with_preprocessing(RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight="balanced",
                                               n_jobs=-1, random_state=SEED))
rf_results = train_eval(rf, X_train, y_train, X_val, y_val, name="random forest")
display(rf_results)

#### Attempt 6: gradient boosting
Default settings for now.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = with_preprocessing(HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED))
gb_results = train_eval(gb, X_train, y_train, X_val, y_val, name="gradient boosting")
display(gb_results)

#### Comparison of all models so far
Validation scores for every attempt, plus the train-validation gap in PR-AUC (a large positive gap means overfitting).

In [ ]:
def compare(results):
    """Validation scores of every model, plus the train - validation PR-AUC gap."""
    allres = pd.concat(results, ignore_index=True)
    pr = allres.pivot(index="model", columns="split", values="PR-AUC")
    summary = allres[allres["split"] == "validation"].set_index("model").drop(columns="split")
    summary["PR-AUC gap (train - val)"] = (pr["train"] - pr["validation"]).round(4)
    return summary.sort_values("PR-AUC", ascending=False)

summary = compare([dummy_results, rule_results, lr_results, lr_bal_results, lr_draft_results, rf_results, gb_results])
display(summary)

lr_pr = summary.loc["logistic regression", "PR-AUC"]
ax = summary["PR-AUC"].sort_values().plot.barh(figsize=(8, 4), color="tab:orange")
ax.axvline(lr_pr, color="black", linestyle="--", label=f"baseline to beat (logistic regression) = {lr_pr:.4f}")
ax.set_title("PR-AUC on validation (2020-2021), all models so far")
ax.set_xlabel("PR-AUC (higher is better)")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**What we see (validation 2020-2021).**
* **The tree models do not beat logistic regression:** PR-AUC random forest 0.726, gradient boosting 0.706, logistic regression 0.738; ROC-AUC is 0.929-0.933 for all three.
* **Gradient boosting with default settings overfits:** train PR-AUC 0.925 vs validation 0.706 (gap +0.22). With about 960 training podiums, trees with 31 leaves learn details of the training seasons that do not repeat. The random forest (`min_samples_leaf=20`) overfits much less (gap +0.047). Tuning (4.7) controls the tree size.
* **Why no gain from trees:** the strongest signal is a smooth "the nearer the front, the likelier the podium" relation that a straight line already captures; the extra interactions the trees can learn are small compared with the noise of 117 validation podiums.
* **At threshold 0.5** both balanced tree models behave like the balanced logistic regression: high recall (0.76-0.88), lower precision (0.53-0.60).

**Decision:** keep both as our two statistical models and tune them in 4.7. The baseline to beat is still logistic regression.

### 4.6 Shallow neural network (PyTorch)

**How it works:** a feed-forward network. Each hidden layer computes weighted sums of its inputs and passes them through a ReLU (negative values become 0), which lets the network learn non-linear patterns. The output layer gives one score that the sigmoid turns into a podium probability. The weights are learned by gradient descent (Adam) on the binary cross-entropy loss. With no hidden layer this would be exactly logistic regression, so the hidden layers are what the network adds.

**Choices** ("shallow" = 1-2 hidden layers, as the brief asks):
* **Dropout 0.2:** during training, 20% of the hidden units are switched off at random in every batch, so the network cannot rely on single units (less overfitting).
* **Early stopping on validation:** after every epoch (one pass over the training rows) we compute the validation loss; training stops when it has not improved for 15 epochs, and the weights of the best epoch are kept.
* **Class imbalance:** `pos_weight` = (non-podium rows / podium rows) in the loss, the same idea as `class_weight="balanced"`.
* **Preprocessing:** the same pipeline as the other models (4.2), fitted on the training rows only inside `fit()`.
* **Reproducible:** `torch.manual_seed(SEED)`, a seeded batch order and a single CPU thread (the number of threads changes the rounding of the sums), so Run All gives the same numbers on a laptop and on Kaggle.

*Limitations:* many settings (layers, units, dropout, learning rate), needs scaling and imputation, a black box without XAI, and on small tables it rarely beats tree models. **Caveat:** the validation set chooses the stopping epoch, so the network's validation score is slightly optimistic compared with the other models; the test set gives the fair comparison.

The class below only wraps the network with `fit` / `predict_proba`, so the same `train_eval()` scores it.

In [ ]:
import torch
from torch import nn


class ShallowNN:
    """Feed-forward network with 1-2 hidden layers, wrapped with fit / predict_proba so train_eval() can score it.
    - preprocessing: the shared pipeline of 4.2, fitted on the training rows only, inside fit()
    - early stopping: training stops when the validation loss has not improved for `patience` epochs,
      and the weights of the best epoch are kept"""

    def __init__(self, X_val, y_val, hidden=(32,), dropout=0.2, lr=1e-3, batch_size=256,
                 max_epochs=200, patience=15, seed=SEED):
        self.X_val, self.y_val = X_val, y_val
        self.hidden, self.dropout, self.lr, self.batch_size = hidden, dropout, lr, batch_size
        self.max_epochs, self.patience, self.seed = max_epochs, patience, seed

    def _build(self, n_inputs):
        layers, size = [], n_inputs
        for h in self.hidden:                                   # hidden layer: linear -> ReLU -> dropout
            layers += [nn.Linear(size, h), nn.ReLU(), nn.Dropout(self.dropout)]
            size = h
        layers.append(nn.Linear(size, 1))                       # output: one score (logit) for "podium"
        return nn.Sequential(*layers)

    def _tensor(self, X):
        return torch.tensor(self.prep.transform(X), dtype=torch.float32)

    def fit(self, X, y):
        torch.manual_seed(self.seed)
        torch.set_num_threads(1)                                # same numbers on every machine
        self.prep = clone(preprocess).fit(X)
        Xt, yt = self._tensor(X), torch.tensor(np.asarray(y), dtype=torch.float32)
        Xv, yv = self._tensor(self.X_val), torch.tensor(np.asarray(self.y_val), dtype=torch.float32)

        self.net = self._build(Xt.shape[1])
        pos_weight = (yt == 0).sum() / (yt == 1).sum()          # class imbalance, like class_weight="balanced"
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(self.net.parameters(), lr=self.lr)
        gen = torch.Generator().manual_seed(self.seed)

        self.history = {"train": [], "validation": []}
        best_loss, best_state, waited = float("inf"), None, 0
        for epoch in range(self.max_epochs):
            self.net.train()                                   # dropout on
            batch_losses = []
            for idx in torch.randperm(len(Xt), generator=gen).split(self.batch_size):
                optimizer.zero_grad()
                loss = loss_fn(self.net(Xt[idx]).squeeze(1), yt[idx])
                loss.backward()
                optimizer.step()
                batch_losses.append(loss.item())
            self.net.eval()                                    # dropout off
            with torch.no_grad():
                val_loss = loss_fn(self.net(Xv).squeeze(1), yv).item()
            self.history["train"].append(float(np.mean(batch_losses)))
            self.history["validation"].append(val_loss)

            if val_loss < best_loss:                           # early stopping on the validation loss
                best_loss, waited = val_loss, 0
                best_state = {k: v.clone() for k, v in self.net.state_dict().items()}
                self.best_epoch = epoch + 1
            else:
                waited += 1
                if waited >= self.patience:
                    break
        self.net.load_state_dict(best_state)                   # keep the best epoch, not the last one
        return self

    def predict_proba(self, X):
        self.net.eval()
        with torch.no_grad():
            p = torch.sigmoid(self.net(self._tensor(X)).squeeze(1)).numpy()
        return np.column_stack([1 - p, p])

#### Attempt 7: one hidden layer (32 units)

In [ ]:
nn1 = ShallowNN(X_val, y_val, hidden=(32,))
nn1_results = train_eval(nn1, X_train, y_train, X_val, y_val, name="neural network (32)")
display(nn1_results)
print(f"stopped after {len(nn1.history['train'])} epochs, best epoch: {nn1.best_epoch}")

#### Attempt 8: two hidden layers (64 and 32 units)

In [ ]:
nn2 = ShallowNN(X_val, y_val, hidden=(64, 32))
nn2_results = train_eval(nn2, X_train, y_train, X_val, y_val, name="neural network (64-32)")
display(nn2_results)
print(f"stopped after {len(nn2.history['train'])} epochs, best epoch: {nn2.best_epoch}")

#### Loss curves
Training loss (average over the batches of each epoch, with dropout on) and validation loss per epoch. The dashed line marks the epoch whose weights were kept.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, net, title in [(axes[0], nn1, "1 hidden layer (32)"), (axes[1], nn2, "2 hidden layers (64-32)")]:
    epochs = range(1, len(net.history["train"]) + 1)
    ax.plot(epochs, net.history["train"], label="training loss")
    ax.plot(epochs, net.history["validation"], label="validation loss")
    ax.axvline(net.best_epoch, color="black", linestyle="--", label=f"best epoch ({net.best_epoch})")
    ax.set_title(f"Neural network, {title}")
    ax.set_xlabel("epoch")
    ax.legend()
axes[0].set_ylabel("weighted binary cross-entropy")
plt.tight_layout()
plt.show()

#### Comparison of all models so far (validation)

In [ ]:
display(compare([dummy_results, rule_results, lr_results, lr_bal_results, lr_draft_results,
                 rf_results, gb_results, nn1_results, nn2_results]))

**What we see (validation 2020-2021).**
* **The networks have the highest validation PR-AUC so far:** 0.758 with one hidden layer and 0.753 with two, vs 0.738 for logistic regression (+0.020). We do **not** call this a win yet: the validation set chose the stopping epoch (so the network's score is slightly optimistic), and with 117 validation podiums a difference this small is within noise (measured in 4.9).
* **Two layers vs one:** the second layer does not help (0.753 vs 0.758), so the simpler one-layer network is preferred.
* **Loss curves:** both losses fall quickly in the first 5-10 epochs and then flatten. The validation loss never turns clearly upward, and early stopping keeps the best epoch (65 for one layer, 24 for two layers), so dropout and early stopping prevent overfitting. The validation loss stays a little above the training loss, a normal small gap, and the train-validation PR-AUC gap is small (-0.04 to -0.05, like logistic regression).
* **At threshold 0.5** the networks behave like the other balanced models: high recall (0.86-0.87), low precision (0.49-0.52).

### 4.7 Hyperparameter tuning (validation only)

For each model we try a small grid of the settings that control its complexity. Every combination is trained on the training years and scored on validation with `train_eval()`; the best combination is the one with the highest **validation PR-AUC** (our primary metric, see 4.8). The test years are not touched.

The grids are deliberately small: the validation set has only about 117 podiums, so a large search would mostly fit its noise.

In [ ]:
from sklearn.model_selection import ParameterGrid

def tune(name, make_model, grid):
    """Train one model per combination in `grid` (training years) and score it on validation.
    Returns the fitted model with the best validation PR-AUC and the table of all results."""
    rows, best_model, best_pr = [], None, -1
    for params in ParameterGrid(grid):
        candidate = make_model(**params)
        res = train_eval(candidate, X_train, y_train, X_val, y_val, name=name).set_index("split")
        rows.append({**params, "train PR-AUC": res.loc["train", "PR-AUC"],
                     "val PR-AUC": res.loc["validation", "PR-AUC"], "val ROC-AUC": res.loc["validation", "ROC-AUC"]})
        if res.loc["validation", "PR-AUC"] > best_pr:
            best_model, best_pr = candidate, res.loc["validation", "PR-AUC"]
    return best_model, pd.DataFrame(rows).sort_values("val PR-AUC", ascending=False, ignore_index=True)

tuned = {}   # model name -> (best fitted model, table of all combinations)

#### Attempt 9: tuning logistic regression
`C` is the inverse regularization strength: a small `C` pushes the weights towards 0 (simpler model), a large `C` lets them grow.

In [ ]:
tuned["logistic regression"] = tune(
    "logistic regression",
    lambda C: with_preprocessing(LogisticRegression(C=C, max_iter=2000, random_state=SEED)),
    {"C": [0.01, 0.1, 1, 10]})
display(tuned["logistic regression"][1])

#### Attempt 10: tuning the random forest
`min_samples_leaf`: the minimum number of races behind each leaf (larger = smoother, less memorizing). `max_features`: how many inputs each split may look at (`"sqrt"` = a random subset, `1.0` = all of them; a subset makes the trees more different from each other).

In [ ]:
tuned["random forest"] = tune(
    "random forest",
    lambda min_samples_leaf, max_features: with_preprocessing(
        RandomForestClassifier(n_estimators=300, min_samples_leaf=min_samples_leaf, max_features=max_features,
                               class_weight="balanced", n_jobs=-1, random_state=SEED)),
    {"min_samples_leaf": [5, 20, 50], "max_features": ["sqrt", 1.0]})
display(tuned["random forest"][1])

#### Attempt 11: tuning gradient boosting
`learning_rate`: how much each new tree may correct the previous ones (smaller = slower, more careful). `max_leaf_nodes`: the size of each tree (smaller = simpler trees).

In [ ]:
tuned["gradient boosting"] = tune(
    "gradient boosting",
    lambda learning_rate, max_leaf_nodes: with_preprocessing(HistGradientBoostingClassifier(
        learning_rate=learning_rate, max_leaf_nodes=max_leaf_nodes, class_weight="balanced", random_state=SEED)),
    {"learning_rate": [0.03, 0.1], "max_leaf_nodes": [7, 15, 31]})
display(tuned["gradient boosting"][1])

#### Attempt 12: tuning the neural network
`hidden`: the size of the hidden layer(s). `dropout`: the share of hidden units switched off during training. (The number of epochs is already chosen by early stopping.)

In [ ]:
tuned["neural network"] = tune(
    "neural network",
    lambda hidden, dropout: ShallowNN(X_val, y_val, hidden=hidden, dropout=dropout),
    {"hidden": [(16,), (32,), (64, 32)], "dropout": [0.2, 0.4]})
display(tuned["neural network"][1])

**What we see (tuning, validation PR-AUC).**
* **Logistic regression:** best with `C=10` (0.739); strong regularization (`C=0.01`) costs 0.02 (0.719). With 17 features and about 6,800 rows, little regularization is needed.
* **Random forest:** best with `max_features="sqrt"` and `min_samples_leaf=5`: 0.729 (0.726 before), but its train PR-AUC is 0.909, a large gap: small leaves memorize the training seasons.
* **Gradient boosting:** best with the **smallest trees** (`max_leaf_nodes=7`, `learning_rate=0.1`): 0.732 (0.706 with the defaults), and the overfitting of 4.5 mostly disappears (train 0.749). The six settings are ordered by tree size: the bigger the trees, the worse the validation score.
* **Neural network:** best with one hidden layer (32 units) and dropout 0.2: 0.758, the setting of Attempt 7.
* **Tuning gains are small** (at most +0.026, for gradient boosting), and the four tuned models end within 0.03 of each other, less than the noise of the validation PR-AUC (standard error 0.039, see 4.9).

### 4.8 Metric choice and decision threshold

**Primary metric: PR-AUC.** Podiums are rare (about 15% of the validation rows). PR-AUC measures how well the real podium finishers are ranked at the top, it does not depend on a threshold, and its floor is the podium rate (the dummy in 4.3) instead of 0.5, so it shows clearly how much a model adds.

**Secondary metric: F1 at the chosen threshold.** In the end the model must say yes or no; F1 balances precision (how many predicted podiums are right) and recall (how many real podiums are found).

ROC-AUC is reported for completeness, but with about 85% non-podium rows it looks good easily (about 0.8 even for the simple rule).

**Threshold:** for every tuned model we try thresholds from 0.05 to 0.95 (step 0.01) on validation and keep the one with the highest F1. Each model gets its own threshold, because class weighting shifts the probabilities (4.4).

In [ ]:
def best_threshold(clf, X, y):
    """Threshold between 0.05 and 0.95 (step 0.01) with the highest F1 on (X, y)."""
    proba = clf.predict_proba(X)[:, 1]
    candidates = np.round(np.arange(0.05, 0.96, 0.01), 2)
    f1s = [f1_score(y, proba >= t, zero_division=0) for t in candidates]
    return candidates[int(np.argmax(f1s))]

rows = []
for name, (clf, _) in tuned.items():
    t = best_threshold(clf, X_val, y_val)
    rows.append({"model": name, "threshold": t, **evaluate(clf, X_val, y_val, threshold=t)})
tuned_table = pd.DataFrame(rows).set_index("model").round(4)
thresholds = tuned_table["threshold"].to_dict()
display(tuned_table)

### 4.9 Choosing the final model (validation only)

Before comparing the models we measure how noisy the validation PR-AUC is: we resample the validation rows with replacement 1000 times (bootstrap) and take the standard deviation of PR-AUC as its standard error (SE).

**Rule (one-standard-error rule):** among the models whose validation PR-AUC is within one SE of the best, take the simplest (order: logistic regression < random forest < gradient boosting < neural network). A difference smaller than the noise is not a reason to accept a more complex model. The final model and its threshold are then used once on the test set (4.10) and by P1's inference function (4.13).

In [ ]:
COMPLEXITY = ["logistic regression", "random forest", "gradient boosting", "neural network"]   # simplest first
val_pr = tuned_table["PR-AUC"]
best_name = val_pr.idxmax()

# bootstrap standard error of the best model's validation PR-AUC
rng = np.random.default_rng(SEED)
proba, y = tuned[best_name][0].predict_proba(X_val)[:, 1], y_val.to_numpy()
boot = []
for _ in range(1000):
    i = rng.integers(0, len(y), len(y))
    boot.append(average_precision_score(y[i], proba[i]))
se = np.std(boot)

within_one_se = [name for name in COMPLEXITY if val_pr[name] >= val_pr[best_name] - se]
FINAL_NAME = within_one_se[0]
FINAL_MODEL, FINAL_THRESHOLD = tuned[FINAL_NAME][0], thresholds[FINAL_NAME]

print(f"best validation PR-AUC: {best_name} = {val_pr[best_name]:.4f}, bootstrap SE = {se:.4f} "
      f"(95% range {np.percentile(boot, 2.5):.3f}-{np.percentile(boot, 97.5):.3f})")
print(f"models within one SE: {within_one_se}")
print(f"final model: {FINAL_NAME}, threshold {FINAL_THRESHOLD}")

**How the final threshold was chosen:** precision, recall and F1 of the final model on validation for every threshold.

In [ ]:
proba = FINAL_MODEL.predict_proba(X_val)[:, 1]
candidates = np.round(np.arange(0.05, 0.96, 0.01), 2)
curve = pd.DataFrame([evaluate(FINAL_MODEL, X_val, y_val, threshold=t) for t in candidates], index=candidates)

ax = curve[["precision", "recall", "F1"]].plot(figsize=(8, 4))
ax.axvline(FINAL_THRESHOLD, color="black", linestyle="--", label=f"chosen threshold = {FINAL_THRESHOLD}")
ax.set_title(f"Final model ({FINAL_NAME}): metrics vs threshold on validation")
ax.set_xlabel("threshold (predict podium if probability >= threshold)")
ax.set_ylabel("score")
ax.set_ylim(0, 1)
ax.legend(loc="center right")
plt.tight_layout()
plt.show()

# compare the final yes/no decisions with the simple rule grid <= 3 from 4.4
rule_pred = X_val["grid"].between(1, 3).to_numpy()
disagree = int(((proba >= FINAL_THRESHOLD) != rule_pred).sum())
print(f"rows where the final model and the rule grid <= 3 disagree: {disagree} of {len(X_val)}")

**What we see (threshold and final model).**
* **Each model needs its own threshold:** 0.23 for logistic regression, 0.64 for the random forest, 0.68 for gradient boosting, 0.72 for the network. The balanced models have their probabilities pushed up by the class weights, so their best threshold is higher; plain logistic regression is cautious, so its best threshold is below 0.5.
* **With its own threshold every model reaches a validation F1 of about 0.71-0.73** (gradient boosting 0.727, network 0.725, logistic regression 0.713, random forest 0.707).
* **Final model:** the bootstrap standard error of the validation PR-AUC is 0.039 (95% range 0.68-0.83). All four models are within one standard error of the best (the network, 0.758), so the one-standard-error rule picks the simplest: **logistic regression with threshold 0.23**.
* **Threshold plot:** on validation F1 is almost flat between about 0.23 and 0.50 (0.69-0.71); 0.23 is the highest point at the low end of that plateau, where recall is high and precision lower.
* **More than the rule:** the final model disagrees with the rule `grid <= 3` on 45 of 774 validation rows, so with P3's features it no longer just repeats "the front three on the grid" (with the 4 draft features it disagreed on only 2 rows).

### 4.10 Final evaluation on the test set (used once)

Until now the test years (2022-2024) were never used: every model, hyperparameter, threshold and the final choice were fixed on the training and validation years. Here each tuned model is scored **once** on the test set with the threshold it got on validation, next to its train and validation scores, so we can see whether the validation results hold on unseen seasons. Nothing is changed after this point. The rule `grid <= 3` is included as the reference to beat.

**Honest note on the test set.** The test set was first scored when P3's table still had 18 features, including `era`. That run showed that the test-year probabilities were pushed upwards (about 5 predicted podiums per race instead of 3), because the test era `ground_effect` never occurs in the training years. P3 then removed `era` from the features (3.8 and 3.12). The removal is justified without the test labels: a category that never occurs in training cannot be learned, and on validation `era` adds nothing (the context group's permutation importance was -0.003; P3's Experiment 3: validation PR-AUC 0.7312 with `era`, 0.7325 without). The numbers below are for the final 17 features. **Nothing else was changed after looking at the test set:** the models, the hyperparameter grids, the threshold rule and the final-model rule are the same as in that first run, and every choice is still made on validation.

In [ ]:
test = table[table["split"] == "test"]
X_test, y_test = test[FEATURES], test["podium"]
print(f"test: {len(X_test)} rows ({test['year'].min()}-{test['year'].max()}), {y_test.sum()} podiums, "
      f"podium rate {y_test.mean():.3f} (= the PR-AUC of a model that knows nothing)")

final_models = {"rule grid <= 3": (GridRule(), 0.5),
                **{name: (clf, thresholds[name]) for name, (clf, _) in tuned.items()}}
rows = []
for name, (clf, t) in final_models.items():
    label = f"{name} (final)" if name == FINAL_NAME else name
    for split, X, y in [("train", X_train, y_train), ("validation", X_val, y_val), ("test", X_test, y_test)]:
        rows.append({"model": label, "split": split, "threshold": t, **evaluate(clf, X, y, threshold=t)})
comparison = pd.DataFrame(rows).round(4)
display(comparison)

# the two metrics we chose (4.8), side by side per split
display(comparison.pivot(index="model", columns="split", values=["PR-AUC", "F1"])
        .reindex(columns=["train", "validation", "test"], level=1))

#### ROC and precision-recall curves (test set)
The ROC curve shows the true-positive rate against the false-positive rate for every threshold; the precision-recall curve shows precision against recall. The area under each curve is the ROC-AUC and the PR-AUC (`AP`) in the table.

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(13, 5))
for name, (clf, _) in tuned.items():
    proba = clf.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(y_test, proba, name=name, ax=ax_roc)
    PrecisionRecallDisplay.from_predictions(y_test, proba, name=name, ax=ax_pr)
ax_roc.plot([0, 1], [0, 1], color="black", linestyle="--", label="random guess (AUC = 0.5)")
ax_pr.axhline(y_test.mean(), color="black", linestyle="--", label=f"podium rate = {y_test.mean():.3f}")
ax_roc.set_title("ROC curves on the test set (2022-2024)")
ax_pr.set_title("Precision-recall curves on the test set (2022-2024)")
ax_roc.legend(loc="lower right")
ax_pr.legend(loc="lower left")
plt.tight_layout()
plt.show()

#### Confusion matrices (test set, threshold chosen on validation)
Rows are the real outcome, columns the prediction. Top-right = false alarms (predicted a podium that did not happen), bottom-left = missed podiums.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, (clf, _)) in zip(axes, tuned.items()):
    pred = (clf.predict_proba(X_test)[:, 1] >= thresholds[name]).astype(int)
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["no podium", "podium"],
                                            ax=ax, colorbar=False)
    ax.set_title(f"{name}{' (final)' if name == FINAL_NAME else ''}\nthreshold {thresholds[name]}")
plt.tight_layout()
plt.show()

**What we see (test set 2022-2024, 1,351 rows, 204 podiums).**
* **The final model is the best of all models on unseen seasons, on both chosen metrics.** PR-AUC (ranking): logistic regression 0.711, network 0.701, gradient boosting 0.685, random forest 0.681; rule 0.474; no-skill floor 0.151. F1 (yes/no decisions): logistic regression 0.683, network 0.679, random forest 0.668, gradient boosting 0.656, rule 0.649. ROC-AUC is 0.93-0.94 for all four.
* **The one-standard-error rule paid off:** the network, best on validation (0.758), drops to 0.701 on test (its stopping epoch was chosen on validation), while the simplest model drops least (0.739 to 0.711).
* **The random forest overfitted:** train PR-AUC 0.909, test 0.681.
* **Final model in numbers (threshold 0.23):** it finds 177 of the 204 podiums (recall 0.87), with 137 false alarms (precision 0.56) and 27 missed podiums. The low threshold, chosen on validation for the best F1, favours finding podiums over avoiding false alarms.
* **Compared with the first test run with 18 features (see the note above):** PR-AUC is the same (0.712 then, 0.711 now), because `era` only pushed all test probabilities up by the same amount, which does not change the ranking. F1 rose from 0.581 to 0.683, because without that push fewer rows cross the threshold by mistake (4.12 shows the probabilities are now realistic on the test years).
* **No choice was made on the test set:** the final model and threshold are the ones chosen on validation in 4.9; we did not switch models or thresholds after seeing these numbers.

### 4.11 Explainability (XAI) of the final model

We explain the final model chosen in 4.9 (`FINAL_MODEL`). The code only calls `predict_proba`, so it works for any of our models. We use the validation years: explanations do not choose anything, so there is no reason to touch the test set again.

* **Permutation importance (global):** shuffle one feature at a time and measure how much the validation PR-AUC drops. A big drop means the model relies on that feature. Because several features carry the same information (for example `grid` and `qual_position`), we also shuffle each of P3's **feature groups** together: shuffling only one of two twin features underestimates both, since the model can lean on the twin.
* **SHAP (global and local):** splits each prediction into one contribution per feature, starting from a baseline (the average prediction). The contributions add up exactly to the predicted probability.
* **LIME (local):** for one row, creates many slightly changed copies, asks the model about them, and fits a small linear model around that row. Its weights say which feature values push this one prediction up or down.

**Not causal.** All three methods describe how *the model* uses the features to make predictions. A feature being important is not proof that it causes podium finishes: starting at the front does not by itself put a driver on the podium; a fast car and driver cause both the good grid slot and the podium, and the model only uses the grid slot as a signal of that.

#### Global: permutation importance (validation, 30 shuffles)

In [ ]:
from sklearn.inspection import permutation_importance

pr_auc_scorer = lambda clf, X, y: average_precision_score(y, clf.predict_proba(X)[:, 1])   # works for any of our models
pi = permutation_importance(FINAL_MODEL, X_val, y_val, scoring=pr_auc_scorer, n_repeats=30, random_state=SEED)
perm = pd.DataFrame({"PR-AUC drop": pi.importances_mean, "std": pi.importances_std}, index=FEATURES)

# the same idea per feature group: shuffle all columns of a group together
rng = np.random.default_rng(SEED)
full_score = pr_auc_scorer(FINAL_MODEL, X_val, y_val)
group_rows = []
for group, cols in FEATURE_GROUPS.items():
    drops = []
    for _ in range(30):
        shuffled = X_val.copy()
        shuffled[cols] = X_val[cols].iloc[rng.permutation(len(X_val))].set_axis(X_val.index)
        drops.append(full_score - pr_auc_scorer(FINAL_MODEL, shuffled, y_val))
    group_rows.append({"group": group, "PR-AUC drop": np.mean(drops), "std": np.std(drops)})
group_perm = pd.DataFrame(group_rows).set_index("group")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [3, 2]})
for ax, imp, title in [(ax1, perm, "per feature"), (ax2, group_perm, "per feature group (shuffled together)")]:
    imp = imp.sort_values("PR-AUC drop")
    ax.barh(imp.index, imp["PR-AUC drop"], xerr=imp["std"], color="tab:blue")
    ax.set_title(f"Permutation importance {title}")
    ax.set_xlabel("drop in validation PR-AUC (bar = mean, line = std)")
fig.suptitle(f"Final model: {FINAL_NAME}")
plt.tight_layout()
plt.show()
display(perm.sort_values("PR-AUC drop", ascending=False).round(4).head(8))
display(group_perm.sort_values("PR-AUC drop", ascending=False).round(4))

#### Global: SHAP summary (validation)
SHAP and LIME call the model with plain number arrays, so a small helper `predict_podium_numeric` first turns them back into a table with the feature names. (Its name differs from P1's inference function `predict_podium` on purpose, so the two do not overwrite each other in the merged notebook.) Rows with a missing value (mostly round 1, without standings) are left out of the explanations, so every explained value is real. Each dot is one validation row. Position = how much that feature moved the predicted podium probability for that row (right = up, left = down); colour = the feature's value (red = high, blue = low).

In [ ]:
import warnings
with warnings.catch_warnings():          # shap's import prints a harmless progress-bar warning in notebooks
    warnings.simplefilter("ignore")
    import shap

def predict_podium_numeric(A):
    """Podium probability for a plain number array in FEATURES order (how SHAP and LIME call the model)."""
    return FINAL_MODEL.predict_proba(pd.DataFrame(np.asarray(A, dtype=float), columns=FEATURES))[:, 1]

complete = val[FEATURES].notna().all(axis=1)
xai_rows = val[complete]                                   # validation rows without missing values
X_xai, y_xai = xai_rows[FEATURES].astype(float), xai_rows["podium"].to_numpy()
print(f"explained rows: {len(X_xai)} of {len(val)} validation rows")

background = X_xai.sample(100, random_state=SEED)          # SHAP's baseline = average prediction over these rows
explainer = shap.Explainer(predict_podium_numeric, shap.maskers.Independent(background), algorithm="permutation",
                           feature_names=FEATURES, seed=SEED)
shap_values = explainer(X_xai, max_evals=2 * len(FEATURES) + 1, silent=True)
print(f"SHAP baseline (average predicted podium probability of the background rows): {shap_values.base_values[0]:.3f}")

shap.plots.beeswarm(shap_values, max_display=12, show=False)
plt.title(f"SHAP summary of the final model ({FINAL_NAME}), validation")
plt.xlabel("SHAP value (change in predicted podium probability)")
plt.tight_layout()
plt.show()

#### Local: one podium row and one non-podium row
Chosen by a fixed rule, not by hand: the podium finisher with the **highest** predicted probability (a clear case), and the non-podium row with the **median** predicted probability among non-podiums (a typical midfield car). The finishing position and status are shown only to tell the story; they are post-race information and never features.

In [ ]:
race_names = tables["races"].set_index("raceId")["name"]
driver_names = tables["drivers"].set_index("driverId")["surname"]
finish = outcomes.set_index(["raceId", "driverId"])[["positionOrder", "status"]]

proba_xai = predict_podium_numeric(X_xai)
non_podium_rows = np.where(y_xai == 0)[0]
examples = {
    "podium": int(np.argmax(np.where(y_xai == 1, proba_xai, -1))),
    "non-podium": int(non_podium_rows[np.argsort(proba_xai[non_podium_rows])[len(non_podium_rows) // 2]]),
}

def describe(i):
    """Who/where for explained row i (finishing position and status from `outcomes`, for the story only)."""
    r = xai_rows.iloc[i]
    pos, status = finish.loc[(r["raceId"], r["driverId"])]
    return (f"{driver_names[r['driverId']]}, {race_names[r['raceId']]} {r['year']}: grid {r['grid']}, "
            f"predicted {proba_xai[i]:.1%}, finished P{pos} ({status})")

for label, i in examples.items():
    print(f"{label:>10}: {describe(i)}")

**Local SHAP (waterfall):** starts at the baseline at the bottom and adds each feature's contribution until it reaches this row's predicted probability at the top.

In [ ]:
for label, i in examples.items():
    shap.plots.waterfall(shap_values[i], max_display=10, show=False)
    plt.title(f"Local SHAP, {label} example\n{describe(i)}")
    plt.tight_layout()
    plt.show()

**LIME:** the bars are the weights of LIME's small linear model around this row (green pushes towards podium, red away from it). We also print how close LIME's local model gets to the real model's probability.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

yes_no_columns = [j for j, c in enumerate(FEATURES) if X_xai[c].nunique() <= 2]   # 0/1 features, treated as categories
lime_explainer = LimeTabularExplainer(X_xai.to_numpy(), feature_names=FEATURES, class_names=["no podium", "podium"],
                                      categorical_features=yes_no_columns, mode="classification", random_state=SEED)
predict_both = lambda A: np.column_stack([1 - predict_podium_numeric(A), predict_podium_numeric(A)])

for label, i in examples.items():
    lime_result = lime_explainer.explain_instance(X_xai.iloc[i].to_numpy(), predict_both, num_features=10)
    lime_result.as_pyplot_figure()
    plt.title(f"LIME, {label} example\n{describe(i)}")
    plt.xlabel("LIME weight (effect on the podium probability)")
    plt.tight_layout()
    plt.show()
    print(f"{label}: model probability {proba_xai[i]:.3f}, LIME's local estimate {lime_result.local_pred[0]:.3f}, "
          f"local fit R^2 {lime_result.score:.2f}")

#### The most confident miss
The non-podium row the model was most sure about: what happened in that race?

In [ ]:
most_confident_miss = int(np.argmax(np.where(y_xai == 0, proba_xai, -1)))
print(f"most confident miss: {describe(most_confident_miss)}")

**Interpretation.**
* **Qualifying and grid carry most of the signal.** Per feature, shuffling `qual_position` drops the validation PR-AUC by 0.169 and `grid` by 0.089; every other feature is below 0.025. Per group, **qualifying (0.112) and grid (0.098)** come first, then **standings (0.057)** and **form (0.028)**; **circuit history (0.004)** adds almost nothing and **context (`home_flag`) adds nothing** (-0.003). P3's ablation (3.13), which retrains the model without each group, gives a similar order: grid, qualifying and standings first; circuit, form and context last.
* **Per feature vs per group:** the qualifying group drops less (0.112) than `qual_position` alone (0.169). This is expected with twin features: shuffling a whole group also shuffles inputs that partly cancel each other inside a linear model (for example `qual_position` and `reached_q3`), while single-feature importance measures how much the model needs one column *given the others*. Neither number alone is "the" importance; that is why we show both.
* **Permutation importance vs ablation:** permutation importance keeps the trained model and breaks one group's values; ablation (P3) retrains without the group, so the model can lean on the remaining twins. That is why ablation drops are smaller (at most 0.014) than permutation drops.
* **Direction (SHAP summary):** a low (blue) `qual_position`, `grid` and `pos_before` push the probability up by up to about +0.25; a high (red) `driver_form5` and `pts_before` push it up too. Some signs look surprising, for example `reached_q3 = 1` pushes *down*: given the exact qualifying position, reaching Q3 adds nothing new, and the linear model uses it to correct the other qualifying features. This is a property of correlated inputs, not of Formula 1.
* **Podium example (Hamilton, Bahrain 2020, from pole, won):** SHAP starts at the baseline 0.167 and adds grid 1 (+0.21), his recent form (+0.16), qualifying P1 (+0.13), leading the championship (+0.10), 307 points (+0.08) and 8 earlier podiums at this circuit (+0.07), reaching 0.924. **Non-podium example (Vettel, Qatar 2021, grid 10, finished 10th):** qualifying 10th (-0.05), grid 10 (-0.03), 12th in the championship (-0.02) and no recent podium (-0.02) bring him down to 0.017. Both explanations are what a fan would expect.
* **LIME agrees on the direction but is only an approximation:** it ranks `qual_position` and `grid` first, but its small linear model estimates 0.34 where the real model says 0.92 (local fit R² 0.53), and it fits the non-podium row poorly (R² 0.11). SHAP values add up exactly to the model's prediction, so we trust SHAP more for the size of each effect.
* **The most confident miss is Hamilton at the 2020 Italian Grand Prix:** pole position, predicted 91.3%, finished 7th after a stop-and-go penalty during the race. Events inside the race cannot be seen by a pre-race model; this is a limit of the task, not of the model (4.12).

**Not causal.** These explanations describe the model, not Formula 1. The model relies on the qualifying and grid positions because they are the best pre-race summary of how fast the car and driver are; it does not show that the starting position *causes* podiums, and moving a driver up the grid would not give him the model's probability.

### 4.12 Model limitations

Two checks give numbers to the main limitations.
1. **Are the predicted probabilities realistic in every split?** Every race has exactly 3 podiums, so the predicted probabilities of one race should add up to about 3. This check needs **no labels**, so it can be run on any future season before its races; we also compare the average predicted probability with the real podium rate.
2. **The false alarms on the test set:** how many of those cars did not finish the race (crash, breakdown), something no pre-race model can see?

In [ ]:
# 1) are the probabilities realistic? (podiums per race should be about 3; this part needs no labels)
calib_rows = []
for split, part in [("train", train), ("validation", val), ("test", test)]:
    p = pd.Series(FINAL_MODEL.predict_proba(part[FEATURES])[:, 1], index=part.index)
    calib_rows.append({"split": split, "predicted podiums per race (median)": p.groupby(part["raceId"]).sum().median(),
                       "average predicted probability": p.mean(), "real podium rate": part["podium"].mean()})
display(pd.DataFrame(calib_rows).set_index("split").round(3))

# 2) false alarms on the test set: how many of those cars did not finish the race?
status = test[["raceId", "driverId"]].merge(outcomes[["raceId", "driverId", "status"]],
                                            on=["raceId", "driverId"], how="left")["status"]
finished = (status.eq("Finished") | status.str.match(r"^\+\d+ Laps?$")).to_numpy()
said_podium = FINAL_MODEL.predict_proba(X_test)[:, 1] >= FINAL_THRESHOLD
false_alarm = said_podium & (y_test.to_numpy() == 0)
print(f"test false alarms: {false_alarm.sum()}; did not finish the race: {(~finished[false_alarm]).sum()} "
      f"({(~finished[false_alarm]).mean():.0%}), compared with {(~finished).mean():.0%} of all test starters")

**What we see.**
* **Probabilities are realistic in every split:** about 3.0 predicted podiums per race on train, validation and test (3.04, 2.98, 3.01), and an average probability of 0.142, 0.148 and 0.149 against real podium rates of 0.142, 0.151 and 0.151. With `era` in the table (the first test run, see 4.10) the test years showed about 5 predicted podiums per race; this per-race check needs no labels, so it should be run on every new season before the threshold is trusted.
* **False alarms:** 137 on the test set; 20% of those cars did not finish the race, compared with 14% of all test starters. Some false alarms are cars that broke down or crashed, something no pre-race model can see.

**Limitations of the models**

| Model | Main limitation in this project |
|---|---|
| Logistic regression (final) | Linear: no interactions unless they are built as features; the weights of correlated features (grid, qualifying, standings) are not individually interpretable; it extrapolates linearly to values never seen in training. |
| Random forest | Overfits with small leaves (train PR-AUC 0.91 vs test 0.68); probabilities are vote shares, not calibrated; cannot extrapolate beyond training values. |
| Gradient boosting | Sensitive to tree size (default settings overfit: train 0.92 vs validation 0.71); many hyperparameters for a small validation set. |
| Shallow neural network | Stopping epoch chosen on validation, so its validation score is optimistic (0.758 on validation, 0.701 on test); many settings; a black box without XAI. |

**Limitations of the data and the method**
* **Era sensitivity:** rules, car design and the points system change between eras (25 points for a win since 2010, 10 before), so the same feature value does not mean the same thing in every season. The test years are a new technical era (2022 ground-effect cars) that the model has never seen; the realistic test probabilities and the small drop from validation to test show that the pre-race features still transfer, but a future rule change could shift them. `era` itself is not a model input: its test value never occurs in training (P3, 3.12).
* **Qualifying coverage:** reliable qualifying data starts in the mid-1990s and the knockout format in 2006, so P3's table starts in 2003; `qual_format`, `reached_q2` and `reached_q3` describe the format changes.
* **Missing values carry meaning:** in the training years a missing qualifying time almost always means a driver who set no time and started near the back, so the model reads "missing" as a bad sign (4.13: a pole-sitter without qualifying data gets about 1%). For a record with a data gap the prediction is unreliable.
* **No flag for features that are never missing in training:** only the 9 features with gaps in the training years get a missing-value flag (4.2). If one of the other features (for example `grid`) is missing in a new record, it is silently replaced by the training median. These features are always known at the prediction cutoff, so a record without them should be treated as invalid input rather than predicted.
* **Raw lap time:** `best_qual_s` depends on the circuit length, so its value is only comparable within one race (`gap_to_pole_pct` is the comparable version).
* **Small validation set:** 117 podiums; the bootstrap standard error of PR-AUC is about 0.04, larger than every difference between the tuned models, so the ranking of the models is uncertain.
* **The threshold is chosen for F1**, which counts a missed podium and a false alarm as equally bad. Another use (for example a betting or broadcast decision) weighs them differently and needs a different threshold.
* **The task itself:** incidents, penalties, safety cars, weather and strategy happen during the race and are invisible before it (Hamilton, Italy 2020, in 4.11). No pre-race model can predict them.

### 4.13 Hand-off to the inference function (P1)

P1's inference function uses the objects below. `FINAL_MODEL` is one pipeline (preprocessing + model), so it takes the raw feature columns of a record and handles missing values itself (4.2). Usage:

```python
proba = FINAL_MODEL.predict_proba(record[FEATURES])[:, 1]   # podium probability
podium = proba >= FINAL_THRESHOLD                           # yes / no decision
```

The check below runs the final model on one complete validation record and on the same record with its qualifying and standings values removed (as for a driver without a qualifying time in the first race of a season).

In [ ]:
complete_record = X_val[X_val.notna().all(axis=1)].iloc[[0]]
incomplete_record = complete_record.copy()
incomplete_record[["qual_position", "best_qual_s", "gap_to_pole_pct", "pts_before", "pos_before",
                   "cons_pts_before", "cons_pos_before"]] = np.nan
check = pd.concat([complete_record, incomplete_record]).assign(record=["complete", "incomplete"]).set_index("record")
check["podium probability"] = FINAL_MODEL.predict_proba(check[FEATURES])[:, 1].round(3)
check["podium (threshold)"] = check["podium probability"] >= FINAL_THRESHOLD
assert check["podium probability"].notna().all()
display(check[["grid", "qual_position", "pts_before", "podium probability", "podium (threshold)"]])
print(f"FINAL_NAME = {FINAL_NAME!r}, FINAL_THRESHOLD = {FINAL_THRESHOLD}, {len(FEATURES)} features")

---
# >>> Section from `notebooks/p1_inference.ipynb`


## Part 5: Inference function

`predict_podium(record)` answers one question: will this driver finish on the podium (top 3) in this race? It takes the pre-race features of **one driver in one race** and returns the podium probability and a yes/no answer. It uses the final model from Part 4, so nothing is retrained.

### 5.1 The function

* **Input:** a dictionary with the features in `FEATURES` (Part 3). A feature that is unknown can be left out or set to `None`.
* **Step 1:** refuse the record if it contains a post-race column (`positionOrder`, `points`, `status`, ...), because those are not known before the race.
* **Step 2:** build one row with the features in the same order as in training.
* **Step 3:** give the row to `FINAL_MODEL`. The model already contains the preprocessing from Part 4 (filling missing values, scaling), so it is not repeated here.
* **Output:** the podium probability, the yes/no answer (probability at least `FINAL_THRESHOLD`) and the list of missing features.

In [ ]:
import numpy as np
import pandas as pd

def predict_podium(record, model=None, threshold=None):
    model = FINAL_MODEL if model is None else model
    threshold = FINAL_THRESHOLD if threshold is None else threshold

    leaky = set(record.keys()) & (set(POST_RACE_COLS) | {"podium"})
    if leaky:
        raise ValueError(f"post-race information is not allowed: {sorted(leaky)}")

    values = {}
    for f in FEATURES:
        v = record.get(f, np.nan)
        values[f] = np.nan if pd.isna(v) else v
    X = pd.DataFrame([values], columns=FEATURES).astype(float)

    proba = float(model.predict_proba(X)[0, 1])
    return {
        "podium_probability": proba,
        "podium_prediction": int(proba >= threshold),
        "missing_features": [f for f in FEATURES if pd.isna(values[f])],
    }

### 5.2 Example A: complete record
A real driver-race from the validation years: the first one where all features are known. The finishing position is only printed to describe the example, it is not an input.

In [ ]:
def describe_row(row):
    driver = tables["drivers"].set_index("driverId").loc[row["driverId"], "surname"]
    race = tables["races"].set_index("raceId").loc[row["raceId"], "name"]
    finish = outcomes.set_index(["raceId", "driverId"]).loc[(row["raceId"], row["driverId"]), "positionOrder"]
    return f"{driver}, {race} {row['year']} (actual finish: P{int(finish)})"

complete_rows = table[(table["split"] == "validation") & table[FEATURES].notna().all(axis=1)]
row_A = complete_rows.iloc[0]
record_A = row_A[FEATURES].to_dict()

print(describe_row(row_A))
print("input:", record_A)
print("output:", predict_podium(record_A))

All features are given, so the model uses them as they are. The output has the podium probability and the yes/no answer at the validation threshold. The function running without errors does not show that the prediction is right: one record proves nothing about accuracy. The accuracy of the model is measured in Part 4 on the validation and test sets.

### 5.3 Example B: partially missing record
A real driver-race where information is missing: the first validation row of a driver's first Grand Prix. He has no earlier races (no form) and it is the first round of the season (no standings yet). The unknown fields are left out of the dictionary.

In [ ]:
first_race_rows = table[(table["split"] == "validation") & table["driver_form5"].isna()]
row_B = first_race_rows.iloc[0]
record_B = {k: v for k, v in row_B[FEATURES].to_dict().items() if pd.notna(v)}

print(describe_row(row_B))
print("input:", record_B)
print("output:", predict_podium(record_B))

The missing features are listed in `missing_features`. The preprocessing inside the model fills each missing number with the median of the training years, and for features that had gaps in training it also adds a "was missing" flag, so the model knows the value was filled in. The prediction is based on a typical value, not the real one, so it is less reliable than for a complete record.

### 5.4 Checks
The function should give the same probability as calling the model directly, including for the validation rows that have missing values, and it should refuse post-race columns.

In [ ]:
val = table[table["split"] == "validation"]
direct = FINAL_MODEL.predict_proba(val[FEATURES])[:, 1]
one_by_one = np.array([predict_podium(r[FEATURES])["podium_probability"] for _, r in val.iterrows()])
print("validation rows:", len(val), "| largest difference:", abs(direct - one_by_one).max())
assert np.allclose(direct, one_by_one)

for col in ["positionOrder", "points", "status", "podium"]:
    try:
        predict_podium({**record_A, col: 1})
    except ValueError:
        print("refused:", col)

### 5.5 Limitations
* The function takes the features computed in Part 3 (form, standings, circuit history). It does not build them from raw race data.
* Missing values are replaced by typical values. Predictions for records with missing qualifying or grid information are not reliable.
* It predicts one driver at a time, so it can say "podium" for more or fewer than three drivers in the same race. It cannot be better than the Part 4 model.